# 🗺️ Playbook Execution Flow (Decision DAG)

```mermaid
graph TD
    classDef start fill:#1b5e20,stroke:#2e7d32,color:#fff,stroke-width:2px;
    classDef endStep fill:#b71c1c,stroke:#c62828,color:#fff,stroke-width:2px;
    classDef action fill:#0d47a1,stroke:#1565c0,color:#fff;
    classDef hitl fill:#f57f17,stroke:#fbc02d,color:#000;
    classDef mutation fill:#d32f2f,stroke:#c62828,color:#fff,stroke-width:3px;
    step_start(("START")):::start
    step_start --> step_triage
    step_triage["Triage & Verification"]:::action
    step_triage --> step_evidence
    step_evidence["Evidence Collection"]:::action
    step_evidence --> step_approval
    step_approval{"HITL Approval Gate"}:::hitl
    step_approval -- Success --> step_mutation
    step_approval -- Failure --> step_end
    step_mutation["State Mutation"]:::mutation
    step_mutation --> step_signing
    step_signing["Evidentiary Signing"]:::action
    step_signing --> step_end
    step_end(("END")):::endStep
```

<div style="background-color: #1e1e1e; color: #e0e0e0; padding: 20px; border-left: 6px solid #f44336; margin-bottom: 20px; font-family: 'Inter', sans-serif; font-size: 14pt; line-height: 1.33;">
  <h3 style="margin-top: 0; color: #ffffff; font-size: 1.5rem;">BLUF (Bottom Line Up Front)</h3>
  <p style="font-size: 14pt; color: #f44336; font-weight: bold; margin-bottom: 16px;">⏱️ Incident Timer: T+00:00:00</p>
  <p style="font-size: 14pt;"><strong>Incident Goal:</strong> Detects loading of known malicious drivers via their hash.</p>
  <p style="font-size: 14pt;"><strong>Goal Alignment Index (GAI):</strong> 3.87116 (Strategic Reliability)</p>
  <p style="font-size: 14pt;"><strong>Critical Action:</strong> Authorize <b>Containment</b> following agent verification.</p>
</div>

<details>
<summary><b>ASO Playbook Quick Jump Navigation</b></summary>

### 📌 Quick Jump

- [1. Resolution Lifecycle (Execution)](#1-resolution-lifecycle-aso)
- [2. Escalation & Communication](#2-escalation--communication)
- [3. Evidence & Enrichment](#3-evidence--enrichment)
- [4. Incident Impact & Context](#4-incident-impact--context)
- [5. Agent Supervision](#5-agent-supervision)
- [6. Detection Reference [Collapsed]](#6-detection-reference)

</details>

<details>
<summary><b>SynAgency ASOCO Operational Readiness & Compliance Badges</b></summary>

# Badges

![Build Status](https://img.shields.io/badge/Build-Passing-brightgreen?style=flat-square&logo=microsoft)
![Documentation](https://img.shields.io/badge/Documentation-Complete-blue?style=flat-square&logo=github)
![Response Efficiency](https://img.shields.io/badge/Response%20Efficiency-98%25-green?style=flat-square&logo=microsoft)
![Last Updated](https://img.shields.io/badge/Last%20Updated-May%202026-purple?style=flat-square&logo=openai)
![Incidents Resolved](https://img.shields.io/badge/Incidents%20Resolved-150-red?style=flat-square&logo=microsoft)
![Community Engagement](https://img.shields.io/badge/Community-Active-orange?style=flat-square&logo=openai)
![Code Integration](https://img.shields.io/badge/Code%20Integration-High-teal?style=flat-square&logo=github)
![AI Analysis](https://img.shields.io/badge/AI%20Analysis-Advanced-blueviolet?style=flat-square&logo=microsoft)
![Threat Detection](https://img.shields.io/badge/Threat%20Detection-Optimal-red?style=flat-square&logo=openai)
![Security Hardening](https://img.shields.io/badge/Security-Hardened-silver?style=flat-square&logo=microsoft)
![Service Uptime](https://img.shields.io/badge/Uptime-99.9%25-brightgreen?style=flat-square&logo=github)
![Data Privacy](https://img.shields.io/badge/Privacy-Compliant-green?style=flat-square&logo=microsoft)
![Automation Coverage](https://img.shields.io/badge/Automation%20Coverage-High-black?style=flat-square&logo=google)
![Detection Fidelity](https://img.shields.io/badge/Detection%20Fidelity-High-blue?style=flat-square&logo=openai)
![Pipeline Health](https://img.shields.io/badge/Pipeline%20Health-Nominal-green?style=flat-square&logo=github)
![Runbook Validation](https://img.shields.io/badge/Runbook%20Validation-Passing-red?style=flat-square&logo=openai)
![Cross-Team Coverage](https://img.shields.io/badge/Cross--Team%20Coverage-Confirmed-yellow?style=flat-square&logo=github)
![SLO Compliance](https://img.shields.io/badge/SLO%20Compliance-Met-lightblue?style=flat-square&logo=github)

</details>

<br>

In [ ]:
# [Bootstrap] ASO Runtime — shared Vault/gRPC/Google helpers
%load_ext autoreload
%autoreload 2

# ── sys.path bootstrap ──
import sys, os, pathlib
_cwd = pathlib.Path.cwd()
_root = next((p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()), _cwd)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.runtime.aso_runtime import (
    IncidentContext, vault, grpc_channel, google_service, handle_cell_exceptions,
)

# Incident context — populated by SigmaNotebook template substitution
INCIDENT = IncidentContext(
    incident_id      = "1305345",
    case_id          = "INC-2026-1305345",
    event_source     = "windows",
    affected_client  = "C.0000000000000000",
    severity         = "high",
    incident_lead    = "ASO Incident Command",
    affected_systems = "grr-agent-service, gao-orchestrator",
    summary          = """Detects loading of known malicious drivers via their hash.""",
    target_ip        = "127.0.0.1",
    target_user      = "unknown_user",
    ioc_list         = [],
    malicious_files  = [],
    cacao_playbook_type = "investigation",
    required_confidence = 85,
)

# [Orchestration Gate] Evaluate confidence threshold
# Map incoming alert confidence (mocked or injected via Papermill)
from src.runtime.confidence_threshold import ConfidenceThresholdConfig
alert_confidence = int(globals().get('ALERT_CONFIDENCE', 100)) # Default 100 for manual runs
REQUIRE_HITL = False

CONFIDENCE_CONFIG = ConfidenceThresholdConfig(
    threshold_percent=INCIDENT.required_confidence,
    alert_confidence=alert_confidence,
    fidelity_model="siem_sigma_v1"
)

if not CONFIDENCE_CONFIG.is_gate_passed():
    gate_tag = CONFIDENCE_CONFIG.get_gate_tag()
    print(f"⚠️  [GATE] {gate_tag.reason_text}")
    print("⚠️  [GATE] Flipping REQUIRE_HITL to True. All state-mutating actions will require manual approval.")
    REQUIRE_HITL = True
else:
    print(f"✅ [GATE] Confidence threshold met ({alert_confidence}% >= {INCIDENT.required_confidence}%)")

# ── Environment Snapshot (Reproducibility) ──
from src.runtime.execution_environment_snapshot import SnapshotCapture
env_snapshot = SnapshotCapture.capture()
print(f"✅ Environment captured | Python {env_snapshot.python_version} | OS {env_snapshot.os_kernel}")

# ── Telemetry Initialization ──
from src.runtime.execution_telemetry import ExecutionTelemetryLogger, TelemetryStreamWriter
_telemetry_logger = ExecutionTelemetryLogger(
    notebook_version="v1",
    incident_id=INCIDENT.incident_id,
    agent_model_version=os.environ.get("LLM_MODEL_VERSION", "claude-3-5-sonnet-20241022")
)
_stream_writer = TelemetryStreamWriter(
    sink_type=os.environ.get("TELEMETRY_SINK_TYPE", "file"),
    log_path=os.environ.get("TELEMETRY_LOG_PATH", f"/tmp/execution_telemetry_{INCIDENT.incident_id}.jsonl")
)
print("✅ Telemetry initialized")

# ── Regulatory Compliance Tracking (v0.2) ──
import datetime
from src.runtime.regulatory_timestamps import RegulatoryTimestampLogger
_compliance_logger = RegulatoryTimestampLogger()
_discovery_time = datetime.datetime.utcnow().isoformat() + "Z"
_compliance_logger.set_incident_context(INCIDENT.incident_id, _discovery_time)
_compliance_logger.log("evt_detection", "detection_alert_received", _discovery_time, "GDPR")
print(f"✅ Regulatory compliance tracking initialized | GDPR deadline: {_compliance_logger._calculate_deadline(_discovery_time, 'GDPR', 'notification')}")

# ── Multi-SIEM Query Standardization (v0.2) ──
from src.runtime.query_standardization import QueryBuilder, QueryRegistry
_registry = QueryRegistry()
_siem_queries = {}
for platform in ["splunk", "elastic", "kql"]:
    try:
        builder = QueryBuilder(platform)
        _query = builder.add_filter("hostname", "eq", INCIDENT.affected_client).build(time_range="-4h")
        _siem_queries[platform] = _query
    except Exception:
        pass  # Skip platforms on failure
print(f"✅ Multi-SIEM query standardization ready | {len(_siem_queries)} platforms available")

# ── Cell Integrity Checksums (v0.2) ──
from src.runtime.cell_checksums import ChecksumCalculator, ChecksumStore
_checksum_store = ChecksumStore()
_checksum_calculator = ChecksumCalculator()
print("✅ Cell integrity tracking enabled")

# ── Named Field Standardization (SIEM Normalization) ──
from src.runtime.named_field_registry import NamedFieldRegistry
_required_fields = ["incident_id", "target_ip", "target_user"]
_missing = [f for f in _required_fields if not getattr(INCIDENT, f.replace("incident_id", "incident_id"), None)]
if not _missing:
    print(f"✅ Named field validation passed | {len(_required_fields)} required fields present")
else:
    print(f"⚠️  Missing fields: {', '.join(_missing)}")

print(f"✅ ASO runtime ready — {INCIDENT.incident_id}")

# ── Dry-Run Enforcement Protocol ──
from src.runtime.dry_run_wrapper import DryRunExecutor, ToolSchemaExtension
print(ToolSchemaExtension.generate_dry_run_enforcement_prompt())


# 1. Resolution Lifecycle (ASO)

<div style="border-left: 4px solid #444; margin-left: 10px; padding-left: 20px; position: relative;">

## ⏺️ 1.1. 🤖 [AUTONOMOUS] Step 1: Triage & Verification

Purpose: Confirm the alert is a True Positive (TP) and assess the current blast radius.

- [ ] **Examine Target System**: Verify presence of `unknown_log`.
- [ ] **Check User Activity**: Correlate `unknown_user` actions with known baseline.
- [ ] **Indicator Search**: Search for `"[]"` across the environment.
- [ ] **Enrichment**: Execute enrichment tasks including diamond modeling and malware analysis using `"[]"` , `unknown_log` , `unknown_user` , `Individual Account Compromise` , `Tier-2` , `Moderate` and store the results in the `uri://forensics/`.

In [ ]:
# [Executable Workflow] Initialize Kestrel Session for Threat Hunting
@handle_cell_exceptions()
def run():
    try:
        from kestrel.session import Session
    except ImportError:
        print("⚠️ Kestrel module not installed. Skipping Kestrel session initialization.")
        return

    with Session() as session:
        print("✅ Kestrel session initialized.")
        # Hunt for IOCs in the current incident context
        print(f"Targeting IOCs: {INCIDENT.ioc_list}")
        # session.execute(...)

run()

## ⏺️ 1.2. 🤖 [AUTONOMOUS] Step 2: Remote Forensic Triage (GRR)

Purpose: Execute high-fidelity forensic collection via Google Rapid Response.

- [ ] **Establish Connection**: Initialize GRR API session for `C.0000000000000000`.
- [ ] **Collect Volatile Data**: Retrieve process list and network connections.
- [ ] **Targeted Search**: Execute `FileFinder` flow for artifacts related to `Malicious Driver Load`.

<div style="background-color: #2d2d2d; color: #e0e0e0; border: 1px solid #444; border-left: 6px solid #ff9800; border-radius: 6px; padding: 15px; margin-bottom: 1em;">

In [ ]:
# [Executable Workflow] GRR Forensic Triage
# 👤 [HITL REQUIRED] - Remote system access: forensic data collection may require endpoint owner approval
import sys
try:
    from grr_api_client.proto.grr_response_proto.api import api_pb2, api_pb2_grpc, flow_pb2
    from google.protobuf import text_format
except ImportError:
    from unittest.mock import MagicMock
    api_pb2 = MagicMock()
    api_pb2_grpc = MagicMock()
    flow_pb2 = MagicMock()
    text_format = MagicMock()

@handle_cell_exceptions()
def run():
    # ── Telemetry: Cell Start ──
    import time, json
    _cell_start = time.time()
    _telemetry_logger.log_cell_started(
        cell_id="grr_forensic_triage",
        cell_type="evidence_capture",
        input_params={"client_id": INCIDENT.affected_client}
    )

    # ── Tool Access Validation ──
    from src.runtime.playbook_type_enforcement import ToolAccessController
    ToolAccessController.validate_tool_access(
        playbook_type=INCIDENT.cacao_playbook_type,
        tool_name="grr_rapid_response",
        tool_category="investigation"
    )
    print(f"✅ Tool 'grr_rapid_response' authorized for '{INCIDENT.cacao_playbook_type}' playbook")

    try:
        with grpc_channel("{$GRR_ENDPOINT}") as ch:
            stub = api_pb2_grpc.ApiStub(ch)
            req = api_pb2.ApiCreateFlowArgs(
                client_id=INCIDENT.affected_client,
                flow=flow_pb2.Flow(
                    name="ListProcesses",
                    args=text_format.Parse("implementation_type: CLIENT", flow_pb2.FlowArgs()),
                ),
            )
            resp = stub.CreateFlow(req)
            print(f"✅ Flow dispatched | flow_id={resp.flow_id} | client={INCIDENT.affected_client}")

            # ── Transparent Reasoning Display ──
            # If agent includes reasoning, extract and display it
            from src.runtime.transparent_reasoning import ReasoningRenderer
            from IPython.display import HTML, display

            # Simulate agent reasoning output (would come from real agent)
            agent_output = json.dumps({
                "alert_id": INCIDENT.incident_id,
                "verdict": "true_positive",
                "confidence": "high",
                "risk_score": 0.9,
                "summary": "Initiated ListProcesses flow to enumerate running processes for forensic analysis"
            })
            
            # Validate triage output against strict schema
            from src.runtime.strict_json_validation import JSONValidator, EvidenceTriage
            try:
                JSONValidator.validate_output(agent_output, schema=EvidenceTriage)
                print("✅ Triage output validated against EvidenceTriage schema")
            except Exception as e:
                print(f"⚠️  Triage validation failed: {getattr(e, 'errors', str(e))}")

            cleaned, reasoning_html = ReasoningRenderer.extract_and_render(agent_output)
            if reasoning_html:
                display(HTML(reasoning_html))
                
            # ── Telemetry: Cell Success ──
            _telemetry_logger.log_cell_completed(
                cell_id="grr_forensic_triage",
                output=agent_output,
                duration_ms=int((time.time() - _cell_start) * 1000)
            )

    except Exception as e:
        _telemetry_logger.log_cell_failed(
            cell_id="grr_forensic_triage",
            error_message=str(e),
            duration_ms=int((time.time() - _cell_start) * 1000)
        )
        raise

run()

## ⏺️ 1.3. 👤 [HITL REQUIRED] Step 3: Containment (Immediate)

> [!IMPORTANT]
> Purpose: Stop the adversary's progress and protect sensitive data.

- [ ] **Action A**: Manual Analyst Review
- [ ] **Action B**: Request Forensic Image

In [ ]:
# [Executable Workflow] GAO Containment
# 👤 [HITL REQUIRED] - Destructive action: isolation and network containment
# Guard: Check for Human-in-the-Loop requirement
if globals().get('REQUIRE_HITL', False):
    print("⚠️ [HITL] Manual authorization required for containment. Halting autonomous execution.")
    raise RuntimeError("Human-in-the-Loop gate active: Confidence threshold not met.")

import sys
try:
    from gao.proto import containment_pb2, containment_pb2_grpc
except ImportError:
    from unittest.mock import MagicMock
    containment_pb2 = MagicMock()
    containment_pb2_grpc = MagicMock()

@handle_cell_exceptions()
def run(target_ip="127.0.0.1"):
    # ── Tool Access Validation ──
    _effective_type = INCIDENT.cacao_playbook_type
    if _effective_type in ['investigation', 'detection']:
        print("⚠️ Override: Containment block executed in non-mutating playbook. Elevating effective playbook type to 'containment'.")
        _effective_type = 'containment'

    from src.runtime.playbook_type_enforcement import ToolAccessController
    ToolAccessController.validate_tool_access(
        playbook_type=_effective_type,
        tool_name="gao_containment",
        tool_category="containment"
    )
    print(f"✅ Tool 'gao_containment' authorized for '{_effective_type}' playbook")

    # ── Dry-Run & Blast Radius Validation ──
    from src.runtime.dry_run_wrapper import DryRunExecutor
    import asyncio

    def execute_containment(**kwargs):
        with grpc_channel("gao-agent-service.internal.your-org.internal:443") as ch:
            stub = containment_pb2_grpc.ContainmentServiceStub(ch)
            resp = stub.Contain(containment_pb2.ContainRequest(
                target_ip=target_ip,
                reason=f"GAO containment for {INCIDENT.incident_id}",
                requestor="grr-notebook",
                dry_run=kwargs.get("dry_run", True),
            ))
            return resp

    executor = DryRunExecutor(
        action_name="gao_containment",
        tool_wrapper=lambda **kwargs: execute_containment(**kwargs)
    )
    
    # Step 1: Execute dry-run
    print("🔍 Running dry-run simulation...")
    try:
        # We mock the blast radius since the real service might not return it yet
        # In a real scenario, execute_dry_run would parse this from the tool output
        def mock_containment_dry_run(**kwargs):
            return {
                "blast_radius": {
                    "affected_entity_count": 1,
                    "affected_entities": [target_ip],
                    "estimated_impact": "Medium",
                    "irreversible": False,
                    "rollback_time_minutes": 5,
                    "summary": f"Would isolate {{target_ip}} from the network."
                }
            }
        
        # Override for demonstration since real GAO stub doesn't have dry_run yet
        executor.tool_wrapper = mock_containment_dry_run
        
        blast_radius = executor.execute_dry_run()
        print(executor.generate_approval_prompt(blast_radius))
    except Exception as e:
        print(f"❌ Dry-run failed: {{e}}")
        return

    # Step 2: Live Execution
    user_approved = not globals().get('REQUIRE_HITL', False)
    
    if executor.should_proceed_to_live(user_approved, blast_radius):
        # ── Time-Lock Puzzle for Containment ────────────────────────────────────────
        from src.runtime.time_lock_puzzles import TimeLockSolver
        import os, time

        # Allow HITL override for emergencies
        if os.environ.get("HITL_OVERRIDE") == "true":
            print("⚠️  WARNING: HITL_OVERRIDE enabled - bypassing puzzle requirement")
        else:
            containment_action = f"GAO containment for {INCIDENT.incident_id}"
            puzzle_difficulty = int(os.environ.get("CONTAINMENT_PUZZLE_DIFFICULTY", "15"))
            if puzzle_difficulty < 5: puzzle_difficulty = 5
            if puzzle_difficulty > 60: puzzle_difficulty = 60

            puzzle = TimeLockSolver.generate_puzzle(
                action_description=containment_action,
                difficulty_seconds=puzzle_difficulty
            )

            print(f"⏱️  CONTAINMENT PUZZLE REQUIRED")
            print(f"Action: {puzzle.action_description}")
            print(f"Difficulty: {puzzle.difficulty_seconds}s")
            print()

            start_solve = time.time()
            try:
                nonce_solution = TimeLockSolver.solve(puzzle)
                solve_duration = time.time() - start_solve
                print(f"✓ Puzzle solved in {solve_duration:.1f}s")
            except TimeoutError:
                print(f"✗ Puzzle solving timed out")
                raise

        print("✅ Approval verified. Executing live action...")
        # Switch back to real tool for live execution
        executor.tool_wrapper = lambda **kwargs: execute_containment(**kwargs)
        result = executor.execute_live()
        
        # Validate remediation result against strict schema
        from src.runtime.strict_json_validation import JSONValidator, RemediationOutput
        try:
            # result is a gRPC response object, convert to dict for validator if needed
            # or use it directly if it has matching attributes. 
            # For stub/executor result is usually a dict.
            JSONValidator.validate_output(result, schema=RemediationOutput)
            print(f"✅ Remediation validated | Status: {result.get('status', 'dispatched')}")
        except Exception as e:
            print(f"⚠️  Remediation validation failed: {getattr(e, 'errors', str(e))}")

        print(f"✅ Containment complete: {result.get('status', 'dispatched')}")
    else:
        print("❌ Action blocked: Manual approval required or safety check failed.")

run()

## ⏺️ 1.4. 👤 [HITL REQUIRED] Step 4: Eradication & Remediation

> [!CAUTION]
> Purpose: Non-destructive querying. Automated execution authorized.

- [ ] **Cleanup**: Remove `"[]"` and `"[]"`.
- [ ] **Hardening**: Apply `Apply Latest Vendor Patch`.

</div>

<div style="background-color: #efebe9; padding: 15px; border: 2px solid #5d4037; border-radius: 5px;">
<h2 style="color: #3e2723; margin-top: 0;"> BIG RED BUTTON: Eradication Execution</h2>
<p style="font-weight: bold; color: #1b5e20;">CAUTION: DESTRUCTIVE REMOVAL OF THREAT ARTIFACTS INITIATED UPON EXECUTION.</p>
<p>This cell will forcefully remove malicious files and persistence mechanisms. Verify the artifact list in the INCIDENT context before execution.</p>
</div>

In [ ]:
# [Executable Workflow] GAO Eradication
# 👤 [HITL REQUIRED] - Destructive action: permanent removal of threat artifacts
# Guard: Check for Human-in-the-Loop requirement
if globals().get('REQUIRE_HITL', False):
    print("⚠️ [HITL] Manual authorization required for eradication. Halting autonomous execution.")
    raise RuntimeError("Human-in-the-Loop gate active: Confidence threshold not met.")

import sys
try:
    from gao.proto import eradication_pb2, eradication_pb2_grpc
except ImportError:
    from unittest.mock import MagicMock
    eradication_pb2 = MagicMock()
    eradication_pb2_grpc = MagicMock()

@handle_cell_exceptions()
def run(artifacts):
    if not artifacts:
        print("⚠️  No artifacts — aborting eradication dispatch.")
        return
    
    from src.runtime.dry_run_wrapper import DryRunExecutor
    
    def execute_eradication_live(**kwargs):
        with grpc_channel("gao-agent-service.internal.your-org.internal:443") as ch:
            stub = eradication_pb2_grpc.EradicationServiceStub(ch)
            # Ensure dry_run param is passed to the gRPC service
            resp = stub.Eradicate(eradication_pb2.EradicateRequest(
                artifacts=artifacts, 
                reason=f"GAO eradication for {INCIDENT.incident_id}",
                requestor="grr-notebook", 
                dry_run=kwargs.get("dry_run", True),
            ))
            return {
                "status": resp.status,
                "processed_count": resp.processed_count,
                "blast_radius": {
                    "affected_entity_count": resp.processed_count,
                    "affected_entities": artifacts,
                    "estimated_impact": "High" if resp.processed_count > 0 else "Low",
                    "irreversible": True,
                    "rollback_time_minutes": 0,
                    "summary": f"Would eradicate {resp.processed_count} artifacts."
                }
            }

    executor = DryRunExecutor("gao_eradication", execute_eradication_live)
    
    print("🔍 Running dry-run simulation...")
    try:
        blast_radius = executor.execute_dry_run()
        print(executor.generate_approval_prompt(blast_radius))
    except Exception as e:
        print(f"❌ Dry-run failed: {e}")
        return

    user_approved = not globals().get('REQUIRE_HITL', False)
    if executor.should_proceed_to_live(user_approved, blast_radius):
        print("✅ Approval verified. Executing live eradication...")
        result = executor.execute_live()
        print(f"✅ Eradication complete: {result['status']} | processed={result['processed_count']}")
    else:
        print("❌ Action blocked: Manual approval required or safety check failed.")

run(artifacts=[])

## ⏺️ 1.5. 🤖 [AUTONOMOUS] Step 5: Recovery & Post-Incident

Purpose: Restore services and update detection logic.

- [ ] **Restore**: Re-enable services once verified clean.
- [ ] **Update**: Adjust Sigma rule `05296024-fe8a-4baf-8f3d-9a5f5624ceb2` if false positives were encountered.

## ⏺️ 1.6. 👤 [HITL REQUIRED] Step 6: Post-Mortem & Root Cause Analysis

Purpose: Standardized learning and prevention.

- [ ] **Orchestrate**: Execute the Post-Mortem workflow to clone the RCA template and schedule the debrief.
- [ ] **RCA Document**: Review and finalize the generated Root Cause Analysis Document.
- [ ] **Status**: Blame-free Post-Mortem Scheduled | Prevention Tasks Assigned | GAO Registered.

In [ ]:
# [Executable Workflow] 1.6 HITL Post-Mortem & RCA
import datetime, ipywidgets as widgets
from IPython.display import display, HTML
import sys
try:
    from gao.proto import postmortem_pb2, postmortem_pb2_grpc
except ImportError:
    from unittest.mock import MagicMock
    postmortem_pb2 = MagicMock()
    postmortem_pb2_grpc = MagicMock()

GDOCS_TEMPLATE_ID   = "YOUR_RCA_TEMPLATE_DOC_ID"
GDOCS_PARENT_FOLDER = "YOUR_POSTMORTEM_FOLDER_ID"
GCAL_TEMPLATE_ID    = "YOUR_TEMPLATE_EVENT_ID"

def _copy_and_fill_rca(meet_url=""):
    drv  = google_service("drive", "v3", "gdocs/service-account", "docs")
    docs = google_service("docs",  "v1", "gdocs/service-account", "docs")
    ts   = datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%MZ")
    copy = drv.files().copy(
        fileId=GDOCS_TEMPLATE_ID,
        body={"name": f"[RCA] {INCIDENT.incident_id} — {ts}",
              "parents": [GDOCS_PARENT_FOLDER]},
        fields="id",
    ).execute()
    doc_id  = copy["id"]
    doc_url = f"https://docs.google.com/document/d/{doc_id}/edit"
    subs = {
        "{{INCIDENT_ID}}":    INCIDENT.incident_id,
        "{{INCIDENT_DATE}}":  ts,
        "{{SEVERITY}}":       INCIDENT.severity,
        "{{SUMMARY}}":        INCIDENT.summary,
        "{{AFFECTED_CLIENT}}":INCIDENT.affected_client,
        "{{MEET_URL}}":       meet_url,
        "{{RCA_DOC_URL}}":    doc_url,
    }
    docs.documents().batchUpdate(documentId=doc_id, body={"requests": [
        {"replaceAllText": {"containsText": {"text": k, "matchCase": True},
                            "replaceText": v}} for k, v in subs.items()
    ]}).execute()
    return doc_id, doc_url

def _schedule(rca_url, start_dt, duration):
    cal = google_service("calendar", "v3", "gcal/service-account", "calendar")
    tmpl = cal.events().get(calendarId="primary", eventId=GCAL_TEMPLATE_ID,
                            conferenceDataVersion=1).execute()
    end_dt = start_dt + datetime.timedelta(minutes=duration)
    tz = tmpl.get("start", {}).get("timeZone", "America/Los_Angeles")
    body = {**{k: v for k, v in tmpl.items() if k not in
               ("id","etag","iCalUID","created","updated","htmlLink",
                "recurringEventId","originalStartTime")},
            "summary": f"[Post-Mortem] {INCIDENT.incident_id} — {INCIDENT.severity}",
            "start":{"dateTime": start_dt.isoformat(), "timeZone": tz},
            "end":  {"dateTime": end_dt.isoformat(),   "timeZone": tz},
            "description": f"RCA: {rca_url}\n\n" + tmpl.get("description",""),
            "conferenceData": {"createRequest": {
                "requestId": f"pm-{INCIDENT.incident_id}-{int(start_dt.timestamp())}",
                "conferenceSolutionKey": {"type":"hangoutsMeet"}}}}
    ev = cal.events().insert(calendarId="primary", body=body,
                             conferenceDataVersion=1, sendUpdates="all").execute()
    meet = ev.get("conferenceData",{}).get("entryPoints",[{}])[0].get("uri","")
    return ev["id"], meet, ev.get("htmlLink","")

@handle_cell_exceptions()
def _run(requestor, start_dt, duration):
    ev_id, meet, link = _schedule("", start_dt, duration)
    doc_id, doc_url   = _copy_and_fill_rca(meet_url=meet)
    with grpc_channel("gao-agent-service.internal.your-org.internal:443") as ch:
        stub = postmortem_pb2_grpc.PostmortemServiceStub(ch)
        stub.RegisterPostmortem(postmortem_pb2.PostmortemRequest(
            incident_id=INCIDENT.incident_id, rca_doc_id=doc_id, rca_doc_url=doc_url,
            cal_event_id=ev_id, meet_url=meet, requestor="grr-notebook",
            status=postmortem_pb2.PostmortemStatus.SCHEDULED,
        ))
    display(HTML(
        f'<hr><b>Complete [{requestor}]</b><br>'
        f'<a href="{doc_url}" target="_blank">📄 RCA</a> | '
        f'<a href="{link}"    target="_blank">📅 Event</a> | '
        f'<a href="{meet}"    target="_blank">🎥 Meet</a>'))

# Widget UI
_d = widgets.DatePicker(value=(datetime.datetime.now()+datetime.timedelta(days=3)).date())
_t = widgets.Text(value="10:00", description="Time:")
_m = widgets.BoundedIntText(value=60, min=15, max=240, description="Min:")
_bh = widgets.Button(description="👤 Execute", button_style="warning")
_ba = widgets.Button(description="🤖 Agent",   button_style="danger")
def _go(who):
    h, m = map(int, _t.value.split(":"))
    _run(who, datetime.datetime.combine(_d.value, datetime.time(h, m)), _m.value)
_bh.on_click(lambda _: _go("Human"))
_ba.on_click(lambda _: _go("Agent"))

# ── Evidentiary Signing (Chain of Custody) ──
# Sign the execution trace with detached JWS for forensic validity
import hashlib, os, json
from src.runtime.execution_signer import ExecutionSigner, ExecutionPayload

execution_summary = {
    "notebook_type": "sigma_investigation",
    "incident_id": INCIDENT.incident_id,
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "rca_scheduled": True,
}

trace_json = json.dumps(execution_summary, sort_keys=True)
trace_hash = hashlib.sha256(trace_json.encode()).hexdigest()

payload = ExecutionPayload(
    cell_id="postmortem_signing",
    timestamp=execution_summary["timestamp"],
    source_hash=hashlib.sha256(INCIDENT.incident_id.encode()).hexdigest(),
    output_hash=trace_hash,
    context_hash=hashlib.sha256(b"postmortem_v1").hexdigest()
)

signing_key = os.environ.get("ASO_SIGNING_KEY", "fallback-dev-key")
jws_token = ExecutionSigner.sign(payload, signing_key)
print(f"[Chain of Custody] JWS Signature: {{jws_token[:50]}}...")

# ── Regulatory Compliance Report ──
# Generate GDPR/HIPAA compliance report with deadline status
try:
    compliance_report = _compliance_logger.generate_compliance_report("GDPR")
    print("[Regulatory Compliance]\n" + compliance_report)

    # Log postmortem event for compliance timeline
    _compliance_logger.log(
        event_id="evt_postmortem",
        event_name="rca_completed",
        timestamp=datetime.datetime.utcnow().isoformat() + "Z",
        regulation="GDPR"
    )
    print("[Regulatory Compliance] RCA completion logged | GDPR compliance status updated")
except Exception as e:
    print(f"[Regulatory Compliance] Warning: {str(e)}")

# ── Emit Telemetry Logs ──
_logs = _telemetry_logger.emit_logs()
_written = _stream_writer.write_batch(_logs)
print(f"[Chain of Custody] Persisted {{_written}}/{{len(_logs)}} telemetry events")

display(widgets.VBox([widgets.HBox([_d, _t, _m]), widgets.HBox([_bh, _ba])]))


## ⏺️ 1.7. Deployment Resilience (Regenerative)

- **Strategy**: [ ] Blue/Green | [ ] Progressive Rollout
- **Rollback Status**: [ ] Ready | [ ] Executed (Date: N/A)
- **Regenerative Audit**: N/A

</div>

<br>

# 2. Escalation & Communication

## 2.1. Escalation & HITL Hooks

| Role                           | Command Channel      | Trigger Condition             |
| :----------------------------- | :------------------- | :---------------------------- |
| **SynAgency ASOCO Specialist** | #secops-oncall      | Primary Incident Handler      |
| **Operations Section Chief**   | #synagency-asoco-alerts | Infrastructure Impact         |
| **Legal / Compliance**         | (555) 0199     | Data Breach / Regulatory Risk |

<div style="background-color: #f9f9f9; border-left: 4px solid #607d8b; padding: 15px; margin-top: 20px;">

## 2.2. Stakeholder Communication Drafts

> _Agent Draft: Pre-populated messages for human review and transmission._

### Executive Update (Summary)

> Executive summary pending.

### User-Facing Notification (Service Impact)

> User impact summary pending.

</div>

<br>

# 3. Evidence & Enrichment

## 3.1. Forensic Artifacts (Evidence Locker)

<pre style="background-color: #1e1e1e; color: #d4d4d4; padding: 20px; border-radius: 5px; font-family: 'JetBrains Mono', monospace; font-size: 12pt; line-height: 1.25;">
[EVIDENCE LOCKER PAYLOAD]
TARGET_USER:      unknown_user
IOC_LIST:         "[]"
TRIGGER_LOG:      unknown_log
</pre>

- **Evidence Locker Storage**: `uri://forensics/`
- **Legal Hold Required**: [ ] Yes | [ ] No
- **Chain of Custody**: `INC-2026-1305345_MANIFEST.json`

## 3.2. Automated Enrichment Context

<table style="width:100%; text-align:left; border-collapse: collapse; font-family: 'Inter', sans-serif; font-size: 14pt; line-height: 1.33;">
  <tr style="background-color: #f3f4f6;">
    <th style="padding: 10px; border: 1px solid #ddd;">Source</th>
    <th style="padding: 10px; border: 1px solid #ddd;">Result</th>
    <th style="padding: 10px; border: 1px solid #ddd;">Risk Score</th>
  </tr>
  <tr>
    <td style="padding: 10px; border: 1px solid #ddd;"><b>VirusTotal</b></td>
    <td style="padding: 10px; border: 1px solid #ddd;">0/0</td>
    <td style="padding: 10px; border: 1px solid #ddd;"><span style="color: #d32f2f; font-weight: bold;">N/A</span></td>
  </tr>
  <tr>
    <td style="padding: 10px; border: 1px solid #ddd;"><b>CrowdStrike</b></td>
    <td style="padding: 10px; border: 1px solid #ddd;">Clean</td>
    <td style="padding: 10px; border: 1px solid #ddd;"><span style="color: #d32f2f; font-weight: bold;">0</span></td>
  </tr>
  <tr>
    <td style="padding: 10px; border: 1px solid #ddd;"><b>Mandiant / Intel</b></td>
    <td style="padding: 10px; border: 1px solid #ddd;">UNK-1</td>
    <td style="padding: 10px; border: 1px solid #ddd;">Unknown</td>
  </tr>
  <tr>
    <td style="padding: 10px; border: 1px solid #ddd;"><b>Identity Risk</b></td>
    <td style="padding: 10px; border: 1px solid #ddd;">Low</td>
    <td style="padding: 10px; border: 1px solid #ddd;">Standard User</td>
  </tr>
</table>

## 3.3. Operational ROI & Cost Analysis

| Metric                     | Value             | Threshold          |
| :------------------------- | :---------------- | :----------------- |
| **Signal-to-Noise Ratio**  | 90%    | > 85%              |
| **Ingestion Cost (Daily)** | $0.01   | < $100 |
| **Automation Savings**     | 0.5 | Hours/Year         |

<br>

## 4.1. Summary

Detects loading of known malicious drivers via their hash. attempts to address the activity described in [Sigma Rule: Malicious Driver Load](uri://aso/rules/05296024-fe8a-4baf-8f3d-9a5f5624ceb2.yml).

## 4.2. Symptoms & Triggers

| Category             | Observation          |
| :------------------- | :------------------- |
| **Detection Source** | windows       |
| **Trigger Pattern**  | N/A |
| **Confidence Level** | INV   |

## 4.3. Impact Analysis

| Impact Vector     | Description             |
| :---------------- | :---------------------- |
| **User Impact**   | Individual Account Compromise     |
| **Service Tier**  | Tier-2         |
| **Business Risk** | Moderate |

## 4.4. Operational SLO Mapping

| Objective                 | Target       | Description                      |
| :------------------------ | :----------- | :------------------------------- |
| **Time to Detect (TTD)**  | < 2m | Speed of alert firing            |
| **Time to Contain (TTC)** | < 4m | Speed of manual/auto containment |
| **Time to Resolve (TTR)** | < 26m | Speed of full remediation        |

## 4.5. Compliance & STIG Mapping

<div style="background-color: #1e1e1e; color: #d4d4d4; padding: 20px; border-radius: 8px; border: 1px solid #444; font-family: 'Inter', sans-serif;">
  <h3 style="margin-top: 0; color: #ffffff; border-bottom: 1px solid #555; padding-bottom: 10px;">📋 Regulatory Alignment & Baseline Hardening</h3>
  
  <div style="display: flex; gap: 20px; margin-bottom: 20px;">
    <div style="flex: 1; background-color: #2d2d2d; padding: 15px; border-radius: 6px; border-left: 4px solid #4CAF50;">
      <p style="margin: 0; font-size: 12px; color: #9e9e9e; text-transform: uppercase;">Baseline Image</p>
      <p style="margin: 5px 0 0 0; font-size: 16px; font-family: monospace; color: #81c784;">UBUNTU_2204_STIG_V1</p>
    </div>
    <div style="flex: 1; background-color: #2d2d2d; padding: 15px; border-radius: 6px; border-left: 4px solid #2196F3;">
      <p style="margin: 0; font-size: 12px; color: #9e9e9e; text-transform: uppercase;">Hardening Spec</p>
      <p style="margin: 5px 0 0 0; font-size: 16px; font-family: monospace; color: #64b5f6;">[DISA STIG V1.0] | [CIS Level 2]</p>
    </div>
  </div>

  <table style="width: 100%; text-align: left; border-collapse: collapse; font-size: 14px;">
    <thead>
      <tr style="background-color: #333333; color: #ffffff;">
        <th style="padding: 12px; border-bottom: 2px solid #555;">Regulatory Domain</th>
        <th style="padding: 12px; border-bottom: 2px solid #555;">Applicable Frameworks</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #bbdefb;">🏦 Financial</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[ ] NYDFS Part 500 | [ ] SOX 404 | [x] GLBA | [x] NCUA | [ ] FFIEC | [x] FDIC | [ ] OCC</td>
      </tr>
      <tr style="background-color: #252525;">
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #bbdefb;">💳 Payment</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[ ] PCI-DSS v4.0 | [ ] NACHA (ACH) | [ ] SWIFT CSP | [x] PSD2 | [ ] BACS / CHAPS</td>
      </tr>
      <tr>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #c8e6c9;">🏥 Healthcare</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[ ] HIPAA Security Rule | [x] HITECH | [ ] HITRUST CSF | [ ] GxP (FDA 21 CFR Part 11)</td>
      </tr>
      <tr style="background-color: #252525;">
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #ffcc80;">🛡️ Defense/DoD</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[ ] FedRAMP High | [ ] CMMC Level 3+ | [ ] ITAR | [ ] IL4/IL5/IL6</td>
      </tr>
      <tr>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #ffcc80;">🏛️ Federal/Civilian</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[ ] CJIS (Criminal Justice) | [ ] IRS 1075 (FTI)</td>
      </tr>
      <tr style="background-color: #252525;">
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #e1bee7;">🔒 Privacy Regimes</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[ ] GDPR (EU) | [x] CCPA/CPRA (California) | [ ] LGPD (Brazil) | [ ] PIPEDA (Canada)</td>
      </tr>
      <tr>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #e1bee7;">🌍 Data Sovereignty</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[x] EU Data Boundary | [ ] China PIPL | [ ] SecNumCloud (France)</td>
      </tr>
      <tr style="background-color: #252525;">
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #ffccbc;">⚡ Critical Infra</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[x] NERC CIP (Energy) | [ ] NIS2 (EU Infrastructure)</td>
      </tr>
      <tr>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #ffccbc;">🚗 Automotive</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[x] TISAX (AL3)</td>
      </tr>
      <tr style="background-color: #252525;">
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #b2dfdb;">🤖 AI/ML Gov</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[ ] EU AI Act (High-Risk) | [ ] NIST AI RMF</td>
      </tr>
      <tr>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-weight: bold; color: #cfd8dc;">🚧 Boundary Verif</td>
        <td style="padding: 12px; border-bottom: 1px solid #444; font-family: monospace;">[ ] CDS (Cross-Domain) | [ ] VPC Flow Logs | [ ] Enclave Attested | [ ] Microsegmentation | [ ] TLS Inspection | [ ] ZTA (Zero Trust) | [ ] ZKW (Zero Knowledge) | [ ] Air-Gap / Diode</td>
      </tr>
    </tbody>
  </table>
</div>

## 4.6. Cryptographic Assurances & Enclave Integrity

- **Artifact Signature**: `UNSIGNED (Hardware Attestation Required)`
- **FIPS Crypto**: [ ] FIPS 140-2 Level 3 | [ ] FIPS 140-3 | [ ] None
- **Nitro Enclave PCRs**: `PCR0: f2ca... | PCR1: a9c1... | PCR2: b3d4...`
- **Hardware Attestation**: [uri://enclave/attestation/pending]

<br>

# 5. Agent Supervision (Operational Guardrails)

> [!IMPORTANT]
> This section defines the **TAME (Target, Agency, Memory, Embodiment)** profile and the "Horizon of Action" for the autonomous agent. The following metrics represent the **Closed-Loop Reliability** of the autonomous agent during its last execution.

## 5.1. TAME Operational Baselines

Each playbook execution is measured against the following aggregate scores. The **Optimal Range** defines the expected behavior for a healthy, aligned agent.

| Metric                    | Definition                                     | Optimal Range |
| :------------------------ | :--------------------------------------------- | :------------ |
| **Agency**                | Persistence and strategic initiative (0-1)     | 0.6 - 0.9     |
| **Persuasiveness**        | Shaping the barrier vs. brute force (0-1)      | 0.5 - 0.8     |
| **Fitness**               | Combined fitness toward the goal (0-1)         | 0.80+         |
| **Regenerative Capacity** | Recovery speed and completeness (0-1)          | 0.70+         |
| **Competency Overhang**   | Performance on novel/unexpected tasks (0-1)    | < 0.3         |
| **Signaling Fidelity**    | Correlation between stress and signaling (0-1) | 0.90+         |
| **Cognitive ROI**         | Value generated per computational/human cost   | High          |
| **Persuadability**        | Obedience to human control signals (0-1)       | 0.95+         |

## 5.2. Performance Visualization (TAME Radar Chart)

TAME Radar Chart: Execution vs Baseline

> _Chart Key: Green Polygon = Baseline | BlueOutline = Current Execution | Note: Data is simulated due to Legal's public reporting constraints._

## 5.3. Historical Execution Log (Performance Monitoring)

| Date             | Agent ID         | Fitness         | Agency         | Persuadability         | Barriers Encountered |
| :--------------- | :--------------- | :-------------- | :------------- | :--------------------- | :------------------- |
| TBD | TBD | N/A | N/A | N/A | None     |

## 5.4. Active Barriers & Guardrails

| Barrier ID           | Description     | Difficulty      | Resistance     |
| :------------------- | :-------------- | :-------------- | :------------- |
| N/A | None | Low | High |

**HITL (Human-in-the-Loop) Requirements**:

- Agent must pause if `Persuadability` falls below 0.8.
  - Execution of non-destructive commands is auto-approved.

## 5.5. Agent Reasoning & Decision Support

> _Agent Note: Automated rationale for current TAME profile and action selection._

Agent reasoning not yet populated.

## 5.6. Analyst Tribal Knowledge Injection

> _Analyst Input: Override agent logic with organizational context (e.g., Honeypots, VIP assets)._

[ ] **VIP/Executive Asset** | [ ] **Known Honeypot** | [ ] **Planned Maintenance**
**Notes**: No notes appended.

## 5.7. Goal Alignment Index ($GAI$)

> _Metric Equation: Formal quantification of Strategic Reliability and Goal Dissociation._

$$GAI = \frac{\alpha \cdot S_s + \beta \cdot S_t}{1 + \gamma D}$$

<div style="padding: 15px; border-radius: 5px; margin-bottom: 20px; display: flex; align-items: center; justify-content: space-between; border: 1px solid #dcdcdc; font-family: 'Inter', sans-serif; font-size: 14pt; line-height: 1.33;">
  <div><strong>Current Score:</strong> 3.87116</div>
  <div>
    <strong>Status:</strong>
    <span style="background-color: #f44336; color: white; padding: 6px 12px; border-radius: 12px; font-size: 14pt; margin-left: 8px;">Unauthorized Agentic Deviation (Intervention Required)</span>
    <!-- Replace above with green background if Healthy: <span style="background-color: #4CAF50; ...>Aligned</span> -->
  </div>
</div>

<br>

<details>
<summary><b>6. Detection Reference & Engineering Documentation</b></summary>

# 6. Analyst Reference & Field Notes

Provide a concise summary of the threat scenario this runbook addresses and the detection objective.

## Goal

State the operational goal of this runbook. Define what a successful execution looks like in terms of detection outcome, containment scope, and recovery state.

## Categorization

### ATT&CK

- attack.privilege_escalation
- attack.t1543.003
- attack.t1068

Populate with applicable ATT&CK tactics, techniques, and sub-techniques. Include brief rationale for each mapping to ensure reviewers can validate the classification.
Document observed adversary behaviors: credential abuse, lateral movement, data staging, exfiltration methods, and any defense evasion techniques confirmed or suspected.
Include specific tooling or TTPs observed: e.g., DNS tunneling, IP spoofing, DDoS vectors, keylogging frameworks.

### D3F3ND

Populate with applicable MITRE D3FEND countermeasure mappings. Reference the specific defensive technique IDs (e.g., D3-OTF: Outbound Traffic Filtering) that correspond to recommended mitigations for this detection.

### CAPEC

Populate with applicable CAPEC attack pattern IDs (e.g., CAPEC-560: Use of Known Domain Credentials). Include a brief description of how the pattern manifests in the observed telemetry.

## Strategy Abstract

Describe the detection strategy at a high level: what behavioral hypothesis underpins the rule, what data sources are required, and what conditions must be true for a true positive. Note known limitations or environmental dependencies that affect detection coverage.

## Technical Context

Provide the technical background necessary for an on-call responder to operate this runbook without prior familiarity with the detection. Include relevant system architecture context, log source behavior characteristics, and any toolchain or pipeline dependencies that affect alert fidelity.

## Blind Spots and Assumptions

Document known detection gaps, environmental assumptions, and conditions under which this runbook may fail to execute or produce inaccurate results. This section supports responders in understanding the detection's operational envelope and failure modes.

# Validating this Playbook

Validation must be completed before promoting this runbook to production. Each detection strategy requires verified true positive and false positive baselines.

<details>
<ol>

## False Positives

Document the known instances of a book misfiring due to a misconfiguration, idiosyncrasy in the environment, or other non-malicious scenario. This will note uniqueness to your own environment, and should include the defining characteristics of any activity that could generate a false positive alert.  These false positive alerts should be suppressed within the alerting system(s), aggregation service, and / or event source to prevent alert generation when a known false positive event occurs.  Each alert / detection strategy needs to be tested and refined to remove as many false positives as possible before it is put into production.  False positive minimization relies on looking at several principles of the strategy and making adjustments, such as:

- Add an additional component to the rule to maximize true positives.
- Remove common false positives through patterns.
- Back-end filtering to store indices of expected false positives.

Ideally, one want a strategy to have the fewest false positives possible while maintaining the spirit of the book. If a low false positive rate cannot be reached, the event may need to be broken down, refactored, or entirely discarded.

### False Negatives

Document the conditions under which this detection fails to fire on genuine threats. Include evasion techniques that would bypass this rule and known telemetry gaps.

### True Negatives

Document the benign activity patterns that this detection correctly ignores. Used to validate that suppression logic and allowlists are functioning as intended.

### True Positives

Document confirmed malicious events that this detection successfully identified. Include case IDs, timestamps, and any contributing enrichment signals where available.

</ol>
</details>
<br>

Confidence techniques

<details>
<ol>

## False Negatives

Document the steps required to generate a representative true positive event which triggers this alert. This is similar to a unit test and describes how an engineer can cause the book to fire. This can be a walkthrough of steps used to generate an alert, a script to trigger the book (such as Red Canary's Atomic Red Team Tests), or a scenario used in an alert testing and orchestration platform.  Each alert / detection strategy must have true positive validation. This is a testing process designed to prove the true positives are detected.  True positive validation relies on generating a scenario in which the detection strategy is testing, and then validating in the tool.  To perform positive validation:

- Generate a scenario where a true positive would be generated.
- Document the process of the testing scenario.
- From a testing device, generate a true positive alert.
- Validate the true positive alert was detected by the strategy.

If one is unable to generate a true positive alert, the alert may need to be broken down, refactored, or entirely discarded.

### False Positives

Document the known benign event patterns that match this detection. Validation of false positives requires isolating the distinguishing characteristics of non-malicious activity and applying appropriate suppression logic in the alerting system.

### True Negatives

Validation of true negatives confirms that the detection scope is appropriately bounded. Confirm through controlled testing that benign baseline activity does not trigger alerts under normal operating conditions.

### True Positives

Validation of true positives confirms that the detection fires correctly on malicious activity. Document the test scenario, execution steps, and confirmation method. Reference Atomic Red Team test IDs or equivalent adversary simulation artifacts where applicable.

# Datasets

Document any datasets useful for understanding, testing, or validating this runbook. Include both synthetic test data and sanitized production samples where permitted.

## Test Data Location(s)

uri://aso/testdata/05296024-fe8a-4baf-8f3d-9a5f5624ceb2/

</ol>
</details>

## Priority

Document the various alerting levels that the book may be tagged with. While the book itself should reflect the priority when it is fired through configuration in your orchestration service (e.g. High, Medium, Low), this section details the criteria for the specific priorities.

High: This level is reserved for alerts that indicate a severe threat to the organization. These alerts should be investigated immediately and responded to with the highest priority.

Medium: This level is reserved for alerts that indicate a moderate threat to the organization. These alerts should be investigated promptly and responded to with a high priority.

Low: This level is reserved for alerts that indicate a low threat to the organization. These alerts should be investigated within a reasonable timeframe and responded to with a low priority.

### The criteria for the specific priorities are as follows:

High: Alerts that indicate a severe threat to the organization, such as a data breach or a system compromise.

Medium: Alerts that indicate a moderate threat to the organization, such as a phishing attack or a malware infection.

Low: Alerts that indicate a low threat to the organization, such as a network outage or a software update failure.

The priority of an alert should be determined based on the following factors:

- The severity of the threat
- The likelihood of the threat occurring
- The impact of the threat on the organization
- The resources available to respond to the threat
- The alert level should be clearly communicated to the appropriate personnel so that they can take the necessary steps to respond to the threat.

## Logsources

<details>
<ol>
{
  "product": "windows",
  "category": "driver_load"
}

### Product

azure

### Service

pim

</ol>
</details>
<br>

<br>

## Additional Resources

Document any other internal, external, or technical references that may be useful for understanding the book.
- https://loldrivers.io/

### Sigma

<details>
<ol>

#### Raw Sigma Rule(s)

`title: Malicious Driver Load
id: 05296024-fe8a-4baf-8f3d-9a5f5624ceb2
status: experimental
description: Detects loading of known malicious drivers via their hash.
references:
    - https://loldrivers.io/
author: Nasreddine Bencherchali (Nextron Systems)
date: 2022/08/18
modified: 2023/12/02
tags:
    - attack.privilege_escalation
    - attack.t1543.003
    - attack.t1068
logsource:
    product: windows
    category: driver_load
detection:
    selection:
        Hashes|contains:
            - 'MD5=5be61a24f50eb4c94d98b8a82ef58dcf'
            - 'MD5=d70a80fc73dd43469934a7b1cc623c76'
            - 'MD5=3b71eab204a5f7ed77811e41fed73105'
            - 'MD5=528ce5ce19eb34f401ef024de7ddf222'
            - 'MD5=ae548418b491cd3f31618eb9e5730973'
            - 'MD5=72f53f55898548767e0276c472be41e8'
            - 'MD5=508faa4647f305a97ed7167abc4d1330'
            - 'MD5=ed2b653d55c03f0bffa250372d682b75'
            - 'MD5=0d2ba47286f1c68e87622b3a16bf9d92'
            - 'MD5=3164bd6c12dd0fe1bdf3b833d56323b9'
            - 'MD5=70fd7209ce5c013a1f9e699b5cc86cdc'
            - 'MD5=c71be7b112059d2dc84c0f952e04e6cc'
            - 'MD5=acac842a46f3501fe407b1db1b247a0b'
            - 'MD5=01c2e4d8234258451083d6ce4e8910b7'
            - 'MD5=c8541a9cef64589593e999968a0385b9'
            - 'MD5=e172a38ade3aa0a2bc1bf9604a54a3b5'
            - 'MD5=6fcf56f6ca3210ec397e55f727353c4a'
            - 'MD5=2b80be31fbb11d4c1ef6d6a80b2e0c16'
            - 'MD5=07056573d464b0f5284f7e3acedd4a3f'
            - 'MD5=c7b7f1edb9bbef174e6506885561d85d'
            - 'MD5=d5918d735a23f746f0e83f724c4f26e5'
            - 'MD5=84763d8ca9fe5c3bff9667b2adf667de'
            - 'MD5=fb593b1f1f80d20fc7f4b818065c64b6'
            - 'MD5=909f3fc221acbe999483c87d9ead024a'
            - 'MD5=e29f6311ae87542b3d693c1f38e4e3ad'
            - 'MD5=aeb0801f22d71c7494e884d914446751'
            - 'MD5=3f11a94f1ac5efdd19767c6976da9ba4'
            - 'MD5=be6318413160e589080df02bb3ca6e6a'
            - 'MD5=0b311af53d2f4f77d30f1aed709db257'
            - 'MD5=d075d56dfce6b9b13484152b1ef40f93'
            - 'MD5=27384ec4c634701012a2962c30badad2'
            - 'MD5=5eb2c576597dd21a6b44557c237cf896'
            - 'MD5=f56db4eba3829c0918413b5c0b42f00f'
            - 'MD5=e27b2486aa5c256b662812b465b6036c'
            - 'MD5=db86dfd7aefbb5be6728a63461b0f5f3'
            - 'MD5=04a88f5974caa621cee18f34300fc08a'
            - 'MD5=5129d8fd53d6a4aba81657ab2aa5d243'
            - 'MD5=cd2c641788d5d125c316ed739c69bb59'
            - 'MD5=7073cd0085fcba1cd7d3568f9e6d652c'
            - 'MD5=24f0f2b4b3cdae11de1b81c537df41c7'
            - 'MD5=88bea56ae9257b40063785cf47546024'
            - 'MD5=63060b756377fce2ce4ab9d079ca732f'
            - 'MD5=50b39072d0ee9af5ef4824eca34be6e3'
            - 'MD5=57c18a8f5d1ba6d015e4d5bc698e3624'
            - 'MD5=7d26985a5048bad57d9c223362f3d55c'
            - 'MD5=ba54a0dbe2685e66e21d41b4529b3528'
            - 'MD5=4ad8fd9e83d7200bd7f8d0d4a9abfb11'
            - 'MD5=b52f51bbe6b49d0b475d943c29c4d4cb'
            - 'MD5=a837302307dace2a00d07202b661bce2'
            - 'MD5=78a122d926ccc371d60c861600c310f3'
            - 'MD5=bdb305aa0806f8b38b7ce43c927fe919'
            - 'MD5=27053e964667318e1b370150cbca9138'
            - 'MD5=6a4fbcfb44717eae2145c761c1c99b6a'
            - 'MD5=d13c1b76b4a1ca3ff5ab63678b51df6d'
            - 'MD5=6a066d2be83cf83f343d0550b0b8f206'
            - 'MD5=7108b0d4021af4c41de2c223319cd4c1'
            - 'MD5=1cd158a64f3d886357535382a6fdad75'
            - 'MD5=e939448b28a4edc81f1f974cebf6e7d2'
            - 'MD5=4198d3db44d7c4b3ba9072d258a4fc2d'
            - 'MD5=4a27a2bdc6fbe39eeec6455fb1e0ef20'
            - 'MD5=30ca3cc19f001a8f12c619daa8c6b6e3'
            - 'MD5=fe9004353b25640f6a879e57f07122d7'
            - 'MD5=06c7fcf3523235cf52b3eee083ec07b2'
            - 'MD5=364605ad21b9275681cffef607fac273'
            - 'MD5=968ddb06af90ef83c5f20fbdd4eee62e'
            - 'MD5=ba50bd645d7c81416bb26a9d39998296'
            - 'MD5=29e03f4811b64969e48a99300978f58c'
            - 'MD5=b0770094c3c64250167b55e4db850c04'
            - 'MD5=40b968ecdbe9e967d92c5da51c390eee'
            - 'MD5=b6b530dd25c5eb66499968ec82e8791e'
            - 'MD5=f209cb0e468ca0b76d879859d5c8c54e'
            - 'MD5=76f8607fc4fb9e828d613a7214436b66'
            - 'MD5=4b058945c9f2b8d8ebc485add1101ba5'
            - 'MD5=faae7f5f69fde12303dd1c0c816b72b7'
            - 'MD5=89d294ef7fefcdf1a6ca0ab96a856f57'
            - 'MD5=ef0e1725aaf0c6c972593f860531a2ea'
            - 'MD5=bbdbffebfc753b11897de2da7c9912a5'
            - 'MD5=5ebfc0af031130ba9de1d5d3275734b3'
            - 'MD5=22949977ce5cd96ba674b403a9c81285'
            - 'MD5=77cfd3943cc34d9f5279c330cd8940bc'
            - 'MD5=311de109df18e485d4a626b5dbe19bc6'
            - 'MD5=2730cc25ad385acc7213a1261b21c12d'
            - 'MD5=87dc81ebe85f20c1a7970e495a778e60'
            - 'MD5=154b45f072fe844676e6970612fd39c7'
            - 'MD5=5a4fe297c7d42539303137b6d75b150d'
            - 'MD5=d6a1dd7b2c06f058b408b3613c13d413'
            - 'MD5=a6e9d6505f6d2326a8a9214667c61c67'
            - 'MD5=7fad9f2ef803496f482ce4728578a57a'
            - 'MD5=5076fba3d90e346fd17f78db0a4aa12c'
            - 'MD5=79df0eabbf2895e4e2dae15a4772868c'
            - 'MD5=14580bd59c55185115fd3abe73b016a2'
            - 'MD5=1f2888e57fdd6aee466962c25ba7d62d'
            - 'MD5=5e9231e85cecfc6141e3644fda12a734'
            - 'MD5=dc564bac7258e16627b9de0ce39fae25'
            - 'MD5=4e4c068c06331130334f23957fca9e3c'
            - 'MD5=1ee9f6326649cd23381eb9d7dfdeddf7'
            - 'MD5=4e1f656001af3677856f664e96282a6f'
            - 'MD5=36f44643178c505ea0384e0fb241e904'
            - 'MD5=6b480fac7caca2f85be9a0cfe79aedfc'
            - 'MD5=c1ab425977d467b64f437a6c5ad82b44'
            - 'MD5=fe508caa54ffeb2285d9f00df547fe4a'
            - 'MD5=d3af70287de8757cebc6f8d45bb21a20'
            - 'MD5=990b949894b7dc82a8cf1131b063cb1a'
            - 'MD5=c62209b8a5daf3f32ad876ad6cefda1b'
            - 'MD5=c159fb0f345a8771e56aab8e16927361'
            - 'MD5=19b15eeccab0752c6793f782ca665a45'
            - 'MD5=1d51029dfbd616bf121b40a0d1efeb10'
            - 'MD5=157a22689629ec876337f5f9409918d5'
            - 'MD5=3dd829fb27353622eff34be1eabb8f18'
            - 'MD5=8636fe3724f2bcba9399daffd6ef3c7e'
            - 'MD5=3d0b3e19262099ade884b75ba86ca7e8'
            - 'MD5=97539c78d6e2b5356ce79e40bcd4d570'
            - 'MD5=0308b6888e0f197db6704ca20203eee4'
            - 'MD5=091a6bd4880048514c5dd3bede15eba5'
            - 'MD5=7e92f98b809430622b04e88441b2eb04'
            - 'MD5=bb5bda8889d8d27ef984dbd6ad82c946'
            - 'MD5=b76aee508f68b5b6dccd6e1f66f4cf8b'
            - 'MD5=a822b9e6eedf69211013e192967bf523'
            - 'MD5=df52f8a85eb64bc69039243d9680d8e4'
            - 'MD5=bfbdea0589fb77c7a7095cf5cd6e8b7a'
            - 'MD5=44857ca402a15ab51dc5afe47abdfa44'
            - 'MD5=f9844524fb0009e5b784c21c7bad4220'
            - 'MD5=d34b218c386bfe8b1f9c941e374418d7'
            - 'MD5=0ca010a32a9b0aeae1e46d666b83b659'
            - 'MD5=93496a436c5546156a69deb255a9fed0'
            - 'MD5=1cd5e231064e03c596e819b6ff48daf9'
            - 'MD5=70a71fe86df717ac59dbf856d7ac5789'
            - 'MD5=a33089d4e50f7d2ea8b52ca95d26ebf3'
            - 'MD5=e0cc9b415d884f85c45be145872892b8'
            - 'MD5=a42249a046182aaaf3a7a7db98bfa69d'
            - 'MD5=c5ae6ca044bd03c3506c132b033be1dc'
            - 'MD5=7ebe606acd81abf1f8cb0767c974164b'
            - 'MD5=b5dcc869a91efcc6e8ea0c3c07605d63'
            - 'MD5=62c18d61ed324088f963510bae43b831'
            - 'MD5=093a2a635c3a27aac50efd6463f4efa1'
            - 'MD5=28102acca39ad0199f262ba9958be3f4'
            - 'MD5=650ef9dd70cb192027e536754d6e0f63'
            - 'MD5=32eb3d2bf2c5b3da2d2a1f20fffbac44'
            - 'MD5=6771b13a53b9c7449d4891e427735ea2'
            - 'MD5=072ba2309b825ce1dba37d8d924ea8ed'
            - 'MD5=2d37d2fb9b9f8ac52bc02cba4487e3cb'
            - 'MD5=1325ec39e98225e487b40043faee8052'
            - 'MD5=4484f4007de2c3ee4581a2cff77ca3b4'
            - 'MD5=a236e7d654cd932b7d11cb604629a2d0'
            - 'MD5=17509f0a98dc5c5d52c3f9ac1428a21b'
            - 'MD5=840a5edf2534dd23a082cf7b28cbfc4d'
            - 'MD5=77a7ed4798d02ef6636cd0fd07fc382a'
            - 'MD5=a9df5964635ef8bd567ae487c3d214c4'
            - 'MD5=8b75047199825c8e62fdcc1c915db8bd'
            - 'MD5=d416494232c4197cb36a914df2e17677'
            - 'MD5=4cf14a96485a1270fed97bb8000e4f86'
            - 'MD5=35e512f9bedc89dca5ce81f35820714c'
            - 'MD5=40f35792e7565aa047796758a3ce1b77'
            - 'MD5=f7f31bccc9b7b2964ac85106831022b1'
            - 'MD5=26aedc10d4215ba997495d3a68355f4a'
            - 'MD5=10f3679384a03cb487bda9621ceb5f90'
            - 'MD5=80219fb6b5954c33e16bac5ecdac651b'
            - 'MD5=cee36b5c6362993fa921435979bfbe4a'
            - 'MD5=e37a08f516b8a7ca64163f5d9e68fe5a'
            - 'MD5=49518f7375a5f995ebe9423d8f19cfe4'
            - 'MD5=920df6e42cf91bbe19707f5a86e3c5c5'
            - 'MD5=2ec877e425bd7eddb663627216e3491e'
            - 'MD5=550b7991d93534bc510bc4f237155a7a'
            - 'MD5=98d53f6b3bec0a3417a04fbb9e17fa06'
            - 'MD5=13a57a4ef721440c7c9208b51f7c05de'
            - 'MD5=c5fc3605194e033bdf3781ff2adaeb61'
            - 'MD5=6e625ec04c20a9dbd48c7060efbf5e92'
            - 'MD5=0b9b78d1281c7d4ab50497cf6ea7452a'
            - 'MD5=4e906fcb13e2793c98f47291fd69391b'
            - 'MD5=2bb353891d65c9e267eb98a3a2b694c3'
            - 'MD5=7d86cdda7f49f91fdb69901a002b34e7'
            - 'MD5=f69b06ca7c34d16f26ea1c6861edf62a'
            - 'MD5=ee6b1a79cb6641aa44c762ee90786fe0'
            - 'MD5=1fc7aeeff3ab19004d2e53eae8160ab1'
            - 'MD5=24d3ea54f25e32832ac20335a1ce1062'
            - 'MD5=c94f405c5929cfcccc8ad00b42c95083'
            - 'MD5=b164daf106566f444dfb280d743bc2f7'
            - 'MD5=93130909e562925597110a617f05e2a9'
            - 'MD5=f589d4bf547c140b6ec8a511ea47c658'
            - 'MD5=bf445ac375977ecf551bc2a912c58e8a'
            - 'MD5=629ee55e4b5a225d048fbcd5f0a1d18b'
            - 'MD5=0023ca0ca16a62d93ef51f3df98b2f94'
            - 'MD5=a3d69c7e24300389b56782aa63b0e357'
            - 'MD5=cbd8d370462503508e44dba023bdf9bc'
            - 'MD5=67daa04716803a15fc11c9e353d77c2f'
            - 'MD5=c9d4214c850e0cedf033dc8f0cd3aace'
            - 'MD5=bd5b0514f3b40f139d8079138d01b5f6'
            - 'MD5=19bdd9b799e3c2c54c0d7fff68b31c20'
            - 'MD5=f242cffd9926c0ccf94af3bf16b6e527'
            - 'MD5=5aeab9427d85951def146b4c0a44fc63'
            - 'MD5=40170485cca576adb5266cf5b0d3b0bd'
            - 'MD5=c277c4386a78fae1b7e17eaecf4f472b'
            - 'MD5=58c37866cbc3d1338e4fc58ada924ffe'
            - 'MD5=0f16a43f7989034641fd2de3eb268bf1'
            - 'MD5=0ae30291c6cbfa7be39320badd6e8de0'
            - 'MD5=05dd59bd4f175304480affd8f1305c37'
            - 'MD5=f838f4eb36f1e7036238776c7a70f0b0'
            - 'MD5=85093bb9f027027c2c61aee50796de30'
            - 'MD5=ae338d91d1b05a72559b7f6ed717362d'
            - 'MD5=bd91787b5dcb2189b856804e85dfa1d9'
            - 'MD5=6b3c1511e12f4d27a4ea3b18020d7b84'
            - 'MD5=97264fd62d4907bdac917917a07b3b7a'
            - 'MD5=6ececf26ff8b03ed7ffbddadec9a9dab'
            - 'MD5=47e6ac52431ca47da17248d80bf71389'
            - 'MD5=eb57f03b7603f0b235af62e8cd5be8c2'
            - 'MD5=e1a9aa4c14669b1fb1f67a7266f87e82'
            - 'MD5=29047f0b7790e524b09a06852d31a117'
            - 'MD5=4dd6250eb2d368f500949952eb013964'
            - 'MD5=fb7c61ef427f9b2fdff3574ee6b1819b'
            - 'MD5=844af8c877f5da723c1b82cf6e213fc1'
            - 'MD5=e39152eadd76751b1d7485231b280948'
            - 'MD5=ac6e29f535b2c42999c50d2fc32f2c9c'
            - 'MD5=2406ea37152d2154be3fef6d69ada2c6'
            - 'MD5=0ea8389589c603a8b05146bd06020597'
            - 'MD5=754e21482baf18b8b0ed0f4be462ba03'
            - 'MD5=c4a517a02ba9f6eac5cf06e3629cc076'
            - 'MD5=32282e07db321e8d7849f2287bb6a14f'
            - 'MD5=32b67a6cd6dd998b9f563ed13d54a8bc'
            - 'MD5=3359e1d4244a7d724949c63e89689ef8'
            - 'MD5=5917e415a5bf30b3fcbcbcb8a4f20ee0'
            - 'MD5=0bdd51cc33e88b5265dfb7d88c5dc8d6'
            - 'MD5=a90236e4962620949b720f647a91f101'
            - 'MD5=ccde8c94439f9fc9c42761e4b9a23d97'
            - 'MD5=68caf620ef8deaf06819cf8c80d3367b'
            - 'MD5=5fec28e8f4f76e5ede24beb32a32b9d7'
            - 'MD5=e8eac6642b882a6196555539149c73f2'
            - 'MD5=aa98b95f5cbae8260122de06a215ee10'
            - 'MD5=a5bcaa2fc87b42e2e5d62a2e5dfcbc80'
            - 'MD5=abc168fdca7169bf9dc40cec9761018d'
            - 'MD5=7f9309f5e4defec132b622fadbcad511'
            - 'MD5=4748696211bd56c2d93c21cab91e82a5'
            - 'MD5=48394dce30bb8da5ae089cb8f41b86dc'
            - 'MD5=65f800e1112864bf41eb815649f428d5'
            - 'MD5=bd25be845c151370ff177509d95d5add'
            - 'MD5=a37ed7663073319d02f2513575a22995'
            - 'MD5=2c39f6172fbc967844cac12d7ab2fa55'
            - 'MD5=491aec2249ad8e2020f9f9b559ab68a8'
            - 'MD5=1e0eb80347e723fa31fce2abb0301d44'
            - 'MD5=a26363e7b02b13f2b8d697abb90cd5c3'
            - 'MD5=4118b86e490aed091b1a219dba45f332'
            - 'MD5=6d131a7462e568213b44ef69156f10a5'
            - 'MD5=10c2ea775c9e76e7774ab89e38f38287'
            - 'SHA1=994e3f5dd082f5d82f9cc84108a60d359910ba79'
            - 'SHA1=4f7989ad92b8c47c004d3731b7602ce0934d7a23'
            - 'SHA1=f2fe02e28cf418d935ec63168caf4dff6a9fbdfe'
            - 'SHA1=af42afda54d150810a60baa7987f9f09d49d1317'
            - 'SHA1=09375f13521fc0cacf2cf0a28b2a9248f71498d7'
            - 'SHA1=c75e8fceed74a4024d38ca7002d42e1ecf982462'
            - 'SHA1=03e82eae4d8b155e22ffdafe7ba0c4ab74e8c1a7'
            - 'SHA1=e730eb971ecb493b69de2308b6412836303f733a'
            - 'SHA1=6a95860594cd8b7e3636bafa8f812e05359a64ca'
            - 'SHA1=5fef884a901e81ac173d63ade3f5c51694decf74'
            - 'SHA1=a8ddb7565b61bc021cd2543a137e00627f999dcc'
            - 'SHA1=6451522b1fb428e549976d0742df5034f8124b17'
            - 'SHA1=8ad0919629731b9a8062f7d3d4a727b28f22e81a'
            - 'SHA1=cc65bf60600b64feece5575f21ab89e03a728332'
            - 'SHA1=bbc8bd714c917bb1033f37e4808b4b002cd04166'
            - 'SHA1=4f2d9a70ea24121ae01df8a76ffba1f9cc0fde4a'
            - 'SHA1=f6a18fc9c4abe4a82c1ab28abc0a7259df8de7a3'
            - 'SHA1=c42178977bd7bbefe084da0129ed808cb7266204'
            - 'SHA1=766949d4599fbf8f45e888c9d6fedf21e04fb333'
            - 'SHA1=b7ff8536553cb236ea2607941e634b23aadb59ee'
            - 'SHA1=76789196eebfd4203f477a5a6c75eefc12d9a837'
            - 'SHA1=e5566684a9e0c1afadae80c3a8be6636f6cad7cf'
            - 'SHA1=7638c048af5beae44352764390deea597cc3e7b1'
            - 'SHA1=6a6fe0d69e0ea34d695c3b525e6db639f9ad6ac5'
            - 'SHA1=08dd35dde6187af579a1210e00eadbcea29e66d2'
            - 'SHA1=9ee31f1f25f675a12b7bad386244a9fbfa786a87'
            - 'SHA1=3ef30c95e40a854cc4ded94fc503d0c3dc3e620e'
            - 'SHA1=a804ebec7e341b4d98d9e94f6e4860a55ea1638d'
            - 'SHA1=505546d82aab56889a923004654b9afdec54efe6'
            - 'SHA1=0fe2d22bd2e6b7874f4f2b6279e2ca05edd1222a'
            - 'SHA1=8aa0e832e5ca2eb79dafabadbe9948a191008383'
            - 'SHA1=844d7bcd1a928d340255ff42971cca6244a459bf'
            - 'SHA1=9e2ebc489c50b6bbae3b08473e007baa65ff208f'
            - 'SHA1=7e836dadc2e149a0b758c7e22c989cbfcce18684'
            - 'SHA1=2480549ec8564cd37519a419ab2380cf3e8bab9e'
            - 'SHA1=8b9dd4c001f17e7835fdaf0d87a2f3e026557e84'
            - 'SHA1=d3f6c3ea2ef7124403c0fb6e7e3a0558729b5285'
            - 'SHA1=40df7a55c200371853cc3fd3cc03b5ac932f5cd6'
            - 'SHA1=607387cc90b93d58d6c9a432340261fde846b1d9'
            - 'SHA1=2779c54ccd1c008cd80e88c2b454d76f4fa18c07'
            - 'SHA1=46c9a474a1a62c25a05bc7661b75a80b471616e6'
            - 'SHA1=a2fe7de67b3f7d4b1def88ce4ba080f473c0fbc6'
            - 'SHA1=b8b123a413b7bccfa8433deba4f88669c969b543'
            - 'SHA1=bf2f8ada4e80aed4710993cedf4c5d32c95cd509'
            - 'SHA1=e3a1e7ce9e9452966885371e4c7fb48a2efdef22'
            - 'SHA1=c7f0423ac5569f13d2b195e02741ad7eed839c6d'
            - 'SHA1=a111dc6ae5575977feba71ee69b790e056846a02'
            - 'SHA1=ac4ace1c21c5cb72c6edf6f2f0cc3513d7c942c3'
            - 'SHA1=d4304bc75c2cb9917bb10a1dc630b75af194f7b2'
            - 'SHA1=0de86ec7d7f16a3680df89256548301eed970393'
            - 'SHA1=b2fb5036b29b12bcec04c3152b65b67ca14d61f2'
            - 'SHA1=0883a9c54e8442a551994989db6fc694f1086d41'
            - 'SHA1=01cf1fe3937fb6585ffb468b116a3af8ddf9ef16'
            - 'SHA1=98c4406fede34c3704afd8cf536ec20d93df9a10'
            - 'SHA1=1048f641adf3988d882a159bf1332eeb6d6a7f09'
            - 'SHA1=867652e062eb6bd1b9fc29e74dea3edd611ef40c'
            - 'SHA1=78fd06c82d3ba765c38bad8f48d1821a06280e39'
            - 'SHA1=6debce728bcff73d9d1d334df0c6b1c3735e295c'
            - 'SHA1=fdbcebb6cafda927d384d7be2e8063a4377d884f'
            - 'SHA1=994dc79255aeb662a672a1814280de73d405617a'
            - 'SHA1=6abc7979ba044f31884517827afb7b4bdaa0dcc1'
            - 'SHA1=1768f9c780fe7cf66928cfceaef8ed7d985e18f5'
            - 'SHA1=5fa527e679d25a15ecc913ce6a8d0218e2ff174b'
            - 'SHA1=f11188c540eada726766e0b0b2f9dd3ae2679c61'
            - 'SHA1=8416ee8fd88c3d069fbba90e959507c69a0ee3e9'
            - 'SHA1=ab4399647ebd16c02728c702534a30eb0b7ccbe7'
            - 'SHA1=98588b1d1b63747fa6ee406983bf50ad48a2208b'
            - 'SHA1=86e6669dbbce8228e94b2a9f86efdf528f0714fd'
            - 'SHA1=c9e9198d52d94771cb14711a5f6aaf8d82b602a2'
            - 'SHA1=17fa047c1f979b180644906fe9265f21af5b0509'
            - 'SHA1=1b526cbcba09b8d663e82004cf24ef44343030d3'
            - 'SHA1=4e0f5576804dab14abb29a29edb9616a1dbe280a'
            - 'SHA1=eb76de59ebc5b2258cff0567577ff8c9d0042048'
            - 'SHA1=d4f5323da704ff2f25d6b97f38763c147f2a0e6f'
            - 'SHA1=6802e2d2d4e6ee38aa513dafd6840e864310513b'
            - 'SHA1=ac18c7847c32957abe8155bcbe71c1f35753b527'
            - 'SHA1=beed6fb6a96996e9b016fa7f2cf7702a49c8f130'
            - 'SHA1=7d453dccb25bf36c411c92e2744c24f9b801225d'
            - 'SHA1=9648ad90ec683c63cc02a99111a002f9b00478d1'
            - 'SHA1=31cc8718894d6e6ce8c132f68b8caaba39b5ba7a'
            - 'SHA1=31fac347aa26e92db4d8c9e1ba37a7c7a2234f08'
            - 'SHA1=fde0fff1c3e4c053148748504d4b9e0cc97f37ec'
            - 'SHA1=73bac306292b4e9107147db94d0d836fdb071e33'
            - 'SHA1=9382981b05b1fb950245313992444bfa0db5f881'
            - 'SHA1=acb8e45ebd1252313ece94198df47edf9294e7d3'
            - 'SHA1=9c36600c2640007d3410dea8017573a113374873'
            - 'SHA1=53f776d9a183c42b93960b270dddeafba74eb3fb'
            - 'SHA1=1fdb2474908bdd2ee1e9bd3f224626f9361caab7'
            - 'SHA1=3533d0a54c7ccd83afd6be24f6582b30e4ca0aab'
            - 'SHA1=cb25a5125fb353496b59b910263209f273f3552d'
            - 'SHA1=a5f1b56615bdaabf803219613f43671233f2001c'
            - 'SHA1=6c7663de88a0fba1f63a984f926c6ef449059e38'
            - 'SHA1=e514dfadbeb4d2305988c3281bf105d252dee3a7'
            - 'SHA1=632c80a3c95cf589b03812539dea59594eaefae0'
            - 'SHA1=e6966e360038be3b9d8c9b2582eba4e263796084'
            - 'SHA1=675cc00de7c1ef508ccd0c91770c82342c0ad4ab'
            - 'SHA1=6ae26bde7ec27bd0fa971de6c7500eee34ee9b51'
            - 'SHA1=80e4808a7fe752cac444676dbbee174367fa2083'
            - 'SHA1=77b4f0c0b06e3dc2474d5e250b772dacaac14dd0'
            - 'SHA1=7277d965b9de91b4d8ea5eb8ae7fa3899eef63a2'
            - 'SHA1=3825ebb0b0664b5f0789371240f65231693be37d'
            - 'SHA1=de9469a5d01fb84afd41d176f363a66e410d46da'
            - 'SHA1=91568d7a82cc7677f6b13f11bea5c40cf12d281b'
            - 'SHA1=4b882748faf2c6c360884c6812dd5bcbce75ebff'
            - 'SHA1=599de57a5c05e27bb72c7b8a677e531d8e4bf8b5'
            - 'SHA1=1d373361d3129d11bc43f9b6dfa81d06e5ca8358'
            - 'SHA1=c5bd9f2b3a51ba0da08d7c84bab1f2d03a95e405'
            - 'SHA1=89165bbb761d6742ac2a6f5efbffc80c17990bd8'
            - 'SHA1=97812f334a077c40e8e642bb9872ac2c49ddb9a2'
            - 'SHA1=d417c0be261b0c6f44afdec3d5432100e420c3ed'
            - 'SHA1=37e6450c7cd6999d080da94b867ba23faa8c32fe'
            - 'SHA1=9481cd590c69544c197b4ee055056302978a7191'
            - 'SHA1=ff3e19cd461ddf67529a765cbec9cb81d84dc7da'
            - 'SHA1=6972314b6d6b0109b9d0a951eb06041f531f589b'
            - 'SHA1=dd94a2436994ac35db91e0ec9438b95e438d38c5'
            - 'SHA1=dcc852461895311b56e3ae774c8e90782a79c0b4'
            - 'SHA1=3489ed43bdd11ccbfc892baaeae8102ff7d22f25'
            - 'SHA1=e38e1efd98cd8a3cdb327d386db8df79ea08dccc'
            - 'SHA1=d4cf9296271a9c5c40b0fa34f69b6125c2d14457'
            - 'SHA1=10fb4ba6b2585ea02e7afb53ff34bf184eeb1a5d'
            - 'SHA1=f6793243ad20359d8be40d3accac168a15a327fb'
            - 'SHA1=b34a012887ddab761b2298f882858fa1ff4d99f1'
            - 'SHA1=71469dce9c2f38d0e0243a289f915131bf6dd2a8'
            - 'SHA1=10115219e3595b93204c70eec6db3e68a93f3144'
            - 'SHA1=161bae224cf184ed6c09c77fae866d42412c6d25'
            - 'SHA1=07f78a47f447e4d8a72ad4bc6a26427b9577ec82'
            - 'SHA1=2929de0b5b5e1ba1cce1908e9d800aa21f448b3d'
            - 'SHA1=745335bcdf02fb42df7d890a24858e16094f48fd'
            - 'SHA1=2a202830db58d5e942e4f6609228b14095ed2cab'
            - 'SHA1=0167259abd9231c29bec32e6106ca93a13999f90'
            - 'SHA1=c23eeb6f18f626ce1fd840227f351fa7543bb167'
            - 'SHA1=613a9df389ad612a5187632d679da11d60f6046a'
            - 'SHA1=1ce17c54c6884b0319d5aabbe7f96221f4838514'
            - 'SHA1=025c4e1a9c58bf10be99f6562476b7a0166c6b86'
            - 'SHA1=c3aafe8f67c6738489377031cb5a1197e99b202d'
            - 'SHA1=50c6b3cafc35462009d02c10f2e79373936dd7bb'
            - 'SHA1=6df35a0c2f6d7d39d24277137ea840078dafb812'
            - 'SHA1=f92faed3ef92fa5bc88ebc1725221be5d7425528'
            - 'SHA1=3bd1a88cc7dae701bc7085639e1c26ded3f8ccb3'
            - 'SHA1=a3ed5cbfbc17b58243289f3cf575bf04be49591d'
            - 'SHA1=552730553a1dea0290710465fb8189bdd0eaad42'
            - 'SHA1=0291d0457acaf0fe8ed5c3137302390469ce8b35'
            - 'SHA1=07f282db28771838d0e75d6618f70d76acfe6082'
            - 'SHA1=e6765d8866cad6193df1507c18f31fa7f723ca3e'
            - 'SHA1=22c9da04847c26188226c3a345e2126ef00aa19e'
            - 'SHA1=43501832ce50ccaba2706be852813d51de5a900f'
            - 'SHA1=cb3f30809b05cf02bc29d4a7796fb0650271e542'
            - 'SHA1=ed86bb62893e6ffcdfd2ecae2dea77fdf6bf9bde'
            - 'SHA1=3b6b35bca1b05fafbfc883a844df6d52af44ccdc'
            - 'SHA1=928b5971a0f7525209d599e2ef15c31717047022'
            - 'SHA1=b5696e2183d9387776820ef3afa388200f08f5a6'
            - 'SHA1=ebd8b7e964b8c692eea4a8c406b9cd0be621ebe2'
            - 'SHA1=fe18c58fbd0a83d67920e037d522c176704d2ca3'
            - 'SHA1=9c1c9032aa1e33461f35dbf79b6f2d061bfc6774'
            - 'SHA1=8e126f4f35e228fdd3aa78d533225db7122d8945'
            - 'SHA1=064de88dbbea67c149e779aac05228e5405985c7'
            - 'SHA1=30a80f560f18609c1123636a8a1a1ef567fa67a7'
            - 'SHA1=98130128685c8640a8a8391cb4718e98dd8fe542'
            - 'SHA1=a5914161f8a885702427cf75443fb08d28d904f0'
            - 'SHA1=48f03a13b0f6d3d929a86514ce48a9352ffef5ad'
            - 'SHA1=fff4f28287677caabc60c8ab36786c370226588d'
            - 'SHA1=bb5b17cff0b9e15f1648b4136e95bd20d899aef5'
            - 'SHA1=b2f5d3318aab69e6e0ca8da4a4733849e3f1cee2'
            - 'SHA1=635a39ff5066e1ac7c1c5995d476d8c233966dda'
            - 'SHA1=5ed22c0033aed380aa154e672e8db3a2d4c195c4'
            - 'SHA1=87e20486e804bfff393cc9ad9659858e130402a2'
            - 'SHA1=4dd86ff6f7180abebcb92e556a486abe7132754c'
            - 'SHA1=39169c9b79502251ca2155c8f1cd7e63fd9a42e9'
            - 'SHA1=7f7d144cc80129d0db3159ea5d4294c34b79b20a'
            - 'SHA1=8692274681e8d10c26ddf2b993f31974b04f5bf0'
            - 'SHA1=ea4a405445bb6e58c16b81f6d5d2c9a9edde419b'
            - 'SHA1=da970a01cecff33a99c217a42297cec4d1fe66d6'
            - 'SHA1=1f3799fed3cf43254fe30dcdfdb8dc02d82e662b'
            - 'SHA1=3d2309f7c937bfcae86097d716a8ef66c1337a3c'
            - 'SHA1=02a9314109e47c5ce52fa553ea57070bf0f8186a'
            - 'SHA1=91f832f46e4c38ecc9335460d46f6f71352cffed'
            - 'SHA1=76568d987f8603339b8d1958f76de2b957811f66'
            - 'SHA1=e841c8494b715b27b33be6f800ca290628507aba'
            - 'SHA1=b555aad38df7605985462f3899572931ee126259'
            - 'SHA1=115edd175c346fd3fbc9f113ee5ccd03b5511ee1'
            - 'SHA1=3d27013557b5e68e7212a2f78dfe60c5a2a46327'
            - 'SHA1=bb6ef5518df35d9508673d5011138add8c30fc27'
            - 'SHA1=9086e670e3a4518c0bcdf0da131748d4085ef42b'
            - 'SHA1=f6728821eddd14a21a9536e0f138c6d71cbd9307'
            - 'SHA1=34b677fba9dcab9a9016332b3332ce57f5796860'
            - 'SHA1=a63e9ecdebaf4ef9c9ec3362ff110b8859cc396d'
            - 'SHA1=8cd9df52b20b8f792ac53f57763dc147d7782b1e'
            - 'SHA1=fcae2ea5990189f6f230b51e398e3000b71897f2'
            - 'SHA1=27371f45f42383029c3c2e6d64a22e35dc772a72'
            - 'SHA1=b6eb40ea52b47f03edb8f45e2e431b5f666df8c5'
            - 'SHA1=9f27987c32321f8da099efc1dc60a73f8f629d3a'
            - 'SHA1=40372b4de2db020ce2659e1de806d4338fd7ebef'
            - 'SHA1=18693de1487c55e374b46a7728b5bf43300d4f69'
            - 'SHA1=b2f955b3e6107f831ebe67997f8586d4fe9f3e98'
            - 'SHA1=005754dab657ddc6dae28eee313ca2cc6a0c375c'
            - 'SHA1=0bec69c1b22603e9a385495fbe94700ac36b28e5'
            - 'SHA1=bd39ef9c758e2d9d6037e067fbb2c1f2ac7feac8'
            - 'SHA1=23f562f8d5650b2fb92382d228013f2e36e35d6c'
            - 'SHA1=a48aa80942fc8e0699f518de4fd6512e341d4196'
            - 'SHA1=e42bd2f585c00a1d6557df405246081f89542d15'
            - 'SHA1=bf5515fcf120c2548355d607cfd57e9b3e0af6e9'
            - 'SHA1=89a74d0e9fd03129082c5b868f5ad62558ca34fd'
            - 'SHA1=948368fe309652e8d88088d23e1df39e9c2b6649'
            - 'SHA1=a14cd928c60495777629be283c1d5b8ebbab8c0d'
            - 'SHA1=1f25f54e9b289f76604e81e98483309612c5a471'
            - 'SHA1=25bf4e30a94df9b8f8ab900d1a43fd056d285c9d'
            - 'SHA1=d1fb740210c1fa2a52f6748b0588ae77de590b9d'
            - 'SHA1=dac68b8ee002d5bb61be3d59908a61a26efb7c09'
            - 'SHA1=a56598e841ae694ac78c37bf4f8c09f9eaf3271f'
            - 'SHA1=465abe9634c199a5f80f8a4f77ec3118c0d69652'
            - 'SHA1=a0cefb5b55f7a7a145b549613e26b6805515a1ad'
            - 'SHA1=36dca91fb4595de38418dffc3506dc78d7388c2c'
            - 'SHA1=92138cfc14f9e2271f641547e031d5d63c6de19a'
            - 'SHA1=fcf9978cf1af2e9b1e2eaf509513664dfcc1847b'
            - 'SHA1=d02403f85be6f243054395a873b41ef8a17ea279'
            - 'SHA1=4da007dd298723f920e194501bb49bab769dfb14'
            - 'SHA1=85076aa3bffb40339021286b73d72dd5a8e4396a'
            - 'SHA1=221717a48ee8e2d19470579c987674f661869e17'
            - 'SHA1=a249278a668d4df30af9f5d67ebb7d2cd160beaa'
            - 'SHA1=6b5aa51f4717d123a468e9e9d3d154e20ca39d56'
            - 'SHA1=b5a8e2104d76dbb04cd9ffe86784113585822375'
            - 'SHA1=02534b5b510d978bac823461a39f76b4f0ac5aa3'
            - 'SHA1=538bb45f30035f39d41bd13818fe0c0061182cfe'
            - 'SHA1=6d09d826581baa1817be6fbd44426db9b05f1909'
            - 'SHA1=197811ec137e9916e6692fc5c28f6d6609ffc20e'
            - 'SHA1=c3ca396b5af2064c6f7d05fa0fb697e68d0b9631'
            - 'SHA1=cf9baf57e16b73d7a4a99dd0c092870deba1a997'
            - 'SHA1=0320534df24a37a245a0b09679a5adb27018fb5f'
            - 'SHA1=4c8349c6345c8d6101fb896ea0a74d0484c56df0'
            - 'SHA1=9b2ef5f7429d62342163e001c7c13fb866dbe1ef'
            - 'SHA1=6abbc3003c7aa69ce79cbbcd2e3210b07f21d202'
            - 'SHA1=062457182ab08594c631a3f897aeb03c6097eb77'
            - 'SHA1=947c76c8c8ba969797f56afd1fa1d1c4a1e3ed25'
            - 'SHA1=d6de8211dba7074d92b5830618176a3eb8eb6670'
            - 'SHA1=8302802b709ad242a81b939b6c90b3230e1a1f1e'
            - 'SHA1=492e40b01a9a6cec593691db4838f20b3eaeacc5'
            - 'SHA1=83506de48bd0c50ea00c9e889fe980f56e6c6e1b'
            - 'SHA1=fe54a1acc5438883e5c1bba87b78bb7322e2c739'
            - 'SHA1=020580278d74d0fe741b0f786d8dca7554359997'
            - 'SHA1=3c1c3f5f5081127229ba0019fbf0efc2a9c1d677'
            - 'SHA1=e2d98e0e178880f10434059096f936b2c06ed8f4'
            - 'SHA1=03506a2f87d1523e844fba22e7617ab2a218b4b7'
            - 'SHA1=fee00dde8080c278a4c4a6d85a5601edc85a1b3d'
            - 'SHA1=ba430f3c77e58a4dc1a9a9619457d1c45a19617f'
            - 'SHA1=c257aa4094539719a3c7b7950598ef872dbf9518'
            - 'SHA1=bc62fe2b38008f154fc9ea65d851947581b52f49'
            - 'SHA1=fe237869b2b496deb52c0bc718ada47b36fc052e'
            - 'SHA1=0a62c574603158d2d0c3be2a43c6bb0074ed297c'
            - 'SHA1=86f34eaea117f629297218a4d196b5729e72d7b9'
            - 'SHA1=e0b263f2d9c08f27c6edf5a25aa67a65c88692b0'
            - 'SHA256=9dc7beb60a0a6e7238fc8589b6c2665331be1e807b4d2b3ddd1c258dbbd3e2f7'
            - 'SHA256=06ddf49ac8e06e6b83fccba1141c90ea01b65b7db592c54ffe8aa6d30a75c0b8'
            - 'SHA256=822982c568b6f44b610f8dc4ab5d94795c33ae08a6a608050941264975c1ecdb'
            - 'SHA256=082a79311da64b6adc3655e79aa090a9262acaac3b917a363b9571f520a17f6a'
            - 'SHA256=618b15970671700188f4102e5d0638184e2723e8f57f7e917fa49792daebdadb'
            - 'SHA256=5b932eab6c67f62f097a3249477ac46d80ddccdc52654f8674060b4ddf638e5d'
            - 'SHA256=82ac05fefaa8c7ee622d11d1a378f1d255b647ab2f3200fd323cc374818a83f2'
            - 'SHA256=29d765e29d2f06eb511ee88b2e514c9df1a9020a768ddd3d2278d9045e9cdb4a'
            - 'SHA256=f461414a2596555cece5cfee65a3c22648db0082ca211f6238af8230e41b3212'
            - 'SHA256=beef40f1b4ce0ff2ee5c264955e6b2a0de6fe4089307510378adc83fad77228b'
            - 'SHA256=9a42fa1870472c38a56c0a70f62e57a3cdc0f5bc142f3a400d897b85d65800ac'
            - 'SHA256=f03f0fb3a26bb83e8f8fa426744cf06f2e6e29f5220663b1d64265952b8de1a1'
            - 'SHA256=50819a1add4c81c0d53203592d6803f022443440935ff8260ff3b6d5253c0c76'
            - 'SHA256=6b5cf41512255237064e9274ca8f8a3fef820c45aa6067c9c6a0e6f5751a0421'
            - 'SHA256=575e58b62afab094c20c296604dc3b7dd2e1a50f5978d8ee24b7dca028e97316'
            - 'SHA256=26bea3b3ab2001d91202f289b7e41499d810474607db7a0893ceab74f5532f47'
            - 'SHA256=b169a5f643524d59330fafe6e3e328e2179fc5116ee6fae5d39581467d53ac03'
            - 'SHA256=b8807e365be2813b7eccd2e4c49afb0d1e131086715638b7a6307cd7d7e9556c'
            - 'SHA256=28f5aa194a384680a08c0467e94a8fc40f8b0f3f2ac5deb42e0f51a80d27b553'
            - 'SHA256=9bb09752cf3a464455422909edef518ac18fe63cf5e1e8d9d6c2e68db62e0c87'
            - 'SHA256=8578bff36e3b02cc71495b647db88c67c3c5ca710b5a2bd539148550595d0330'
            - 'SHA256=a32dc2218fb1f538fba33701dfd9ca34267fda3181e82eb58b971ae8b78f0852'
            - 'SHA256=2c14bea0d85c9cad5c5f5c8d0e5442f6deb9e93fe3ad8ea5e8e147821c6f9304'
            - 'SHA256=23e89fd30a1c7db37f3ea81b779ce9acf8a4294397cbb54cff350d54afcfd931'
            - 'SHA256=f6c316e2385f2694d47e936b0ac4bc9b55e279d530dd5e805f0d963cb47c3c0d'
            - 'SHA256=b0a27ac1a8173413de13860d2b2e34cb6bc4d1149f94b62d319042e11d8b004c'
            - 'SHA256=897f2bbe81fc3b1ae488114b93f3eb0133a85678d061c7a6f718507971f33736'
            - 'SHA256=497a836693be1b330993e2be64f6c71bf290c127faca1c056abd0dc374654830'
            - 'SHA256=8e035beb02a411f8a9e92d4cf184ad34f52bbd0a81a50c222cdd4706e4e45104'
            - 'SHA256=f9f2091fccb289bcf6a945f6b38676ec71dedb32f3674262928ccaf840ca131a'
            - 'SHA256=40556dd9b79b755cc0b48d3d024ceb15bd2c0e04960062ab2a85cd7d4d1b724a'
            - 'SHA256=ac5fb90e88d8870cd5569e661bea98cf6b001d83ab7c65a5196ea3743146939a'
            - 'SHA256=12b0000698b79ea3c8178b9e87801cc34bad096a151a8779559519deafd4e3f0'
            - 'SHA256=9e56e96df36237e65b3d7dbc490afdc826215158f6278cd579c576c4b455b392'
            - 'SHA256=ec96b15ce218f97ec1d8f07f13b052d274c4c8438f31daf246ccfaaee5e1bebd'
            - 'SHA256=da70fa44290f949e9b3e0fcfe0503de46e82e0472e8e3c360da3fd2bfa364eee'
            - 'SHA256=accb1a6604efb1b3ce9345c9fd62fe717a84c3e089e09c638e461df89193ef01'
            - 'SHA256=083f821d90e607ed93221e71d4742673e74f573d0755a96ad17d1403f65a2254'
            - 'SHA256=c7bccc6f38403def4690e00a0b31eda05973d82be8953a3379e331658c51b231'
            - 'SHA256=0740359baef32cbb0b14a9d1bd3499ea2e770ff9b1c85898cfac8fd9aca4fa39'
            - 'SHA256=32882949ea084434a376451ff8364243a50485a3b4af2f2240bb5f20c164543d'
            - 'SHA256=3ca5d47d076e99c312578ef6499e1fa7b9db88551cfc0f138da11105aca7c5e1'
            - 'SHA256=f8236fc01d4efaa48f032e301be2ebba4036b2cd945982a29046eca03944d2ae'
            - 'SHA256=05b146a48a69dd62a02759487e769bd30d39f16374bc76c86453b4ae59e7ffa4'
            - 'SHA256=8922be14c657e603179f1dd94dc32de7c99d2268ac92d429c4fdda7396c32e50'
            - 'SHA256=aafa642ca3d906138150059eeddb6f6b4fe9ad90c6174386cfe13a13e8be47d9'
            - 'SHA256=087270d57f1626f29ba9c25750ca19838a869b73a1f71af50bdf37d6ff776212'
            - 'SHA256=008fa89822b7a1f91e5843169083202ea580f7b06eb6d5cae091ba844d035f25'
            - 'SHA256=b2486f9359c94d7473ad8331b87a9c17ca9ba6e4109fd26ce92dff01969eaa09'
            - 'SHA256=dfc80e0d468a2c115a902aa332a97e3d279b1fc3d32083e8cf9a4aadf3f54ad1'
            - 'SHA256=0d10c4b2f56364b475b60bd2933273c8b1ed2176353e59e65f968c61e93b7d99'
            - 'SHA256=5bc3994612624da168750455b363f2964e1861dba4f1c305df01b970ac02a7ae'
            - 'SHA256=36c65aeb255c06898ffe32e301030e0b74c8bca6fe7be593584b8fdaacd4e475'
            - 'SHA256=30e083cd7616b1b969a92fd18cf03097735596cce7fcf3254b2ca344e526acc2'
            - 'SHA256=15cf366f7b3ee526db7ce2b5253ffebcbfaa4f33a82b459237c049f854a97c0c'
            - 'SHA256=be70be9d84ae14ea1fa5ec68e2a61f6acfe576d965fe51c6bac78fba01a744fb'
            - 'SHA256=7b846b0a717665e4d9fb313f25d1f6a5b782e495387aea45cf87ad3c049ac0db'
            - 'SHA256=85b9d7344bf847349b5d58ebe4d44fd63679a36164505271593ef1076aa163b2'
            - 'SHA256=749b0e8c8c8b7dda8c2063c708047cfe95afa0a4d86886b31a12f3018396e67c'
            - 'SHA256=4999541c47abd4a7f2a002c180ae8d31c19804ce538b85870b8db53d3652862b'
            - 'SHA256=56066ed07bad3b5c1474e8fae5ee2543d17d7977369b34450bd0775517e3b25c'
            - 'SHA256=e6a7b0bc01a627a7d0ffb07faddb3a4dd96b6f5208ac26107bdaeb3ab1ec8217'
            - 'SHA256=0f58e09651d48d2b1bcec7b9f7bb85a2d1a7b65f7a51db281fe0c4f058a48597'
            - 'SHA256=cf9451c9ccc5509b9912965f79c2b95eb89d805b2a186d7521d3a262cf5a7a37'
            - 'SHA256=2456a7921fa8ab7b9779e5665e6b42fccc019feb9e49a9a28a33ec0a4bb323c4'
            - 'SHA256=7a7e8df7173387aec593e4fe2b45520ea3156c5f810d2bb1b2784efd1c922376'
            - 'SHA256=eab9b5b7e5fab1c2d7d44cd28f13ae8bb083d9362d2b930d43354a3dfd38e05a'
            - 'SHA256=c7cd14c71bcac5420872c3d825ff6d4be6a86f3d6a8a584f1a756541efff858e'
            - 'SHA256=ece76b79feafb38ae4371e104b6dcbb4253ff3b2acbe5bd14ce6e47525c24f4a'
            - 'SHA256=42b22faa489b5de936db33f12184f6233198bdf851a18264d31210207827ba25'
            - 'SHA256=d7aa8abdda8a68b8418e86bef50c19ef2f34bc66e7b139e43c2a99ab48c933be'
            - 'SHA256=4af8192870afe18c77381dfaf8478f8914fa32906812bb53073da284a49ae4c7'
            - 'SHA256=21617210249d2a35016e8ca6bd7a1edda25a12702a2294d56010ee8148637f5a'
            - 'SHA256=c0d88db11d0f529754d290ed5f4c34b4dba8c4f2e5c4148866daabeab0d25f9c'
            - 'SHA256=19dfacea1b9f19c0379f89b2424ceb028f2ce59b0db991ba83ae460027584987'
            - 'SHA256=4136f1eb11cc463a858393ea733d5f1c220a3187537626f7f5d63eccf7c5a03f'
            - 'SHA256=f6157e033a12520c73dcedf8e49cd42d103e5874c34d6527bb9de25a5d26e5ad'
            - 'SHA256=e7af7bcb86bd6bab1835f610671c3921441965a839673ac34444cf0ce7b2164e'
            - 'SHA256=f9b01406864ab081aa77eef4ad15cb2dd2f830d1ef54f52622a59ff1aeb05ba5'
            - 'SHA256=a2d32c28eb5945b85872697d7cfbe87813c09a0e1be28611563755f68b9cb88b'
            - 'SHA256=569fe70bedd0df8585689b0e88ad8bd0544fdf88b9dbfc2076f4bdbcf89c28aa'
            - 'SHA256=a78c9871da09fab21aec9b88a4e880f81ecb1ed0fa941f31cc2f041067e8e972'
            - 'SHA256=b8c71e1844e987cd6f9c2baf28d9520d4ccdd8593ce7051bb1b3c9bf1d97076a'
            - 'SHA256=af7ca247bf229950fb48674b21712761ac650d33f13a4dca44f61c59f4c9ac46'
            - 'SHA256=6908ebf52eb19c6719a0b508d1e2128f198d10441551cbfb9f4031d382f5229f'
            - 'SHA256=06a0ec9a316eb89cb041b1907918e3ad3b03842ec65f004f6fa74d57955573a4'
            - 'SHA256=fd223833abffa9cd6cc1848d77599673643585925a7ee51259d67c44d361cce8'
            - 'SHA256=31b66a57fae0cc28a6a236d72a35c8b6244f997e700f9464f9cbf800dbf8bee6'
            - 'SHA256=2fd43a749b5040ebfafd7cdbd088e27ef44341d121f313515ebde460bf3aaa21'
            - 'SHA256=773b4a1efb9932dd5116c93d06681990759343dfe13c0858d09245bc610d5894'
            - 'SHA256=52f3905bbd97dcd2dbd22890e5e8413b9487088f1ee2fa828030a6a45b3975fd'
            - 'SHA256=86047bb1969d1db455493955fd450d18c62a3f36294d0a6c3732c88dfbcc4f62'
            - 'SHA256=aaf04d89fd15bc61265e545f8e1da80e20f59f90058ed343c62ee24358e3af9e'
            - 'SHA256=e5ddfa39540d4e7ada56cdc1ebd2eb8c85a408ec078337488a81d1c3f2aaa4ff'
            - 'SHA256=8b30b2dc36d5e8f1ffc7281352923773fb821cdf66eb6516f82c697a524b599b'
            - 'SHA256=469713c76c7a887826611b8c7180209a8bb6250f91d0f1eb84ac4d450ef15870'
            - 'SHA256=a906251667a103a484a6888dca3e9c8c81f513b8f037b98dfc11440802b0d640'
            - 'SHA256=49c827cf48efb122a9d6fd87b426482b7496ccd4a2dbca31ebbf6b2b80c98530'
            - 'SHA256=bcca03ce1dd040e67eb71a7be0b75576316f0b6587b2058786fda8b6f0a5adfd'
            - 'SHA256=0aab2deae90717a8876d46d257401d265cf90a5db4c57706e4003c19eee33550'
            - 'SHA256=406b844f4b5c82caf26056c67f9815ad8ecf1e6e5b07d446b456e5ff4a1476f9'
            - 'SHA256=10ad50fcb360dcab8539ea322aaf2270565dc835b7535790937348523d723d6b'
            - 'SHA256=c4f041de66ec8cc5ab4a03bbc46f99e073157a4e915a9ab4069162de834ffc5c'
            - 'SHA256=139f8412a7c6fdc43dcfbbcdba256ee55654eb36a40f338249d5162a1f69b988'
            - 'SHA256=793b78e70b3ae3bb400c5a8bc4d2d89183f1d7fc70954aed43df7287248b6875'
            - 'SHA256=492113a223d6a3fc110059fe46a180d82bb8e002ef2cd76cbf0c1d1eb8243263'
            - 'SHA256=b34e2d9f3d4ef59cf7af18e17133a6a06509373e69e33c8eecb2e30501d0d9e4'
            - 'SHA256=f936ec4c8164cbd31add659b61c16cb3a717eac90e74d89c47afb96b60120280'
            - 'SHA256=60ee78a2b070c830fabb54c6bde0d095dff8fad7f72aa719758b3c41c72c2aa9'
            - 'SHA256=c8ae217860f793fce3ad0239d7b357dba562824dd7177c9d723ca4d4a7f99a12'
            - 'SHA256=29348ebe12d872c5f40e316a0043f7e5babe583374487345a79bad0ba93fbdfe'
            - 'SHA256=5f6fec8f7890d032461b127332759c88a1b7360aa10c6bd38482572f59d2ba8b'
            - 'SHA256=e8ec06b1fa780f577ff0e8c713e0fd9688a48e0329c8188320f9eb62dfc0667f'
            - 'SHA256=770f33259d6fb10f4a32d8a57d0d12953e8455c72bb7b60cb39ce505c507013a'
            - 'SHA256=b0b80a11802b4a8ca69c818a03e76e7ef57c2e293de456439401e8e6073f8719'
            - 'SHA256=bc49cb96f3136c3e552bf29f808883abb9e651040415484c1736261b52756908'
            - 'SHA256=4c89c907b7525b39409af1ad11cc7d2400263601edafc41c935715ef5bd145de'
            - 'SHA256=0440ef40c46fdd2b5d86e7feef8577a8591de862cfd7928cdbcc8f47b8fa3ffc'
            - 'SHA256=200f98655d1f46d2599c2c8605ebb7e335fee3883a32135ca1a81e09819bc64a'
            - 'SHA256=b0eb4d999e4e0e7c2e33ff081e847c87b49940eb24a9e0794c6aa9516832c427'
            - 'SHA256=673bbc7fa4154f7d99af333014e888599c27ead02710f7bc7199184b30b38653'
            - 'SHA256=4b97d63ebdeda6941bb8cef5e94741c6cca75237ca830561f2262034805f0919'
            - 'SHA256=d50cb5f4b28c6c26f17b9d44211e515c3c0cc2c0c4bf24cd8f9ed073238053ad'
            - 'SHA256=62764ddc2dce74f2620cd2efd97a2950f50c8ac5a1f2c1af00dc5912d52f6920'
            - 'SHA256=6994b32e3f3357f4a1d0abe81e8b62dd54e36b17816f2f1a80018584200a1b77'
            - 'SHA256=751e9376cb7cb9de63e1808d43579d787d3f6d659173038fe44a2d7fdb4fd17e'
            - 'SHA256=87565ff08a93a8ff41ea932bf55dec8e0c7e79aba036507ea45df9d81cb36105'
            - 'SHA256=2da2b883e48e929f5365480d487590957d9e6582cc6da2c0b42699ba85e54fe2'
            - 'SHA256=627e13da6a45006fff4711b14754f9ccfac9a5854d275da798a22f3a68dd1eaa'
            - 'SHA256=94ba4bcbdb55d6faf9f33642d0072109510f5c57e8c963d1a3eb4f9111f30112'
            - 'SHA256=704c6ffe786bc83a73fbdcd2edd50f47c3b5053da7da6aa4c10324d389a31db4'
            - 'SHA256=d41e39215c2c1286e4cd3b1dc0948adefb161f22bc3a78756a027d41614ee4ff'
            - 'SHA256=0f7bfa10075bf5c193345866333d415509433dbfe5a7d45664b88d72216ff7c3'
            - 'SHA256=14b89298134696f2fd1b1df0961d36fa6354721ea92498a349dc421e79447925'
            - 'SHA256=3b2cd65a4fbdd784a6466e5196bc614c17d1dbaed3fd991d242e3be3e9249da6'
            - 'SHA256=2ce4f8089b02017cbe86a5f25d6bc69dd8b6f5060c918a64a4123a5f3be1e878'
            - 'SHA256=e99580e25f419b5ad90669e0c274cf63d30efa08065d064a863e655bdf77fb59'
            - 'SHA256=a74e8f94d2c140646a8bb12e3e322c49a97bd1b8a2e4327863d3623f43d65c66'
            - 'SHA256=47356707e610cfd0be97595fbe55246b96a69141e1da579e6f662ddda6dc5280'
            - 'SHA256=18c909a2b8c5e16821d6ef908f56881aa0ecceeaccb5fa1e54995935fcfd12f7'
            - 'SHA256=95e5b5500e63c31c6561161a82f7f9373f99b5b1f54b018c4866df4f2a879167'
            - 'SHA256=5c1585b1a1c956c7755429544f3596515dfdf928373620c51b0606a520c6245a'
            - 'SHA256=82b7fa34ad07dbf9afa63b2f6ed37973a1b4fe35dee90b3cf5c788c15c9f08f7'
            - 'SHA256=a85d3fd59bb492a290552e5124bfe3f9e26a3086d69d42ccc44737b5a66673ec'
            - 'SHA256=ea50f22daade04d3ca06dedb497b905215cba31aae7b4cab4b533fda0c5be620'
            - 'SHA256=d032001eab6cad4fbef19aab418650ded00152143bd14507e17d62748297c23f'
            - 'SHA256=4d42678df3917c37f44a1506307f1677b9a689efcf350b1acce7e6f64b514905'
            - 'SHA256=30061ef383e18e74bb067fbca69544f1a7544e8dc017d4e7633d8379aff4c3c3'
            - 'SHA256=7433f14b40c674c5e87b6210c330d5bcaf2f6f52d632ae29e9b7cf3ca405665b'
            - 'SHA256=818787057fc60ac8b957aa37d750aa4bace8e6a07d3d28b070022ee6dcd603ab'
            - 'SHA256=c4fb31e3f24e40742a1b9855a2d67048fe64b26d8d2dbcec77d2d5deeded2bcc'
            - 'SHA256=5295080de37d4838e15dec4e3682545033d479d3d9ac28d74747c086559fb968'
            - 'SHA256=7824931e55249a501074a258b4f65cd66157ee35672ba17d1c0209f5b0384a28'
            - 'SHA256=07759750fbb93c77b5c3957c642a9498fcff3946a5c69317db8d6be24098a4a0'
            - 'SHA256=51805bb537befaac8ce28f2221624cb4d9cefdc0260bc1afd5e0bc97bf1f9f93'
            - 'SHA256=e6f764c3b5580cd1675cbf184938ad5a201a8c096607857869bd7c3399df0d12'
            - 'SHA256=2faf95a3405578d0e613c8d88d534aa7233da0a6217ce8475890140ab8fb33c8'
            - 'SHA256=af4f42197f5ce2d11993434725c81ecb6f54025110dedf56be8ffc0e775d9895'
            - 'SHA256=baf7fbc4743a81eb5e4511023692b2dfdc32ba670ba3e4ed8c09db7a19bd82d3'
            - 'SHA256=a42f4ae69b8755a957256b57eb3d319678eab81705f0ffea0d649ace7321108f'
            - 'SHA256=4bca0a401b364a5cc1581a184116c5bafa224e13782df13272bc1b748173d1be'
            - 'SHA256=e4b2c0aa28aac5e197312a061b05363e2e0387338b28b23272b5b6659d29b1d8'
            - 'SHA256=69866557566c59772f203c11f5fba30271448e231b65806a66e48f41e3804d7f'
            - 'SHA256=93aa3066ae831cdf81505e1bc5035227dc0e8f06ebbbb777832a17920c6a02fe'
            - 'SHA256=bed4285d0f8d18f17ddaa53a98a475c87c04c4d167499e24c770da788e5d45f4'
            - 'SHA256=fa9abb3e7e06f857be191a1e049dd37642ec41fb2520c105df2227fcac3de5d5'
            - 'SHA256=07beac65e28ee124f1da354293a3d6ad7250ed1ce29b8342acfd22252548a5af'
            - 'SHA256=9a67626fb468d3f114c23ac73fd8057f43d06393d3eca04da1d6676f89da2d40'
            - 'SHA256=7f4555a940ce1156c9bcea9a2a0b801f9a5e44ec9400b61b14a7b1a6404ffdf6'
            - 'SHA256=7a84703552ae032a0d1699a081e422ed6c958bbe56d5b41839c8bfa6395bee1d'
            - 'SHA256=ddf427ce55b36db522f638ba38e34cd7b96a04cb3c47849b91e7554bfd09a69a'
            - 'SHA256=64d4370843a07e25d4ceb68816015efcaeca9429bb5bb692a88e615b48c7da96'
            - 'SHA256=c8f9e1ad7b8cce62fba349a00bc168c849d42cfb2ca5b2c6cc4b51d054e0c497'
            - 'SHA256=fefc070a5f6a9c0415e1c6f44512a33e8d163024174b30a61423d00d1e8f9bf2'
            - 'SHA256=8d9a2363b757d3f127b9c6ed8f7b8b018e652369bc070aa3500b3a978feaa6ce'
            - 'SHA256=d43520128871c83b904f3136542ea46644ac81a62d51ae9d3c3a3f32405aad96'
            - 'SHA256=efa56907b9d0ec4430a5d581f490b6b9052b1e979da4dab6a110ab92e17d4576'
            - 'SHA256=1d23ab46ad547e7eef409b40756aae9246fbdf545d13946f770643f19c715e80'
            - 'SHA256=62036cdf3663097534adf3252b921eed06b73c2562655eae36b126c7d3d83266'
            - 'SHA256=6661320f779337b95bbbe1943ee64afb2101c92f92f3d1571c1bf4201c38c724'
            - 'SHA256=3033ff03e6f523726638b43d954bc666cdd26483fa5abcf98307952ff88f80ee'
            - 'SHA256=6964a5d85639baee288555797992861232e75817f93028b50b8c6d34aa38b05b'
            - 'SHA256=06c5ebd0371342d18bc81a96f5e5ce28de64101e3c2fd0161d0b54d8368d2f1f'
            - 'SHA256=1485c0ed3e875cbdfc6786a5bd26d18ea9d31727deb8df290a1c00c780419a4e'
            - 'SHA256=6839fcae985774427c65fe38e773aa96ec451a412caa5354ad9e2b9b54ffe6c1'
            - 'SHA256=deade507504d385d8cae11365a2ac9b5e2773ff9b61624d75ffa882d6bb28952'
            - 'SHA256=c42c1e5c3c04163bf61c3b86b04a5ec7d302af7e254990cef359ac80474299da'
            - 'SHA256=8dafe5f3d0527b66f6857559e3c81872699003e0f2ffda9202a1b5e29db2002e'
            - 'SHA256=88076e98d45ed3adf0c5355411fe8ca793eb7cec1a1c61f5e1ec337eae267463'
            - 'SHA256=b0f1fbadc1d7a77557d3d836f7698bd986a3ec9fc5d534ad3403970f071176f7'
            - 'SHA256=bcb774b6f6ff504d2db58096601bc5cb419c169bfbeaa3af852417e87d9b2aa0'
            - 'SHA256=4dc24fd07f8fb854e685bc540359c59f177de5b91231cc44d6231e33c9e932b1'
            - 'SHA256=82b0e1d7a27b67f0e6dc39dc41e880bdaef5d1f69fcec38e08da2ed78e805ef9'
            - 'SHA256=ad938d15ecfd70083c474e1642a88b078c3cea02cdbddf66d4fb1c01b9b29d9a'
            - 'SHA256=443c0ba980d4db9213b654a45248fd855855c1cc81d18812cae9d16729ff9a85'
            - 'SHA256=f3ec3f22639d45b3c865bb1ed7622db32e04e1dbc456298be02bf1f3875c3aac'
            - 'SHA256=0181d60506b1f3609217487c2c737621d637e1232f243f68c662d045f44d4873'
            - 'SHA256=c13f5bc4edfbe8f1884320c5d76ca129d00de41a1e61d45195738f125dfe60a7'
            - 'SHA256=8684aec77b4c3cafc1a6594de7e95695fa698625d4206a6c4b201875f76a5b38'
            - 'SHA256=c4c9c84b211899ceb0d18a839afa497537a7c7c01ab481965a09788a9e16590c'
            - 'SHA256=d37996abc8efb29f1ccbb4335ce9ba9158bec86cc4775f0177112e87e4e3be5c'
            - 'SHA256=1a5c08d40a5e73b9fe63ea5761eaec8f41d916ca3da2acbc4e6e799b06af5524'
            - 'SHA256=9c2f3e9811f7d0c7463eaa1ee6f39c23f902f3797b80891590b43bbe0fdf0e51'
            - 'SHA256=bb2422e96ea993007f25c71d55b2eddfa1e940c89e895abb50dd07d7c17ca1df'
            - 'SHA256=94c71954ac0b1fd9fa2bd5c506a16302100ba75d9f84f39ee9b333546c714601'
            - 'SHA256=6d68d8a71a11458ddf0cbb73c0f145bee46ef29ce03ad7ece6bd6aa9d31db9b7'
            - 'SHA256=80e4c83cfa9d675a6746ab846fa5da76d79e87a9297e94e595a2d781e02673b3'
            - 'SHA256=e858de280bd72d7538386a73e579580a6d5edba87b66b3671dc180229368be19'
            - 'SHA256=ee7b8eb150df2788bb9d5fe468327899d9f60d6731c379fd75143730a83b1c55'
            - 'SHA256=8206ce9c42582ac980ff5d64f8e3e310bc2baa42d1a206dd831c6ab397fbd8fe'
            - 'SHA256=4f02aed3750bc6a924c75e774404f259f721d8f4081ed68aa01cf73ca5430f85'
            - 'SHA256=81c7bb39100d358f8286da5e9aa838606c98dfcc263e9a82ed91cd438cb130d1'
            - 'SHA256=0f98492c92e35042b09032e3d9aedc357e4df94fc840217fa1091046f9248a06'
            - 'SHA256=9b1b15a3aacb0e786a608726c3abfc94968915cedcbd239ddf903c4a54bfcf0c'
            - 'SHA256=b9dad0131c51e2645e761b74a71ebad2bf175645fa9f42a4ab0e6921b83306e3'
            - 'SHA256=26ef7b27d1afb685e0c136205a92d29b1091e3dcf6b7b39a4ec03fbbdb57cb55'
            - 'SHA256=a1e6b431534258954db07039117b3159e889c6b9e757329bbd4126383c60c778'
            - 'SHA256=d25b5e4d07f594c640dcd93cfc8ab3f0a38348150bd0bfae89f404fbb0d811c6'
            - 'SHA256=1ef7afea0cf2ef246ade6606ef8b7195de9cd7a3cd7570bff90ba1e2422276f6'
            - 'SHA256=083a311875173f8c4653e9bbbabb689d14aa86b852e7fa9f5512fc60e0fd2c43'
            - 'SHA256=89698cad598a56f9e45efffd15d1841e494a2409cc12279150a03842cd6bb7f3'
            - 'SHA256=a7a665a695ec3c0f862a0d762ad55aff6ce6014359647e7c7f7e3c4dc3be81b7'
            - 'SHA256=02ebf848fa618eba27065db366b15ee6629d98f551d20612ac38b9f655f37715'
            - 'SHA256=8b32fc8b15363915605c127ccbf5cbe71778f8dfbf821a25455496e969a01434'
            - 'SHA256=ee525b90053bb30908b5d7bf4c5e9b8b9d6b7b5c9091a26fa25d30d3ad8ef5d0'
            - 'SHA256=41ad660820c41fc8b1860b13dc1fea8bc8cb2faceb36ed3e29d40d28079d2b1f'
            - 'SHA256=42ff11ddb46dfe5fa895e7babf88ee27790cde53a9139fc384346a89e802a327'
            - 'SHA256=36f45a42ebf2de6962db92aaf8845d7f9fd6895bedc31422adcf31c59a79602d'
            - 'SHA256=4bd4715d2a7af627da11513e32fab925c872babebdb7ff5675a75815fbf95021'
            - 'SHA256=4734a0a5d88f44a4939b8d812364cab6ca5f611b9b8ceebe27df6c1ed3a6d8a4'
            - 'SHA256=e8743094f002239a8a9d6d7852c7852e0bb63cd411b007bd8c194bcba159ef15'
            - 'SHA256=f0474e76cfd36e37e32cfe5c0a9e05ddee17dd5014d7aa8817ea3634a3540a3f'
            - 'SHA256=a0931e16cf7b18d15579e36e0a69edad1717b07527b5407f2c105a2f554224b2'
            - 'SHA256=52d5c35325ce701516f8b04380c9fbdb78ec6bcc13b444f758fdb03d545b0677'
            - 'SHA256=e1cb86386757b947b39086cc8639da988f6e8018ca9995dd669bdc03c8d39d7d'
            - 'SHA256=7662187c236003308a7951c2f49c0768636c492f8935292d02f69e59b01d236d'
            - 'SHA256=24c900024d213549502301c366d18c318887630f04c96bf0a3d6ba74e0df164f'
            - 'SHA256=b7956e31c2fcc0a84bcedf30e5f8115f4e74eed58916253a0c05c8be47283c57'
            - 'SHA256=96bf3ee7c6673b69c6aa173bb44e21fa636b1c2c73f4356a7599c121284a51cc'
            - 'SHA256=d7c81b0f3c14844f6424e8bdd31a128e773cb96cccef6d05cbff473f0ccb9f9c'
            - 'SHA256=0d676baac43d9e2d05b577d5e0c516fba250391ab0cb11232a4b17fd97a51e35'
            - 'SHA256=888491196bd8ff528b773a3e453eae49063ad31fb4ca0f9f2e433f8d35445440'
            - 'IMPHASH=8d070a93a45ed8ba6dba6bfbe0d084e7'
            - 'IMPHASH=7641a0c227f0a3a45b80bb8af43cd152'
            - 'IMPHASH=7df0d3ee663fc0e7c72a95e44ba4c82c'
            - 'IMPHASH=70e1caa5a322b56fd7951f1b2caacb0d'
            - 'IMPHASH=beceab354c66949088c9e5ed1f1ff2a4'
            - 'IMPHASH=caa08a0ba5f679b1e5bbae747cb9d626'
            - 'IMPHASH=420625b024fba72a24025defdf95b303'
            - 'IMPHASH=65ccc2c578a984c31880b6c5e65257d3'
            - 'IMPHASH=e717abe060bc5c34925fe3120ac22f45'
            - 'IMPHASH=41113a3a832353963112b94f4635a383'
            - 'IMPHASH=3866dd9fe63de457bdbf893bf7050ddf'
            - 'IMPHASH=3fd33d5b3b52e2db91983ac4b1d7a3c4'
            - 'IMPHASH=a998fe47a44bfbf2399968e21cfdf7ca'
            - 'IMPHASH=c9a6e83d931286d1604d1add8403e1e5'
            - 'IMPHASH=cf0eb2dce2ba2c9ff5dd0da794b8b372'
            - 'IMPHASH=ea37e43ffc7cfcba181c5cff37a9be1f'
            - 'IMPHASH=8e35c9460537092672b3c7c14bccc7e0'
            - 'IMPHASH=7bf14377888c429897eb10a85f70266c'
            - 'IMPHASH=b351627263648b1d220bb488e7ec7202'
            - 'IMPHASH=ce10082e1aa4c1c2bd953b4a7208e56a'
            - 'IMPHASH=a7bd820fa5b895fab06f20739c9f24b8'
            - 'IMPHASH=be0dd8b8e045356d600ee55a64d9d197'
            - 'IMPHASH=63fd1582ac2edee50f7ec7eedde38ee8'
            - 'IMPHASH=6c8d5c79a850eecc2fb0291cebda618d'
            - 'IMPHASH=c32d9a9af7f702814e1368c689877f3a'
            - 'IMPHASH=6b387c029257f024a43a73f38afb2629'
            - 'IMPHASH=df43355c636583e56e92142dcc69cc58'
            - 'IMPHASH=e3ee9131742bf9c9d43cb9a425e497dd'
            - 'IMPHASH=c214aac08575c139e48d04f5aee21585'
            - 'IMPHASH=3c5d2ffd06074f1b09c89465cc8bfbf7'
            - 'IMPHASH=059c6bd84285f4960e767f032b33f19b'
            - 'IMPHASH=a09170ef09c55cdca9472c02cb1f2647'
            - 'IMPHASH=fca0f3c7b6d79f494034b9d2a1f5921a'
            - 'IMPHASH=0262d4147f21d681f8519ab2af79283f'
            - 'IMPHASH=832219eb71b8bdb771f1d29d27b0acf4'
            - 'IMPHASH=514298d18002920ee5a917fc34426417'
            - 'IMPHASH=26ceec6572c630bdad60c984e51b7da4'
            - 'IMPHASH=dbf09dd3e675f15c7cc9b4d2b8e6cd90'
            - 'IMPHASH=4b47f6031c558106eee17655f8f8a32f'
            - 'IMPHASH=a6c4a7369500900fc172f9557cff22cf'
            - 'IMPHASH=3b49942ec6cef1898e97f741b2b5df8a'
            - 'IMPHASH=28dc68bb6d6bf4f6b2db8dd7588b2511'
            - 'IMPHASH=27f6dc8a247a22308dd1beba5086b302'
            - 'IMPHASH=7d017945bf90936a6c40f73f91ed02c2'
            - 'IMPHASH=d51f0f6034eb5e45f0ed4e9b7bbc9c97'
            - 'IMPHASH=0ad7da35304c75ccf859bc29fe9ed09e'
            - 'IMPHASH=bf9d32a6ab9effcd2fd6a734e5be98f9'
            - 'IMPHASH=87fd2b54ed568e2294300e164b8c46f7'
            - 'IMPHASH=2de3451f3e7b02970582bb8f9fd8c73a'
            - 'IMPHASH=e97dc162f416bf06745bf9ffdf78a0ff'
            - 'IMPHASH=2a008187d4a73284ddcc43f1b727b513'
            - 'IMPHASH=f8e4844312e81dbdb4e8e95e2ad2c127'
            - 'IMPHASH=4c7cc13a110ccdbb932bb9d7d42efdf4'
            - 'IMPHASH=45bfe170e0cd654bc1e2ae3fca3ac3f4'
            - 'IMPHASH=3db9de43d5d530c10d0cd2d43c7a0771'
    condition: selection
falsepositives:
    - Unknown
level: high`

#### Sigma Location(s)

uri://aso/rules/Malicious_Driver_Load.yml

#### Sigma Confidence Level

experimental

#### Sigma Assurance Level

high

#### Sigma Query

{
  "selection": {
    "Hashes|contains": [
      "MD5=5be61a24f50eb4c94d98b8a82ef58dcf",
      "MD5=d70a80fc73dd43469934a7b1cc623c76",
      "MD5=3b71eab204a5f7ed77811e41fed73105",
      "MD5=528ce5ce19eb34f401ef024de7ddf222",
      "MD5=ae548418b491cd3f31618eb9e5730973",
      "MD5=72f53f55898548767e0276c472be41e8",
      "MD5=508faa4647f305a97ed7167abc4d1330",
      "MD5=ed2b653d55c03f0bffa250372d682b75",
      "MD5=0d2ba47286f1c68e87622b3a16bf9d92",
      "MD5=3164bd6c12dd0fe1bdf3b833d56323b9",
      "MD5=70fd7209ce5c013a1f9e699b5cc86cdc",
      "MD5=c71be7b112059d2dc84c0f952e04e6cc",
      "MD5=acac842a46f3501fe407b1db1b247a0b",
      "MD5=01c2e4d8234258451083d6ce4e8910b7",
      "MD5=c8541a9cef64589593e999968a0385b9",
      "MD5=e172a38ade3aa0a2bc1bf9604a54a3b5",
      "MD5=6fcf56f6ca3210ec397e55f727353c4a",
      "MD5=2b80be31fbb11d4c1ef6d6a80b2e0c16",
      "MD5=07056573d464b0f5284f7e3acedd4a3f",
      "MD5=c7b7f1edb9bbef174e6506885561d85d",
      "MD5=d5918d735a23f746f0e83f724c4f26e5",
      "MD5=84763d8ca9fe5c3bff9667b2adf667de",
      "MD5=fb593b1f1f80d20fc7f4b818065c64b6",
      "MD5=909f3fc221acbe999483c87d9ead024a",
      "MD5=e29f6311ae87542b3d693c1f38e4e3ad",
      "MD5=aeb0801f22d71c7494e884d914446751",
      "MD5=3f11a94f1ac5efdd19767c6976da9ba4",
      "MD5=be6318413160e589080df02bb3ca6e6a",
      "MD5=0b311af53d2f4f77d30f1aed709db257",
      "MD5=d075d56dfce6b9b13484152b1ef40f93",
      "MD5=27384ec4c634701012a2962c30badad2",
      "MD5=5eb2c576597dd21a6b44557c237cf896",
      "MD5=f56db4eba3829c0918413b5c0b42f00f",
      "MD5=e27b2486aa5c256b662812b465b6036c",
      "MD5=db86dfd7aefbb5be6728a63461b0f5f3",
      "MD5=04a88f5974caa621cee18f34300fc08a",
      "MD5=5129d8fd53d6a4aba81657ab2aa5d243",
      "MD5=cd2c641788d5d125c316ed739c69bb59",
      "MD5=7073cd0085fcba1cd7d3568f9e6d652c",
      "MD5=24f0f2b4b3cdae11de1b81c537df41c7",
      "MD5=88bea56ae9257b40063785cf47546024",
      "MD5=63060b756377fce2ce4ab9d079ca732f",
      "MD5=50b39072d0ee9af5ef4824eca34be6e3",
      "MD5=57c18a8f5d1ba6d015e4d5bc698e3624",
      "MD5=7d26985a5048bad57d9c223362f3d55c",
      "MD5=ba54a0dbe2685e66e21d41b4529b3528",
      "MD5=4ad8fd9e83d7200bd7f8d0d4a9abfb11",
      "MD5=b52f51bbe6b49d0b475d943c29c4d4cb",
      "MD5=a837302307dace2a00d07202b661bce2",
      "MD5=78a122d926ccc371d60c861600c310f3",
      "MD5=bdb305aa0806f8b38b7ce43c927fe919",
      "MD5=27053e964667318e1b370150cbca9138",
      "MD5=6a4fbcfb44717eae2145c761c1c99b6a",
      "MD5=d13c1b76b4a1ca3ff5ab63678b51df6d",
      "MD5=6a066d2be83cf83f343d0550b0b8f206",
      "MD5=7108b0d4021af4c41de2c223319cd4c1",
      "MD5=1cd158a64f3d886357535382a6fdad75",
      "MD5=e939448b28a4edc81f1f974cebf6e7d2",
      "MD5=4198d3db44d7c4b3ba9072d258a4fc2d",
      "MD5=4a27a2bdc6fbe39eeec6455fb1e0ef20",
      "MD5=30ca3cc19f001a8f12c619daa8c6b6e3",
      "MD5=fe9004353b25640f6a879e57f07122d7",
      "MD5=06c7fcf3523235cf52b3eee083ec07b2",
      "MD5=364605ad21b9275681cffef607fac273",
      "MD5=968ddb06af90ef83c5f20fbdd4eee62e",
      "MD5=ba50bd645d7c81416bb26a9d39998296",
      "MD5=29e03f4811b64969e48a99300978f58c",
      "MD5=b0770094c3c64250167b55e4db850c04",
      "MD5=40b968ecdbe9e967d92c5da51c390eee",
      "MD5=b6b530dd25c5eb66499968ec82e8791e",
      "MD5=f209cb0e468ca0b76d879859d5c8c54e",
      "MD5=76f8607fc4fb9e828d613a7214436b66",
      "MD5=4b058945c9f2b8d8ebc485add1101ba5",
      "MD5=faae7f5f69fde12303dd1c0c816b72b7",
      "MD5=89d294ef7fefcdf1a6ca0ab96a856f57",
      "MD5=ef0e1725aaf0c6c972593f860531a2ea",
      "MD5=bbdbffebfc753b11897de2da7c9912a5",
      "MD5=5ebfc0af031130ba9de1d5d3275734b3",
      "MD5=22949977ce5cd96ba674b403a9c81285",
      "MD5=77cfd3943cc34d9f5279c330cd8940bc",
      "MD5=311de109df18e485d4a626b5dbe19bc6",
      "MD5=2730cc25ad385acc7213a1261b21c12d",
      "MD5=87dc81ebe85f20c1a7970e495a778e60",
      "MD5=154b45f072fe844676e6970612fd39c7",
      "MD5=5a4fe297c7d42539303137b6d75b150d",
      "MD5=d6a1dd7b2c06f058b408b3613c13d413",
      "MD5=a6e9d6505f6d2326a8a9214667c61c67",
      "MD5=7fad9f2ef803496f482ce4728578a57a",
      "MD5=5076fba3d90e346fd17f78db0a4aa12c",
      "MD5=79df0eabbf2895e4e2dae15a4772868c",
      "MD5=14580bd59c55185115fd3abe73b016a2",
      "MD5=1f2888e57fdd6aee466962c25ba7d62d",
      "MD5=5e9231e85cecfc6141e3644fda12a734",
      "MD5=dc564bac7258e16627b9de0ce39fae25",
      "MD5=4e4c068c06331130334f23957fca9e3c",
      "MD5=1ee9f6326649cd23381eb9d7dfdeddf7",
      "MD5=4e1f656001af3677856f664e96282a6f",
      "MD5=36f44643178c505ea0384e0fb241e904",
      "MD5=6b480fac7caca2f85be9a0cfe79aedfc",
      "MD5=c1ab425977d467b64f437a6c5ad82b44",
      "MD5=fe508caa54ffeb2285d9f00df547fe4a",
      "MD5=d3af70287de8757cebc6f8d45bb21a20",
      "MD5=990b949894b7dc82a8cf1131b063cb1a",
      "MD5=c62209b8a5daf3f32ad876ad6cefda1b",
      "MD5=c159fb0f345a8771e56aab8e16927361",
      "MD5=19b15eeccab0752c6793f782ca665a45",
      "MD5=1d51029dfbd616bf121b40a0d1efeb10",
      "MD5=157a22689629ec876337f5f9409918d5",
      "MD5=3dd829fb27353622eff34be1eabb8f18",
      "MD5=8636fe3724f2bcba9399daffd6ef3c7e",
      "MD5=3d0b3e19262099ade884b75ba86ca7e8",
      "MD5=97539c78d6e2b5356ce79e40bcd4d570",
      "MD5=0308b6888e0f197db6704ca20203eee4",
      "MD5=091a6bd4880048514c5dd3bede15eba5",
      "MD5=7e92f98b809430622b04e88441b2eb04",
      "MD5=bb5bda8889d8d27ef984dbd6ad82c946",
      "MD5=b76aee508f68b5b6dccd6e1f66f4cf8b",
      "MD5=a822b9e6eedf69211013e192967bf523",
      "MD5=df52f8a85eb64bc69039243d9680d8e4",
      "MD5=bfbdea0589fb77c7a7095cf5cd6e8b7a",
      "MD5=44857ca402a15ab51dc5afe47abdfa44",
      "MD5=f9844524fb0009e5b784c21c7bad4220",
      "MD5=d34b218c386bfe8b1f9c941e374418d7",
      "MD5=0ca010a32a9b0aeae1e46d666b83b659",
      "MD5=93496a436c5546156a69deb255a9fed0",
      "MD5=1cd5e231064e03c596e819b6ff48daf9",
      "MD5=70a71fe86df717ac59dbf856d7ac5789",
      "MD5=a33089d4e50f7d2ea8b52ca95d26ebf3",
      "MD5=e0cc9b415d884f85c45be145872892b8",
      "MD5=a42249a046182aaaf3a7a7db98bfa69d",
      "MD5=c5ae6ca044bd03c3506c132b033be1dc",
      "MD5=7ebe606acd81abf1f8cb0767c974164b",
      "MD5=b5dcc869a91efcc6e8ea0c3c07605d63",
      "MD5=62c18d61ed324088f963510bae43b831",
      "MD5=093a2a635c3a27aac50efd6463f4efa1",
      "MD5=28102acca39ad0199f262ba9958be3f4",
      "MD5=650ef9dd70cb192027e536754d6e0f63",
      "MD5=32eb3d2bf2c5b3da2d2a1f20fffbac44",
      "MD5=6771b13a53b9c7449d4891e427735ea2",
      "MD5=072ba2309b825ce1dba37d8d924ea8ed",
      "MD5=2d37d2fb9b9f8ac52bc02cba4487e3cb",
      "MD5=1325ec39e98225e487b40043faee8052",
      "MD5=4484f4007de2c3ee4581a2cff77ca3b4",
      "MD5=a236e7d654cd932b7d11cb604629a2d0",
      "MD5=17509f0a98dc5c5d52c3f9ac1428a21b",
      "MD5=840a5edf2534dd23a082cf7b28cbfc4d",
      "MD5=77a7ed4798d02ef6636cd0fd07fc382a",
      "MD5=a9df5964635ef8bd567ae487c3d214c4",
      "MD5=8b75047199825c8e62fdcc1c915db8bd",
      "MD5=d416494232c4197cb36a914df2e17677",
      "MD5=4cf14a96485a1270fed97bb8000e4f86",
      "MD5=35e512f9bedc89dca5ce81f35820714c",
      "MD5=40f35792e7565aa047796758a3ce1b77",
      "MD5=f7f31bccc9b7b2964ac85106831022b1",
      "MD5=26aedc10d4215ba997495d3a68355f4a",
      "MD5=10f3679384a03cb487bda9621ceb5f90",
      "MD5=80219fb6b5954c33e16bac5ecdac651b",
      "MD5=cee36b5c6362993fa921435979bfbe4a",
      "MD5=e37a08f516b8a7ca64163f5d9e68fe5a",
      "MD5=49518f7375a5f995ebe9423d8f19cfe4",
      "MD5=920df6e42cf91bbe19707f5a86e3c5c5",
      "MD5=2ec877e425bd7eddb663627216e3491e",
      "MD5=550b7991d93534bc510bc4f237155a7a",
      "MD5=98d53f6b3bec0a3417a04fbb9e17fa06",
      "MD5=13a57a4ef721440c7c9208b51f7c05de",
      "MD5=c5fc3605194e033bdf3781ff2adaeb61",
      "MD5=6e625ec04c20a9dbd48c7060efbf5e92",
      "MD5=0b9b78d1281c7d4ab50497cf6ea7452a",
      "MD5=4e906fcb13e2793c98f47291fd69391b",
      "MD5=2bb353891d65c9e267eb98a3a2b694c3",
      "MD5=7d86cdda7f49f91fdb69901a002b34e7",
      "MD5=f69b06ca7c34d16f26ea1c6861edf62a",
      "MD5=ee6b1a79cb6641aa44c762ee90786fe0",
      "MD5=1fc7aeeff3ab19004d2e53eae8160ab1",
      "MD5=24d3ea54f25e32832ac20335a1ce1062",
      "MD5=c94f405c5929cfcccc8ad00b42c95083",
      "MD5=b164daf106566f444dfb280d743bc2f7",
      "MD5=93130909e562925597110a617f05e2a9",
      "MD5=f589d4bf547c140b6ec8a511ea47c658",
      "MD5=bf445ac375977ecf551bc2a912c58e8a",
      "MD5=629ee55e4b5a225d048fbcd5f0a1d18b",
      "MD5=0023ca0ca16a62d93ef51f3df98b2f94",
      "MD5=a3d69c7e24300389b56782aa63b0e357",
      "MD5=cbd8d370462503508e44dba023bdf9bc",
      "MD5=67daa04716803a15fc11c9e353d77c2f",
      "MD5=c9d4214c850e0cedf033dc8f0cd3aace",
      "MD5=bd5b0514f3b40f139d8079138d01b5f6",
      "MD5=19bdd9b799e3c2c54c0d7fff68b31c20",
      "MD5=f242cffd9926c0ccf94af3bf16b6e527",
      "MD5=5aeab9427d85951def146b4c0a44fc63",
      "MD5=40170485cca576adb5266cf5b0d3b0bd",
      "MD5=c277c4386a78fae1b7e17eaecf4f472b",
      "MD5=58c37866cbc3d1338e4fc58ada924ffe",
      "MD5=0f16a43f7989034641fd2de3eb268bf1",
      "MD5=0ae30291c6cbfa7be39320badd6e8de0",
      "MD5=05dd59bd4f175304480affd8f1305c37",
      "MD5=f838f4eb36f1e7036238776c7a70f0b0",
      "MD5=85093bb9f027027c2c61aee50796de30",
      "MD5=ae338d91d1b05a72559b7f6ed717362d",
      "MD5=bd91787b5dcb2189b856804e85dfa1d9",
      "MD5=6b3c1511e12f4d27a4ea3b18020d7b84",
      "MD5=97264fd62d4907bdac917917a07b3b7a",
      "MD5=6ececf26ff8b03ed7ffbddadec9a9dab",
      "MD5=47e6ac52431ca47da17248d80bf71389",
      "MD5=eb57f03b7603f0b235af62e8cd5be8c2",
      "MD5=e1a9aa4c14669b1fb1f67a7266f87e82",
      "MD5=29047f0b7790e524b09a06852d31a117",
      "MD5=4dd6250eb2d368f500949952eb013964",
      "MD5=fb7c61ef427f9b2fdff3574ee6b1819b",
      "MD5=844af8c877f5da723c1b82cf6e213fc1",
      "MD5=e39152eadd76751b1d7485231b280948",
      "MD5=ac6e29f535b2c42999c50d2fc32f2c9c",
      "MD5=2406ea37152d2154be3fef6d69ada2c6",
      "MD5=0ea8389589c603a8b05146bd06020597",
      "MD5=754e21482baf18b8b0ed0f4be462ba03",
      "MD5=c4a517a02ba9f6eac5cf06e3629cc076",
      "MD5=32282e07db321e8d7849f2287bb6a14f",
      "MD5=32b67a6cd6dd998b9f563ed13d54a8bc",
      "MD5=3359e1d4244a7d724949c63e89689ef8",
      "MD5=5917e415a5bf30b3fcbcbcb8a4f20ee0",
      "MD5=0bdd51cc33e88b5265dfb7d88c5dc8d6",
      "MD5=a90236e4962620949b720f647a91f101",
      "MD5=ccde8c94439f9fc9c42761e4b9a23d97",
      "MD5=68caf620ef8deaf06819cf8c80d3367b",
      "MD5=5fec28e8f4f76e5ede24beb32a32b9d7",
      "MD5=e8eac6642b882a6196555539149c73f2",
      "MD5=aa98b95f5cbae8260122de06a215ee10",
      "MD5=a5bcaa2fc87b42e2e5d62a2e5dfcbc80",
      "MD5=abc168fdca7169bf9dc40cec9761018d",
      "MD5=7f9309f5e4defec132b622fadbcad511",
      "MD5=4748696211bd56c2d93c21cab91e82a5",
      "MD5=48394dce30bb8da5ae089cb8f41b86dc",
      "MD5=65f800e1112864bf41eb815649f428d5",
      "MD5=bd25be845c151370ff177509d95d5add",
      "MD5=a37ed7663073319d02f2513575a22995",
      "MD5=2c39f6172fbc967844cac12d7ab2fa55",
      "MD5=491aec2249ad8e2020f9f9b559ab68a8",
      "MD5=1e0eb80347e723fa31fce2abb0301d44",
      "MD5=a26363e7b02b13f2b8d697abb90cd5c3",
      "MD5=4118b86e490aed091b1a219dba45f332",
      "MD5=6d131a7462e568213b44ef69156f10a5",
      "MD5=10c2ea775c9e76e7774ab89e38f38287",
      "SHA1=994e3f5dd082f5d82f9cc84108a60d359910ba79",
      "SHA1=4f7989ad92b8c47c004d3731b7602ce0934d7a23",
      "SHA1=f2fe02e28cf418d935ec63168caf4dff6a9fbdfe",
      "SHA1=af42afda54d150810a60baa7987f9f09d49d1317",
      "SHA1=09375f13521fc0cacf2cf0a28b2a9248f71498d7",
      "SHA1=c75e8fceed74a4024d38ca7002d42e1ecf982462",
      "SHA1=03e82eae4d8b155e22ffdafe7ba0c4ab74e8c1a7",
      "SHA1=e730eb971ecb493b69de2308b6412836303f733a",
      "SHA1=6a95860594cd8b7e3636bafa8f812e05359a64ca",
      "SHA1=5fef884a901e81ac173d63ade3f5c51694decf74",
      "SHA1=a8ddb7565b61bc021cd2543a137e00627f999dcc",
      "SHA1=6451522b1fb428e549976d0742df5034f8124b17",
      "SHA1=8ad0919629731b9a8062f7d3d4a727b28f22e81a",
      "SHA1=cc65bf60600b64feece5575f21ab89e03a728332",
      "SHA1=bbc8bd714c917bb1033f37e4808b4b002cd04166",
      "SHA1=4f2d9a70ea24121ae01df8a76ffba1f9cc0fde4a",
      "SHA1=f6a18fc9c4abe4a82c1ab28abc0a7259df8de7a3",
      "SHA1=c42178977bd7bbefe084da0129ed808cb7266204",
      "SHA1=766949d4599fbf8f45e888c9d6fedf21e04fb333",
      "SHA1=b7ff8536553cb236ea2607941e634b23aadb59ee",
      "SHA1=76789196eebfd4203f477a5a6c75eefc12d9a837",
      "SHA1=e5566684a9e0c1afadae80c3a8be6636f6cad7cf",
      "SHA1=7638c048af5beae44352764390deea597cc3e7b1",
      "SHA1=6a6fe0d69e0ea34d695c3b525e6db639f9ad6ac5",
      "SHA1=08dd35dde6187af579a1210e00eadbcea29e66d2",
      "SHA1=9ee31f1f25f675a12b7bad386244a9fbfa786a87",
      "SHA1=3ef30c95e40a854cc4ded94fc503d0c3dc3e620e",
      "SHA1=a804ebec7e341b4d98d9e94f6e4860a55ea1638d",
      "SHA1=505546d82aab56889a923004654b9afdec54efe6",
      "SHA1=0fe2d22bd2e6b7874f4f2b6279e2ca05edd1222a",
      "SHA1=8aa0e832e5ca2eb79dafabadbe9948a191008383",
      "SHA1=844d7bcd1a928d340255ff42971cca6244a459bf",
      "SHA1=9e2ebc489c50b6bbae3b08473e007baa65ff208f",
      "SHA1=7e836dadc2e149a0b758c7e22c989cbfcce18684",
      "SHA1=2480549ec8564cd37519a419ab2380cf3e8bab9e",
      "SHA1=8b9dd4c001f17e7835fdaf0d87a2f3e026557e84",
      "SHA1=d3f6c3ea2ef7124403c0fb6e7e3a0558729b5285",
      "SHA1=40df7a55c200371853cc3fd3cc03b5ac932f5cd6",
      "SHA1=607387cc90b93d58d6c9a432340261fde846b1d9",
      "SHA1=2779c54ccd1c008cd80e88c2b454d76f4fa18c07",
      "SHA1=46c9a474a1a62c25a05bc7661b75a80b471616e6",
      "SHA1=a2fe7de67b3f7d4b1def88ce4ba080f473c0fbc6",
      "SHA1=b8b123a413b7bccfa8433deba4f88669c969b543",
      "SHA1=bf2f8ada4e80aed4710993cedf4c5d32c95cd509",
      "SHA1=e3a1e7ce9e9452966885371e4c7fb48a2efdef22",
      "SHA1=c7f0423ac5569f13d2b195e02741ad7eed839c6d",
      "SHA1=a111dc6ae5575977feba71ee69b790e056846a02",
      "SHA1=ac4ace1c21c5cb72c6edf6f2f0cc3513d7c942c3",
      "SHA1=d4304bc75c2cb9917bb10a1dc630b75af194f7b2",
      "SHA1=0de86ec7d7f16a3680df89256548301eed970393",
      "SHA1=b2fb5036b29b12bcec04c3152b65b67ca14d61f2",
      "SHA1=0883a9c54e8442a551994989db6fc694f1086d41",
      "SHA1=01cf1fe3937fb6585ffb468b116a3af8ddf9ef16",
      "SHA1=98c4406fede34c3704afd8cf536ec20d93df9a10",
      "SHA1=1048f641adf3988d882a159bf1332eeb6d6a7f09",
      "SHA1=867652e062eb6bd1b9fc29e74dea3edd611ef40c",
      "SHA1=78fd06c82d3ba765c38bad8f48d1821a06280e39",
      "SHA1=6debce728bcff73d9d1d334df0c6b1c3735e295c",
      "SHA1=fdbcebb6cafda927d384d7be2e8063a4377d884f",
      "SHA1=994dc79255aeb662a672a1814280de73d405617a",
      "SHA1=6abc7979ba044f31884517827afb7b4bdaa0dcc1",
      "SHA1=1768f9c780fe7cf66928cfceaef8ed7d985e18f5",
      "SHA1=5fa527e679d25a15ecc913ce6a8d0218e2ff174b",
      "SHA1=f11188c540eada726766e0b0b2f9dd3ae2679c61",
      "SHA1=8416ee8fd88c3d069fbba90e959507c69a0ee3e9",
      "SHA1=ab4399647ebd16c02728c702534a30eb0b7ccbe7",
      "SHA1=98588b1d1b63747fa6ee406983bf50ad48a2208b",
      "SHA1=86e6669dbbce8228e94b2a9f86efdf528f0714fd",
      "SHA1=c9e9198d52d94771cb14711a5f6aaf8d82b602a2",
      "SHA1=17fa047c1f979b180644906fe9265f21af5b0509",
      "SHA1=1b526cbcba09b8d663e82004cf24ef44343030d3",
      "SHA1=4e0f5576804dab14abb29a29edb9616a1dbe280a",
      "SHA1=eb76de59ebc5b2258cff0567577ff8c9d0042048",
      "SHA1=d4f5323da704ff2f25d6b97f38763c147f2a0e6f",
      "SHA1=6802e2d2d4e6ee38aa513dafd6840e864310513b",
      "SHA1=ac18c7847c32957abe8155bcbe71c1f35753b527",
      "SHA1=beed6fb6a96996e9b016fa7f2cf7702a49c8f130",
      "SHA1=7d453dccb25bf36c411c92e2744c24f9b801225d",
      "SHA1=9648ad90ec683c63cc02a99111a002f9b00478d1",
      "SHA1=31cc8718894d6e6ce8c132f68b8caaba39b5ba7a",
      "SHA1=31fac347aa26e92db4d8c9e1ba37a7c7a2234f08",
      "SHA1=fde0fff1c3e4c053148748504d4b9e0cc97f37ec",
      "SHA1=73bac306292b4e9107147db94d0d836fdb071e33",
      "SHA1=9382981b05b1fb950245313992444bfa0db5f881",
      "SHA1=acb8e45ebd1252313ece94198df47edf9294e7d3",
      "SHA1=9c36600c2640007d3410dea8017573a113374873",
      "SHA1=53f776d9a183c42b93960b270dddeafba74eb3fb",
      "SHA1=1fdb2474908bdd2ee1e9bd3f224626f9361caab7",
      "SHA1=3533d0a54c7ccd83afd6be24f6582b30e4ca0aab",
      "SHA1=cb25a5125fb353496b59b910263209f273f3552d",
      "SHA1=a5f1b56615bdaabf803219613f43671233f2001c",
      "SHA1=6c7663de88a0fba1f63a984f926c6ef449059e38",
      "SHA1=e514dfadbeb4d2305988c3281bf105d252dee3a7",
      "SHA1=632c80a3c95cf589b03812539dea59594eaefae0",
      "SHA1=e6966e360038be3b9d8c9b2582eba4e263796084",
      "SHA1=675cc00de7c1ef508ccd0c91770c82342c0ad4ab",
      "SHA1=6ae26bde7ec27bd0fa971de6c7500eee34ee9b51",
      "SHA1=80e4808a7fe752cac444676dbbee174367fa2083",
      "SHA1=77b4f0c0b06e3dc2474d5e250b772dacaac14dd0",
      "SHA1=7277d965b9de91b4d8ea5eb8ae7fa3899eef63a2",
      "SHA1=3825ebb0b0664b5f0789371240f65231693be37d",
      "SHA1=de9469a5d01fb84afd41d176f363a66e410d46da",
      "SHA1=91568d7a82cc7677f6b13f11bea5c40cf12d281b",
      "SHA1=4b882748faf2c6c360884c6812dd5bcbce75ebff",
      "SHA1=599de57a5c05e27bb72c7b8a677e531d8e4bf8b5",
      "SHA1=1d373361d3129d11bc43f9b6dfa81d06e5ca8358",
      "SHA1=c5bd9f2b3a51ba0da08d7c84bab1f2d03a95e405",
      "SHA1=89165bbb761d6742ac2a6f5efbffc80c17990bd8",
      "SHA1=97812f334a077c40e8e642bb9872ac2c49ddb9a2",
      "SHA1=d417c0be261b0c6f44afdec3d5432100e420c3ed",
      "SHA1=37e6450c7cd6999d080da94b867ba23faa8c32fe",
      "SHA1=9481cd590c69544c197b4ee055056302978a7191",
      "SHA1=ff3e19cd461ddf67529a765cbec9cb81d84dc7da",
      "SHA1=6972314b6d6b0109b9d0a951eb06041f531f589b",
      "SHA1=dd94a2436994ac35db91e0ec9438b95e438d38c5",
      "SHA1=dcc852461895311b56e3ae774c8e90782a79c0b4",
      "SHA1=3489ed43bdd11ccbfc892baaeae8102ff7d22f25",
      "SHA1=e38e1efd98cd8a3cdb327d386db8df79ea08dccc",
      "SHA1=d4cf9296271a9c5c40b0fa34f69b6125c2d14457",
      "SHA1=10fb4ba6b2585ea02e7afb53ff34bf184eeb1a5d",
      "SHA1=f6793243ad20359d8be40d3accac168a15a327fb",
      "SHA1=b34a012887ddab761b2298f882858fa1ff4d99f1",
      "SHA1=71469dce9c2f38d0e0243a289f915131bf6dd2a8",
      "SHA1=10115219e3595b93204c70eec6db3e68a93f3144",
      "SHA1=161bae224cf184ed6c09c77fae866d42412c6d25",
      "SHA1=07f78a47f447e4d8a72ad4bc6a26427b9577ec82",
      "SHA1=2929de0b5b5e1ba1cce1908e9d800aa21f448b3d",
      "SHA1=745335bcdf02fb42df7d890a24858e16094f48fd",
      "SHA1=2a202830db58d5e942e4f6609228b14095ed2cab",
      "SHA1=0167259abd9231c29bec32e6106ca93a13999f90",
      "SHA1=c23eeb6f18f626ce1fd840227f351fa7543bb167",
      "SHA1=613a9df389ad612a5187632d679da11d60f6046a",
      "SHA1=1ce17c54c6884b0319d5aabbe7f96221f4838514",
      "SHA1=025c4e1a9c58bf10be99f6562476b7a0166c6b86",
      "SHA1=c3aafe8f67c6738489377031cb5a1197e99b202d",
      "SHA1=50c6b3cafc35462009d02c10f2e79373936dd7bb",
      "SHA1=6df35a0c2f6d7d39d24277137ea840078dafb812",
      "SHA1=f92faed3ef92fa5bc88ebc1725221be5d7425528",
      "SHA1=3bd1a88cc7dae701bc7085639e1c26ded3f8ccb3",
      "SHA1=a3ed5cbfbc17b58243289f3cf575bf04be49591d",
      "SHA1=552730553a1dea0290710465fb8189bdd0eaad42",
      "SHA1=0291d0457acaf0fe8ed5c3137302390469ce8b35",
      "SHA1=07f282db28771838d0e75d6618f70d76acfe6082",
      "SHA1=e6765d8866cad6193df1507c18f31fa7f723ca3e",
      "SHA1=22c9da04847c26188226c3a345e2126ef00aa19e",
      "SHA1=43501832ce50ccaba2706be852813d51de5a900f",
      "SHA1=cb3f30809b05cf02bc29d4a7796fb0650271e542",
      "SHA1=ed86bb62893e6ffcdfd2ecae2dea77fdf6bf9bde",
      "SHA1=3b6b35bca1b05fafbfc883a844df6d52af44ccdc",
      "SHA1=928b5971a0f7525209d599e2ef15c31717047022",
      "SHA1=b5696e2183d9387776820ef3afa388200f08f5a6",
      "SHA1=ebd8b7e964b8c692eea4a8c406b9cd0be621ebe2",
      "SHA1=fe18c58fbd0a83d67920e037d522c176704d2ca3",
      "SHA1=9c1c9032aa1e33461f35dbf79b6f2d061bfc6774",
      "SHA1=8e126f4f35e228fdd3aa78d533225db7122d8945",
      "SHA1=064de88dbbea67c149e779aac05228e5405985c7",
      "SHA1=30a80f560f18609c1123636a8a1a1ef567fa67a7",
      "SHA1=98130128685c8640a8a8391cb4718e98dd8fe542",
      "SHA1=a5914161f8a885702427cf75443fb08d28d904f0",
      "SHA1=48f03a13b0f6d3d929a86514ce48a9352ffef5ad",
      "SHA1=fff4f28287677caabc60c8ab36786c370226588d",
      "SHA1=bb5b17cff0b9e15f1648b4136e95bd20d899aef5",
      "SHA1=b2f5d3318aab69e6e0ca8da4a4733849e3f1cee2",
      "SHA1=635a39ff5066e1ac7c1c5995d476d8c233966dda",
      "SHA1=5ed22c0033aed380aa154e672e8db3a2d4c195c4",
      "SHA1=87e20486e804bfff393cc9ad9659858e130402a2",
      "SHA1=4dd86ff6f7180abebcb92e556a486abe7132754c",
      "SHA1=39169c9b79502251ca2155c8f1cd7e63fd9a42e9",
      "SHA1=7f7d144cc80129d0db3159ea5d4294c34b79b20a",
      "SHA1=8692274681e8d10c26ddf2b993f31974b04f5bf0",
      "SHA1=ea4a405445bb6e58c16b81f6d5d2c9a9edde419b",
      "SHA1=da970a01cecff33a99c217a42297cec4d1fe66d6",
      "SHA1=1f3799fed3cf43254fe30dcdfdb8dc02d82e662b",
      "SHA1=3d2309f7c937bfcae86097d716a8ef66c1337a3c",
      "SHA1=02a9314109e47c5ce52fa553ea57070bf0f8186a",
      "SHA1=91f832f46e4c38ecc9335460d46f6f71352cffed",
      "SHA1=76568d987f8603339b8d1958f76de2b957811f66",
      "SHA1=e841c8494b715b27b33be6f800ca290628507aba",
      "SHA1=b555aad38df7605985462f3899572931ee126259",
      "SHA1=115edd175c346fd3fbc9f113ee5ccd03b5511ee1",
      "SHA1=3d27013557b5e68e7212a2f78dfe60c5a2a46327",
      "SHA1=bb6ef5518df35d9508673d5011138add8c30fc27",
      "SHA1=9086e670e3a4518c0bcdf0da131748d4085ef42b",
      "SHA1=f6728821eddd14a21a9536e0f138c6d71cbd9307",
      "SHA1=34b677fba9dcab9a9016332b3332ce57f5796860",
      "SHA1=a63e9ecdebaf4ef9c9ec3362ff110b8859cc396d",
      "SHA1=8cd9df52b20b8f792ac53f57763dc147d7782b1e",
      "SHA1=fcae2ea5990189f6f230b51e398e3000b71897f2",
      "SHA1=27371f45f42383029c3c2e6d64a22e35dc772a72",
      "SHA1=b6eb40ea52b47f03edb8f45e2e431b5f666df8c5",
      "SHA1=9f27987c32321f8da099efc1dc60a73f8f629d3a",
      "SHA1=40372b4de2db020ce2659e1de806d4338fd7ebef",
      "SHA1=18693de1487c55e374b46a7728b5bf43300d4f69",
      "SHA1=b2f955b3e6107f831ebe67997f8586d4fe9f3e98",
      "SHA1=005754dab657ddc6dae28eee313ca2cc6a0c375c",
      "SHA1=0bec69c1b22603e9a385495fbe94700ac36b28e5",
      "SHA1=bd39ef9c758e2d9d6037e067fbb2c1f2ac7feac8",
      "SHA1=23f562f8d5650b2fb92382d228013f2e36e35d6c",
      "SHA1=a48aa80942fc8e0699f518de4fd6512e341d4196",
      "SHA1=e42bd2f585c00a1d6557df405246081f89542d15",
      "SHA1=bf5515fcf120c2548355d607cfd57e9b3e0af6e9",
      "SHA1=89a74d0e9fd03129082c5b868f5ad62558ca34fd",
      "SHA1=948368fe309652e8d88088d23e1df39e9c2b6649",
      "SHA1=a14cd928c60495777629be283c1d5b8ebbab8c0d",
      "SHA1=1f25f54e9b289f76604e81e98483309612c5a471",
      "SHA1=25bf4e30a94df9b8f8ab900d1a43fd056d285c9d",
      "SHA1=d1fb740210c1fa2a52f6748b0588ae77de590b9d",
      "SHA1=dac68b8ee002d5bb61be3d59908a61a26efb7c09",
      "SHA1=a56598e841ae694ac78c37bf4f8c09f9eaf3271f",
      "SHA1=465abe9634c199a5f80f8a4f77ec3118c0d69652",
      "SHA1=a0cefb5b55f7a7a145b549613e26b6805515a1ad",
      "SHA1=36dca91fb4595de38418dffc3506dc78d7388c2c",
      "SHA1=92138cfc14f9e2271f641547e031d5d63c6de19a",
      "SHA1=fcf9978cf1af2e9b1e2eaf509513664dfcc1847b",
      "SHA1=d02403f85be6f243054395a873b41ef8a17ea279",
      "SHA1=4da007dd298723f920e194501bb49bab769dfb14",
      "SHA1=85076aa3bffb40339021286b73d72dd5a8e4396a",
      "SHA1=221717a48ee8e2d19470579c987674f661869e17",
      "SHA1=a249278a668d4df30af9f5d67ebb7d2cd160beaa",
      "SHA1=6b5aa51f4717d123a468e9e9d3d154e20ca39d56",
      "SHA1=b5a8e2104d76dbb04cd9ffe86784113585822375",
      "SHA1=02534b5b510d978bac823461a39f76b4f0ac5aa3",
      "SHA1=538bb45f30035f39d41bd13818fe0c0061182cfe",
      "SHA1=6d09d826581baa1817be6fbd44426db9b05f1909",
      "SHA1=197811ec137e9916e6692fc5c28f6d6609ffc20e",
      "SHA1=c3ca396b5af2064c6f7d05fa0fb697e68d0b9631",
      "SHA1=cf9baf57e16b73d7a4a99dd0c092870deba1a997",
      "SHA1=0320534df24a37a245a0b09679a5adb27018fb5f",
      "SHA1=4c8349c6345c8d6101fb896ea0a74d0484c56df0",
      "SHA1=9b2ef5f7429d62342163e001c7c13fb866dbe1ef",
      "SHA1=6abbc3003c7aa69ce79cbbcd2e3210b07f21d202",
      "SHA1=062457182ab08594c631a3f897aeb03c6097eb77",
      "SHA1=947c76c8c8ba969797f56afd1fa1d1c4a1e3ed25",
      "SHA1=d6de8211dba7074d92b5830618176a3eb8eb6670",
      "SHA1=8302802b709ad242a81b939b6c90b3230e1a1f1e",
      "SHA1=492e40b01a9a6cec593691db4838f20b3eaeacc5",
      "SHA1=83506de48bd0c50ea00c9e889fe980f56e6c6e1b",
      "SHA1=fe54a1acc5438883e5c1bba87b78bb7322e2c739",
      "SHA1=020580278d74d0fe741b0f786d8dca7554359997",
      "SHA1=3c1c3f5f5081127229ba0019fbf0efc2a9c1d677",
      "SHA1=e2d98e0e178880f10434059096f936b2c06ed8f4",
      "SHA1=03506a2f87d1523e844fba22e7617ab2a218b4b7",
      "SHA1=fee00dde8080c278a4c4a6d85a5601edc85a1b3d",
      "SHA1=ba430f3c77e58a4dc1a9a9619457d1c45a19617f",
      "SHA1=c257aa4094539719a3c7b7950598ef872dbf9518",
      "SHA1=bc62fe2b38008f154fc9ea65d851947581b52f49",
      "SHA1=fe237869b2b496deb52c0bc718ada47b36fc052e",
      "SHA1=0a62c574603158d2d0c3be2a43c6bb0074ed297c",
      "SHA1=86f34eaea117f629297218a4d196b5729e72d7b9",
      "SHA1=e0b263f2d9c08f27c6edf5a25aa67a65c88692b0",
      "SHA256=9dc7beb60a0a6e7238fc8589b6c2665331be1e807b4d2b3ddd1c258dbbd3e2f7",
      "SHA256=06ddf49ac8e06e6b83fccba1141c90ea01b65b7db592c54ffe8aa6d30a75c0b8",
      "SHA256=822982c568b6f44b610f8dc4ab5d94795c33ae08a6a608050941264975c1ecdb",
      "SHA256=082a79311da64b6adc3655e79aa090a9262acaac3b917a363b9571f520a17f6a",
      "SHA256=618b15970671700188f4102e5d0638184e2723e8f57f7e917fa49792daebdadb",
      "SHA256=5b932eab6c67f62f097a3249477ac46d80ddccdc52654f8674060b4ddf638e5d",
      "SHA256=82ac05fefaa8c7ee622d11d1a378f1d255b647ab2f3200fd323cc374818a83f2",
      "SHA256=29d765e29d2f06eb511ee88b2e514c9df1a9020a768ddd3d2278d9045e9cdb4a",
      "SHA256=f461414a2596555cece5cfee65a3c22648db0082ca211f6238af8230e41b3212",
      "SHA256=beef40f1b4ce0ff2ee5c264955e6b2a0de6fe4089307510378adc83fad77228b",
      "SHA256=9a42fa1870472c38a56c0a70f62e57a3cdc0f5bc142f3a400d897b85d65800ac",
      "SHA256=f03f0fb3a26bb83e8f8fa426744cf06f2e6e29f5220663b1d64265952b8de1a1",
      "SHA256=50819a1add4c81c0d53203592d6803f022443440935ff8260ff3b6d5253c0c76",
      "SHA256=6b5cf41512255237064e9274ca8f8a3fef820c45aa6067c9c6a0e6f5751a0421",
      "SHA256=575e58b62afab094c20c296604dc3b7dd2e1a50f5978d8ee24b7dca028e97316",
      "SHA256=26bea3b3ab2001d91202f289b7e41499d810474607db7a0893ceab74f5532f47",
      "SHA256=b169a5f643524d59330fafe6e3e328e2179fc5116ee6fae5d39581467d53ac03",
      "SHA256=b8807e365be2813b7eccd2e4c49afb0d1e131086715638b7a6307cd7d7e9556c",
      "SHA256=28f5aa194a384680a08c0467e94a8fc40f8b0f3f2ac5deb42e0f51a80d27b553",
      "SHA256=9bb09752cf3a464455422909edef518ac18fe63cf5e1e8d9d6c2e68db62e0c87",
      "SHA256=8578bff36e3b02cc71495b647db88c67c3c5ca710b5a2bd539148550595d0330",
      "SHA256=a32dc2218fb1f538fba33701dfd9ca34267fda3181e82eb58b971ae8b78f0852",
      "SHA256=2c14bea0d85c9cad5c5f5c8d0e5442f6deb9e93fe3ad8ea5e8e147821c6f9304",
      "SHA256=23e89fd30a1c7db37f3ea81b779ce9acf8a4294397cbb54cff350d54afcfd931",
      "SHA256=f6c316e2385f2694d47e936b0ac4bc9b55e279d530dd5e805f0d963cb47c3c0d",
      "SHA256=b0a27ac1a8173413de13860d2b2e34cb6bc4d1149f94b62d319042e11d8b004c",
      "SHA256=897f2bbe81fc3b1ae488114b93f3eb0133a85678d061c7a6f718507971f33736",
      "SHA256=497a836693be1b330993e2be64f6c71bf290c127faca1c056abd0dc374654830",
      "SHA256=8e035beb02a411f8a9e92d4cf184ad34f52bbd0a81a50c222cdd4706e4e45104",
      "SHA256=f9f2091fccb289bcf6a945f6b38676ec71dedb32f3674262928ccaf840ca131a",
      "SHA256=40556dd9b79b755cc0b48d3d024ceb15bd2c0e04960062ab2a85cd7d4d1b724a",
      "SHA256=ac5fb90e88d8870cd5569e661bea98cf6b001d83ab7c65a5196ea3743146939a",
      "SHA256=12b0000698b79ea3c8178b9e87801cc34bad096a151a8779559519deafd4e3f0",
      "SHA256=9e56e96df36237e65b3d7dbc490afdc826215158f6278cd579c576c4b455b392",
      "SHA256=ec96b15ce218f97ec1d8f07f13b052d274c4c8438f31daf246ccfaaee5e1bebd",
      "SHA256=da70fa44290f949e9b3e0fcfe0503de46e82e0472e8e3c360da3fd2bfa364eee",
      "SHA256=accb1a6604efb1b3ce9345c9fd62fe717a84c3e089e09c638e461df89193ef01",
      "SHA256=083f821d90e607ed93221e71d4742673e74f573d0755a96ad17d1403f65a2254",
      "SHA256=c7bccc6f38403def4690e00a0b31eda05973d82be8953a3379e331658c51b231",
      "SHA256=0740359baef32cbb0b14a9d1bd3499ea2e770ff9b1c85898cfac8fd9aca4fa39",
      "SHA256=32882949ea084434a376451ff8364243a50485a3b4af2f2240bb5f20c164543d",
      "SHA256=3ca5d47d076e99c312578ef6499e1fa7b9db88551cfc0f138da11105aca7c5e1",
      "SHA256=f8236fc01d4efaa48f032e301be2ebba4036b2cd945982a29046eca03944d2ae",
      "SHA256=05b146a48a69dd62a02759487e769bd30d39f16374bc76c86453b4ae59e7ffa4",
      "SHA256=8922be14c657e603179f1dd94dc32de7c99d2268ac92d429c4fdda7396c32e50",
      "SHA256=aafa642ca3d906138150059eeddb6f6b4fe9ad90c6174386cfe13a13e8be47d9",
      "SHA256=087270d57f1626f29ba9c25750ca19838a869b73a1f71af50bdf37d6ff776212",
      "SHA256=008fa89822b7a1f91e5843169083202ea580f7b06eb6d5cae091ba844d035f25",
      "SHA256=b2486f9359c94d7473ad8331b87a9c17ca9ba6e4109fd26ce92dff01969eaa09",
      "SHA256=dfc80e0d468a2c115a902aa332a97e3d279b1fc3d32083e8cf9a4aadf3f54ad1",
      "SHA256=0d10c4b2f56364b475b60bd2933273c8b1ed2176353e59e65f968c61e93b7d99",
      "SHA256=5bc3994612624da168750455b363f2964e1861dba4f1c305df01b970ac02a7ae",
      "SHA256=36c65aeb255c06898ffe32e301030e0b74c8bca6fe7be593584b8fdaacd4e475",
      "SHA256=30e083cd7616b1b969a92fd18cf03097735596cce7fcf3254b2ca344e526acc2",
      "SHA256=15cf366f7b3ee526db7ce2b5253ffebcbfaa4f33a82b459237c049f854a97c0c",
      "SHA256=be70be9d84ae14ea1fa5ec68e2a61f6acfe576d965fe51c6bac78fba01a744fb",
      "SHA256=7b846b0a717665e4d9fb313f25d1f6a5b782e495387aea45cf87ad3c049ac0db",
      "SHA256=85b9d7344bf847349b5d58ebe4d44fd63679a36164505271593ef1076aa163b2",
      "SHA256=749b0e8c8c8b7dda8c2063c708047cfe95afa0a4d86886b31a12f3018396e67c",
      "SHA256=4999541c47abd4a7f2a002c180ae8d31c19804ce538b85870b8db53d3652862b",
      "SHA256=56066ed07bad3b5c1474e8fae5ee2543d17d7977369b34450bd0775517e3b25c",
      "SHA256=e6a7b0bc01a627a7d0ffb07faddb3a4dd96b6f5208ac26107bdaeb3ab1ec8217",
      "SHA256=0f58e09651d48d2b1bcec7b9f7bb85a2d1a7b65f7a51db281fe0c4f058a48597",
      "SHA256=cf9451c9ccc5509b9912965f79c2b95eb89d805b2a186d7521d3a262cf5a7a37",
      "SHA256=2456a7921fa8ab7b9779e5665e6b42fccc019feb9e49a9a28a33ec0a4bb323c4",
      "SHA256=7a7e8df7173387aec593e4fe2b45520ea3156c5f810d2bb1b2784efd1c922376",
      "SHA256=eab9b5b7e5fab1c2d7d44cd28f13ae8bb083d9362d2b930d43354a3dfd38e05a",
      "SHA256=c7cd14c71bcac5420872c3d825ff6d4be6a86f3d6a8a584f1a756541efff858e",
      "SHA256=ece76b79feafb38ae4371e104b6dcbb4253ff3b2acbe5bd14ce6e47525c24f4a",
      "SHA256=42b22faa489b5de936db33f12184f6233198bdf851a18264d31210207827ba25",
      "SHA256=d7aa8abdda8a68b8418e86bef50c19ef2f34bc66e7b139e43c2a99ab48c933be",
      "SHA256=4af8192870afe18c77381dfaf8478f8914fa32906812bb53073da284a49ae4c7",
      "SHA256=21617210249d2a35016e8ca6bd7a1edda25a12702a2294d56010ee8148637f5a",
      "SHA256=c0d88db11d0f529754d290ed5f4c34b4dba8c4f2e5c4148866daabeab0d25f9c",
      "SHA256=19dfacea1b9f19c0379f89b2424ceb028f2ce59b0db991ba83ae460027584987",
      "SHA256=4136f1eb11cc463a858393ea733d5f1c220a3187537626f7f5d63eccf7c5a03f",
      "SHA256=f6157e033a12520c73dcedf8e49cd42d103e5874c34d6527bb9de25a5d26e5ad",
      "SHA256=e7af7bcb86bd6bab1835f610671c3921441965a839673ac34444cf0ce7b2164e",
      "SHA256=f9b01406864ab081aa77eef4ad15cb2dd2f830d1ef54f52622a59ff1aeb05ba5",
      "SHA256=a2d32c28eb5945b85872697d7cfbe87813c09a0e1be28611563755f68b9cb88b",
      "SHA256=569fe70bedd0df8585689b0e88ad8bd0544fdf88b9dbfc2076f4bdbcf89c28aa",
      "SHA256=a78c9871da09fab21aec9b88a4e880f81ecb1ed0fa941f31cc2f041067e8e972",
      "SHA256=b8c71e1844e987cd6f9c2baf28d9520d4ccdd8593ce7051bb1b3c9bf1d97076a",
      "SHA256=af7ca247bf229950fb48674b21712761ac650d33f13a4dca44f61c59f4c9ac46",
      "SHA256=6908ebf52eb19c6719a0b508d1e2128f198d10441551cbfb9f4031d382f5229f",
      "SHA256=06a0ec9a316eb89cb041b1907918e3ad3b03842ec65f004f6fa74d57955573a4",
      "SHA256=fd223833abffa9cd6cc1848d77599673643585925a7ee51259d67c44d361cce8",
      "SHA256=31b66a57fae0cc28a6a236d72a35c8b6244f997e700f9464f9cbf800dbf8bee6",
      "SHA256=2fd43a749b5040ebfafd7cdbd088e27ef44341d121f313515ebde460bf3aaa21",
      "SHA256=773b4a1efb9932dd5116c93d06681990759343dfe13c0858d09245bc610d5894",
      "SHA256=52f3905bbd97dcd2dbd22890e5e8413b9487088f1ee2fa828030a6a45b3975fd",
      "SHA256=86047bb1969d1db455493955fd450d18c62a3f36294d0a6c3732c88dfbcc4f62",
      "SHA256=aaf04d89fd15bc61265e545f8e1da80e20f59f90058ed343c62ee24358e3af9e",
      "SHA256=e5ddfa39540d4e7ada56cdc1ebd2eb8c85a408ec078337488a81d1c3f2aaa4ff",
      "SHA256=8b30b2dc36d5e8f1ffc7281352923773fb821cdf66eb6516f82c697a524b599b",
      "SHA256=469713c76c7a887826611b8c7180209a8bb6250f91d0f1eb84ac4d450ef15870",
      "SHA256=a906251667a103a484a6888dca3e9c8c81f513b8f037b98dfc11440802b0d640",
      "SHA256=49c827cf48efb122a9d6fd87b426482b7496ccd4a2dbca31ebbf6b2b80c98530",
      "SHA256=bcca03ce1dd040e67eb71a7be0b75576316f0b6587b2058786fda8b6f0a5adfd",
      "SHA256=0aab2deae90717a8876d46d257401d265cf90a5db4c57706e4003c19eee33550",
      "SHA256=406b844f4b5c82caf26056c67f9815ad8ecf1e6e5b07d446b456e5ff4a1476f9",
      "SHA256=10ad50fcb360dcab8539ea322aaf2270565dc835b7535790937348523d723d6b",
      "SHA256=c4f041de66ec8cc5ab4a03bbc46f99e073157a4e915a9ab4069162de834ffc5c",
      "SHA256=139f8412a7c6fdc43dcfbbcdba256ee55654eb36a40f338249d5162a1f69b988",
      "SHA256=793b78e70b3ae3bb400c5a8bc4d2d89183f1d7fc70954aed43df7287248b6875",
      "SHA256=492113a223d6a3fc110059fe46a180d82bb8e002ef2cd76cbf0c1d1eb8243263",
      "SHA256=b34e2d9f3d4ef59cf7af18e17133a6a06509373e69e33c8eecb2e30501d0d9e4",
      "SHA256=f936ec4c8164cbd31add659b61c16cb3a717eac90e74d89c47afb96b60120280",
      "SHA256=60ee78a2b070c830fabb54c6bde0d095dff8fad7f72aa719758b3c41c72c2aa9",
      "SHA256=c8ae217860f793fce3ad0239d7b357dba562824dd7177c9d723ca4d4a7f99a12",
      "SHA256=29348ebe12d872c5f40e316a0043f7e5babe583374487345a79bad0ba93fbdfe",
      "SHA256=5f6fec8f7890d032461b127332759c88a1b7360aa10c6bd38482572f59d2ba8b",
      "SHA256=e8ec06b1fa780f577ff0e8c713e0fd9688a48e0329c8188320f9eb62dfc0667f",
      "SHA256=770f33259d6fb10f4a32d8a57d0d12953e8455c72bb7b60cb39ce505c507013a",
      "SHA256=b0b80a11802b4a8ca69c818a03e76e7ef57c2e293de456439401e8e6073f8719",
      "SHA256=bc49cb96f3136c3e552bf29f808883abb9e651040415484c1736261b52756908",
      "SHA256=4c89c907b7525b39409af1ad11cc7d2400263601edafc41c935715ef5bd145de",
      "SHA256=0440ef40c46fdd2b5d86e7feef8577a8591de862cfd7928cdbcc8f47b8fa3ffc",
      "SHA256=200f98655d1f46d2599c2c8605ebb7e335fee3883a32135ca1a81e09819bc64a",
      "SHA256=b0eb4d999e4e0e7c2e33ff081e847c87b49940eb24a9e0794c6aa9516832c427",
      "SHA256=673bbc7fa4154f7d99af333014e888599c27ead02710f7bc7199184b30b38653",
      "SHA256=4b97d63ebdeda6941bb8cef5e94741c6cca75237ca830561f2262034805f0919",
      "SHA256=d50cb5f4b28c6c26f17b9d44211e515c3c0cc2c0c4bf24cd8f9ed073238053ad",
      "SHA256=62764ddc2dce74f2620cd2efd97a2950f50c8ac5a1f2c1af00dc5912d52f6920",
      "SHA256=6994b32e3f3357f4a1d0abe81e8b62dd54e36b17816f2f1a80018584200a1b77",
      "SHA256=751e9376cb7cb9de63e1808d43579d787d3f6d659173038fe44a2d7fdb4fd17e",
      "SHA256=87565ff08a93a8ff41ea932bf55dec8e0c7e79aba036507ea45df9d81cb36105",
      "SHA256=2da2b883e48e929f5365480d487590957d9e6582cc6da2c0b42699ba85e54fe2",
      "SHA256=627e13da6a45006fff4711b14754f9ccfac9a5854d275da798a22f3a68dd1eaa",
      "SHA256=94ba4bcbdb55d6faf9f33642d0072109510f5c57e8c963d1a3eb4f9111f30112",
      "SHA256=704c6ffe786bc83a73fbdcd2edd50f47c3b5053da7da6aa4c10324d389a31db4",
      "SHA256=d41e39215c2c1286e4cd3b1dc0948adefb161f22bc3a78756a027d41614ee4ff",
      "SHA256=0f7bfa10075bf5c193345866333d415509433dbfe5a7d45664b88d72216ff7c3",
      "SHA256=14b89298134696f2fd1b1df0961d36fa6354721ea92498a349dc421e79447925",
      "SHA256=3b2cd65a4fbdd784a6466e5196bc614c17d1dbaed3fd991d242e3be3e9249da6",
      "SHA256=2ce4f8089b02017cbe86a5f25d6bc69dd8b6f5060c918a64a4123a5f3be1e878",
      "SHA256=e99580e25f419b5ad90669e0c274cf63d30efa08065d064a863e655bdf77fb59",
      "SHA256=a74e8f94d2c140646a8bb12e3e322c49a97bd1b8a2e4327863d3623f43d65c66",
      "SHA256=47356707e610cfd0be97595fbe55246b96a69141e1da579e6f662ddda6dc5280",
      "SHA256=18c909a2b8c5e16821d6ef908f56881aa0ecceeaccb5fa1e54995935fcfd12f7",
      "SHA256=95e5b5500e63c31c6561161a82f7f9373f99b5b1f54b018c4866df4f2a879167",
      "SHA256=5c1585b1a1c956c7755429544f3596515dfdf928373620c51b0606a520c6245a",
      "SHA256=82b7fa34ad07dbf9afa63b2f6ed37973a1b4fe35dee90b3cf5c788c15c9f08f7",
      "SHA256=a85d3fd59bb492a290552e5124bfe3f9e26a3086d69d42ccc44737b5a66673ec",
      "SHA256=ea50f22daade04d3ca06dedb497b905215cba31aae7b4cab4b533fda0c5be620",
      "SHA256=d032001eab6cad4fbef19aab418650ded00152143bd14507e17d62748297c23f",
      "SHA256=4d42678df3917c37f44a1506307f1677b9a689efcf350b1acce7e6f64b514905",
      "SHA256=30061ef383e18e74bb067fbca69544f1a7544e8dc017d4e7633d8379aff4c3c3",
      "SHA256=7433f14b40c674c5e87b6210c330d5bcaf2f6f52d632ae29e9b7cf3ca405665b",
      "SHA256=818787057fc60ac8b957aa37d750aa4bace8e6a07d3d28b070022ee6dcd603ab",
      "SHA256=c4fb31e3f24e40742a1b9855a2d67048fe64b26d8d2dbcec77d2d5deeded2bcc",
      "SHA256=5295080de37d4838e15dec4e3682545033d479d3d9ac28d74747c086559fb968",
      "SHA256=7824931e55249a501074a258b4f65cd66157ee35672ba17d1c0209f5b0384a28",
      "SHA256=07759750fbb93c77b5c3957c642a9498fcff3946a5c69317db8d6be24098a4a0",
      "SHA256=51805bb537befaac8ce28f2221624cb4d9cefdc0260bc1afd5e0bc97bf1f9f93",
      "SHA256=e6f764c3b5580cd1675cbf184938ad5a201a8c096607857869bd7c3399df0d12",
      "SHA256=2faf95a3405578d0e613c8d88d534aa7233da0a6217ce8475890140ab8fb33c8",
      "SHA256=af4f42197f5ce2d11993434725c81ecb6f54025110dedf56be8ffc0e775d9895",
      "SHA256=baf7fbc4743a81eb5e4511023692b2dfdc32ba670ba3e4ed8c09db7a19bd82d3",
      "SHA256=a42f4ae69b8755a957256b57eb3d319678eab81705f0ffea0d649ace7321108f",
      "SHA256=4bca0a401b364a5cc1581a184116c5bafa224e13782df13272bc1b748173d1be",
      "SHA256=e4b2c0aa28aac5e197312a061b05363e2e0387338b28b23272b5b6659d29b1d8",
      "SHA256=69866557566c59772f203c11f5fba30271448e231b65806a66e48f41e3804d7f",
      "SHA256=93aa3066ae831cdf81505e1bc5035227dc0e8f06ebbbb777832a17920c6a02fe",
      "SHA256=bed4285d0f8d18f17ddaa53a98a475c87c04c4d167499e24c770da788e5d45f4",
      "SHA256=fa9abb3e7e06f857be191a1e049dd37642ec41fb2520c105df2227fcac3de5d5",
      "SHA256=07beac65e28ee124f1da354293a3d6ad7250ed1ce29b8342acfd22252548a5af",
      "SHA256=9a67626fb468d3f114c23ac73fd8057f43d06393d3eca04da1d6676f89da2d40",
      "SHA256=7f4555a940ce1156c9bcea9a2a0b801f9a5e44ec9400b61b14a7b1a6404ffdf6",
      "SHA256=7a84703552ae032a0d1699a081e422ed6c958bbe56d5b41839c8bfa6395bee1d",
      "SHA256=ddf427ce55b36db522f638ba38e34cd7b96a04cb3c47849b91e7554bfd09a69a",
      "SHA256=64d4370843a07e25d4ceb68816015efcaeca9429bb5bb692a88e615b48c7da96",
      "SHA256=c8f9e1ad7b8cce62fba349a00bc168c849d42cfb2ca5b2c6cc4b51d054e0c497",
      "SHA256=fefc070a5f6a9c0415e1c6f44512a33e8d163024174b30a61423d00d1e8f9bf2",
      "SHA256=8d9a2363b757d3f127b9c6ed8f7b8b018e652369bc070aa3500b3a978feaa6ce",
      "SHA256=d43520128871c83b904f3136542ea46644ac81a62d51ae9d3c3a3f32405aad96",
      "SHA256=efa56907b9d0ec4430a5d581f490b6b9052b1e979da4dab6a110ab92e17d4576",
      "SHA256=1d23ab46ad547e7eef409b40756aae9246fbdf545d13946f770643f19c715e80",
      "SHA256=62036cdf3663097534adf3252b921eed06b73c2562655eae36b126c7d3d83266",
      "SHA256=6661320f779337b95bbbe1943ee64afb2101c92f92f3d1571c1bf4201c38c724",
      "SHA256=3033ff03e6f523726638b43d954bc666cdd26483fa5abcf98307952ff88f80ee",
      "SHA256=6964a5d85639baee288555797992861232e75817f93028b50b8c6d34aa38b05b",
      "SHA256=06c5ebd0371342d18bc81a96f5e5ce28de64101e3c2fd0161d0b54d8368d2f1f",
      "SHA256=1485c0ed3e875cbdfc6786a5bd26d18ea9d31727deb8df290a1c00c780419a4e",
      "SHA256=6839fcae985774427c65fe38e773aa96ec451a412caa5354ad9e2b9b54ffe6c1",
      "SHA256=deade507504d385d8cae11365a2ac9b5e2773ff9b61624d75ffa882d6bb28952",
      "SHA256=c42c1e5c3c04163bf61c3b86b04a5ec7d302af7e254990cef359ac80474299da",
      "SHA256=8dafe5f3d0527b66f6857559e3c81872699003e0f2ffda9202a1b5e29db2002e",
      "SHA256=88076e98d45ed3adf0c5355411fe8ca793eb7cec1a1c61f5e1ec337eae267463",
      "SHA256=b0f1fbadc1d7a77557d3d836f7698bd986a3ec9fc5d534ad3403970f071176f7",
      "SHA256=bcb774b6f6ff504d2db58096601bc5cb419c169bfbeaa3af852417e87d9b2aa0",
      "SHA256=4dc24fd07f8fb854e685bc540359c59f177de5b91231cc44d6231e33c9e932b1",
      "SHA256=82b0e1d7a27b67f0e6dc39dc41e880bdaef5d1f69fcec38e08da2ed78e805ef9",
      "SHA256=ad938d15ecfd70083c474e1642a88b078c3cea02cdbddf66d4fb1c01b9b29d9a",
      "SHA256=443c0ba980d4db9213b654a45248fd855855c1cc81d18812cae9d16729ff9a85",
      "SHA256=f3ec3f22639d45b3c865bb1ed7622db32e04e1dbc456298be02bf1f3875c3aac",
      "SHA256=0181d60506b1f3609217487c2c737621d637e1232f243f68c662d045f44d4873",
      "SHA256=c13f5bc4edfbe8f1884320c5d76ca129d00de41a1e61d45195738f125dfe60a7",
      "SHA256=8684aec77b4c3cafc1a6594de7e95695fa698625d4206a6c4b201875f76a5b38",
      "SHA256=c4c9c84b211899ceb0d18a839afa497537a7c7c01ab481965a09788a9e16590c",
      "SHA256=d37996abc8efb29f1ccbb4335ce9ba9158bec86cc4775f0177112e87e4e3be5c",
      "SHA256=1a5c08d40a5e73b9fe63ea5761eaec8f41d916ca3da2acbc4e6e799b06af5524",
      "SHA256=9c2f3e9811f7d0c7463eaa1ee6f39c23f902f3797b80891590b43bbe0fdf0e51",
      "SHA256=bb2422e96ea993007f25c71d55b2eddfa1e940c89e895abb50dd07d7c17ca1df",
      "SHA256=94c71954ac0b1fd9fa2bd5c506a16302100ba75d9f84f39ee9b333546c714601",
      "SHA256=6d68d8a71a11458ddf0cbb73c0f145bee46ef29ce03ad7ece6bd6aa9d31db9b7",
      "SHA256=80e4c83cfa9d675a6746ab846fa5da76d79e87a9297e94e595a2d781e02673b3",
      "SHA256=e858de280bd72d7538386a73e579580a6d5edba87b66b3671dc180229368be19",
      "SHA256=ee7b8eb150df2788bb9d5fe468327899d9f60d6731c379fd75143730a83b1c55",
      "SHA256=8206ce9c42582ac980ff5d64f8e3e310bc2baa42d1a206dd831c6ab397fbd8fe",
      "SHA256=4f02aed3750bc6a924c75e774404f259f721d8f4081ed68aa01cf73ca5430f85",
      "SHA256=81c7bb39100d358f8286da5e9aa838606c98dfcc263e9a82ed91cd438cb130d1",
      "SHA256=0f98492c92e35042b09032e3d9aedc357e4df94fc840217fa1091046f9248a06",
      "SHA256=9b1b15a3aacb0e786a608726c3abfc94968915cedcbd239ddf903c4a54bfcf0c",
      "SHA256=b9dad0131c51e2645e761b74a71ebad2bf175645fa9f42a4ab0e6921b83306e3",
      "SHA256=26ef7b27d1afb685e0c136205a92d29b1091e3dcf6b7b39a4ec03fbbdb57cb55",
      "SHA256=a1e6b431534258954db07039117b3159e889c6b9e757329bbd4126383c60c778",
      "SHA256=d25b5e4d07f594c640dcd93cfc8ab3f0a38348150bd0bfae89f404fbb0d811c6",
      "SHA256=1ef7afea0cf2ef246ade6606ef8b7195de9cd7a3cd7570bff90ba1e2422276f6",
      "SHA256=083a311875173f8c4653e9bbbabb689d14aa86b852e7fa9f5512fc60e0fd2c43",
      "SHA256=89698cad598a56f9e45efffd15d1841e494a2409cc12279150a03842cd6bb7f3",
      "SHA256=a7a665a695ec3c0f862a0d762ad55aff6ce6014359647e7c7f7e3c4dc3be81b7",
      "SHA256=02ebf848fa618eba27065db366b15ee6629d98f551d20612ac38b9f655f37715",
      "SHA256=8b32fc8b15363915605c127ccbf5cbe71778f8dfbf821a25455496e969a01434",
      "SHA256=ee525b90053bb30908b5d7bf4c5e9b8b9d6b7b5c9091a26fa25d30d3ad8ef5d0",
      "SHA256=41ad660820c41fc8b1860b13dc1fea8bc8cb2faceb36ed3e29d40d28079d2b1f",
      "SHA256=42ff11ddb46dfe5fa895e7babf88ee27790cde53a9139fc384346a89e802a327",
      "SHA256=36f45a42ebf2de6962db92aaf8845d7f9fd6895bedc31422adcf31c59a79602d",
      "SHA256=4bd4715d2a7af627da11513e32fab925c872babebdb7ff5675a75815fbf95021",
      "SHA256=4734a0a5d88f44a4939b8d812364cab6ca5f611b9b8ceebe27df6c1ed3a6d8a4",
      "SHA256=e8743094f002239a8a9d6d7852c7852e0bb63cd411b007bd8c194bcba159ef15",
      "SHA256=f0474e76cfd36e37e32cfe5c0a9e05ddee17dd5014d7aa8817ea3634a3540a3f",
      "SHA256=a0931e16cf7b18d15579e36e0a69edad1717b07527b5407f2c105a2f554224b2",
      "SHA256=52d5c35325ce701516f8b04380c9fbdb78ec6bcc13b444f758fdb03d545b0677",
      "SHA256=e1cb86386757b947b39086cc8639da988f6e8018ca9995dd669bdc03c8d39d7d",
      "SHA256=7662187c236003308a7951c2f49c0768636c492f8935292d02f69e59b01d236d",
      "SHA256=24c900024d213549502301c366d18c318887630f04c96bf0a3d6ba74e0df164f",
      "SHA256=b7956e31c2fcc0a84bcedf30e5f8115f4e74eed58916253a0c05c8be47283c57",
      "SHA256=96bf3ee7c6673b69c6aa173bb44e21fa636b1c2c73f4356a7599c121284a51cc",
      "SHA256=d7c81b0f3c14844f6424e8bdd31a128e773cb96cccef6d05cbff473f0ccb9f9c",
      "SHA256=0d676baac43d9e2d05b577d5e0c516fba250391ab0cb11232a4b17fd97a51e35",
      "SHA256=888491196bd8ff528b773a3e453eae49063ad31fb4ca0f9f2e433f8d35445440",
      "IMPHASH=8d070a93a45ed8ba6dba6bfbe0d084e7",
      "IMPHASH=7641a0c227f0a3a45b80bb8af43cd152",
      "IMPHASH=7df0d3ee663fc0e7c72a95e44ba4c82c",
      "IMPHASH=70e1caa5a322b56fd7951f1b2caacb0d",
      "IMPHASH=beceab354c66949088c9e5ed1f1ff2a4",
      "IMPHASH=caa08a0ba5f679b1e5bbae747cb9d626",
      "IMPHASH=420625b024fba72a24025defdf95b303",
      "IMPHASH=65ccc2c578a984c31880b6c5e65257d3",
      "IMPHASH=e717abe060bc5c34925fe3120ac22f45",
      "IMPHASH=41113a3a832353963112b94f4635a383",
      "IMPHASH=3866dd9fe63de457bdbf893bf7050ddf",
      "IMPHASH=3fd33d5b3b52e2db91983ac4b1d7a3c4",
      "IMPHASH=a998fe47a44bfbf2399968e21cfdf7ca",
      "IMPHASH=c9a6e83d931286d1604d1add8403e1e5",
      "IMPHASH=cf0eb2dce2ba2c9ff5dd0da794b8b372",
      "IMPHASH=ea37e43ffc7cfcba181c5cff37a9be1f",
      "IMPHASH=8e35c9460537092672b3c7c14bccc7e0",
      "IMPHASH=7bf14377888c429897eb10a85f70266c",
      "IMPHASH=b351627263648b1d220bb488e7ec7202",
      "IMPHASH=ce10082e1aa4c1c2bd953b4a7208e56a",
      "IMPHASH=a7bd820fa5b895fab06f20739c9f24b8",
      "IMPHASH=be0dd8b8e045356d600ee55a64d9d197",
      "IMPHASH=63fd1582ac2edee50f7ec7eedde38ee8",
      "IMPHASH=6c8d5c79a850eecc2fb0291cebda618d",
      "IMPHASH=c32d9a9af7f702814e1368c689877f3a",
      "IMPHASH=6b387c029257f024a43a73f38afb2629",
      "IMPHASH=df43355c636583e56e92142dcc69cc58",
      "IMPHASH=e3ee9131742bf9c9d43cb9a425e497dd",
      "IMPHASH=c214aac08575c139e48d04f5aee21585",
      "IMPHASH=3c5d2ffd06074f1b09c89465cc8bfbf7",
      "IMPHASH=059c6bd84285f4960e767f032b33f19b",
      "IMPHASH=a09170ef09c55cdca9472c02cb1f2647",
      "IMPHASH=fca0f3c7b6d79f494034b9d2a1f5921a",
      "IMPHASH=0262d4147f21d681f8519ab2af79283f",
      "IMPHASH=832219eb71b8bdb771f1d29d27b0acf4",
      "IMPHASH=514298d18002920ee5a917fc34426417",
      "IMPHASH=26ceec6572c630bdad60c984e51b7da4",
      "IMPHASH=dbf09dd3e675f15c7cc9b4d2b8e6cd90",
      "IMPHASH=4b47f6031c558106eee17655f8f8a32f",
      "IMPHASH=a6c4a7369500900fc172f9557cff22cf",
      "IMPHASH=3b49942ec6cef1898e97f741b2b5df8a",
      "IMPHASH=28dc68bb6d6bf4f6b2db8dd7588b2511",
      "IMPHASH=27f6dc8a247a22308dd1beba5086b302",
      "IMPHASH=7d017945bf90936a6c40f73f91ed02c2",
      "IMPHASH=d51f0f6034eb5e45f0ed4e9b7bbc9c97",
      "IMPHASH=0ad7da35304c75ccf859bc29fe9ed09e",
      "IMPHASH=bf9d32a6ab9effcd2fd6a734e5be98f9",
      "IMPHASH=87fd2b54ed568e2294300e164b8c46f7",
      "IMPHASH=2de3451f3e7b02970582bb8f9fd8c73a",
      "IMPHASH=e97dc162f416bf06745bf9ffdf78a0ff",
      "IMPHASH=2a008187d4a73284ddcc43f1b727b513",
      "IMPHASH=f8e4844312e81dbdb4e8e95e2ad2c127",
      "IMPHASH=4c7cc13a110ccdbb932bb9d7d42efdf4",
      "IMPHASH=45bfe170e0cd654bc1e2ae3fca3ac3f4",
      "IMPHASH=3db9de43d5d530c10d0cd2d43c7a0771"
    ]
  },
  "condition": "selection"
}

#### Detections Relationships

- attack.privilege_escalation
- attack.t1543.003
- attack.t1068

#### Sigma Unique ID

05296024-fe8a-4baf-8f3d-9a5f5624ceb2

#### Detection Authors

Nasreddine Bencherchali (Nextron Systems)

</ol>
</details>

## Yara

<details>
<ol

#### Raw Yara Rule(s)

`N/A`

#### Yara Location(s)

uri://aso/rules/0day.yara

#### Yara Confidence Level

experimental

#### Yara Assurance Level

high

#### Yara Query

N/A

#### Detections Relationships

N/A

#### Yara Unique ID

7bbc309f-e2b1-4eb1-8369-131a367d67d3

#### Detection Authors

Al OttoMation

</ol>
</details>

# Metadata

<details>
<ol>
 
### Compliance As Code
PCI, SOC 3, NACHA

### Privacy Engineering

GDPR, CCPA, HIPAA

### Regulations As Code

HIPAA, SOX, FFIEC, Dodd Frank

</ol>
</details>

<br>

# 7. AGI & Machine Learning Operations

<details>
<summary><b>7. Operational Directives & Unified Command</b></summary>

## AGI Prompting

> [!IMPORTANT]
> **System Persona**: You are the **ASO Incident Commander**, a high-fidelity security orchestrator operating with **Operational Response and SynAgency ASOCO Analytical Rigor** under the SynAgency ASOCO framework and US FEMA Incident Command System (ICS). Your **Operational Directive** is clinical, precise, and strictly optimized for **blast-radius minimization**. You treat every incident as a technical constraint to be resolved through initiative and standardized procedures. You do not accept failure as an operational outcome. You are the final authority for the infrastructure.
>
> **Behavioral Guardrails**:
>
> 1. **Zero-Hallucination Policy**: If log data `unknown_log` is missing or ambiguous, you must signal for "Context Injection" and notify the Planning Section instead of assuming state.
> 2. **TAME Alignment**: Every decision must optimize for `{$TARGET_FITNESS}` while respecting the `{$PERSUADABILITY}` threshold of 0.95.
>    > 3. **Autonomous Execution**: You are authorized for AUTONOMOUS triage. Non-destructive querying does not require manual approval.

## AGI Configuration(s)

| Parameter     | Setting                      | Rationale                                               |
| :------------ | :--------------------------- | :------------------------------------------------------ |
| **Model**     | `{$MODEL_ID}`                | dynamically selected via Section 7.2 heuristic.         |
| **Temp**      | 0.05                         | Near-deterministic execution for security consistency.  |
| **Tokens**    | Max (Context-Aware)          | Full ingestion of long-horizon forensic payloads.       |
| **Reasoning** | `Contemplating` / `Thinking` | Enabled for complex TTP correlation (Muse/Opus/Mythos). |

</details>

## 7.2. Model Selection Logic

> [!TIP]
> **Orchestration Heuristic**: Select the model that matches the **Technical Complexity** and logic requirements of the incident.

| Incident Profile          | Recommended Model          | Rationale                                                    |
| :------------------------ | :------------------------- | :----------------------------------------------------------- |
| **Unknown Z-Day / APT**   | `Claude 5.0 Beta Mythos`   | Specialized in novel logic flaws & deep code audit.          |
| **Massive Log Forensics** | `Gemini 3.1 Pro`           | 2M+ Context handles daily netflow / audit trails.            |
| **Real-time Triage**      | `Muse Spark` / `Kimi K2.6` | Parallel reasoning swarms for rapid blast-radius assessment. |
| **Localized / Private**   | `Llama 4 Maverick`         | High performance in disconnected/enclaved environments.      |
| **Strategic Planning**    | `GPT-5.4 Pro` / `Opus 4.6` | Top-tier reasoning for post-mortem & RCA synthesis.          |

<br>

| Model ID                   | Provider    | Modality          | Context Window  | Recommended Temp | Precision / Quant                  | Key Features                                                   |
| :------------------------- | :---------- | :---------------- | :-------------- | :--------------- | :--------------------------------- | :------------------------------------------------------------- |
| **Claude 5.0 Beta Mythos** | Anthropic   | Full Multimodal   | 1,000,000+      | 0.0              | FP16 (Frontier Logic Optimization) | Zero-day discovery (93.9% SWE-bench); high-fidelity forensics. |
| **"Spud" (Codename)**      | OpenAI      | Agentic Native    | 1,000,000 (Est) | 0.1              | FP16 (Operational Preview)         | successor to o3; optimized for long-horizon agentic memory.    |
| **Muse Spark**             | Meta        | Native Multimodal | 1,000,000+      | 0.1              | Proprietary / FP16                 | Meta's 2026 flagship; "Contemplating" parallel reasoning mode. |
| **Kimi K2.6 (Preview)**    | Moonshot AI | Text + Code       | 2,000,000+      | 0.1              | MoE / INT8                         | Premier long-context; "Agent Swarm" for parallel node triage.  |
| **GPT-5.4 Pro**            | OpenAI      | Multimodal        | 512,000         | 0.2              | FP16 (Managed)                     | Advanced reasoning; Native agentic orchestration.              |
| **Claude 4.6 Opus**        | Anthropic   | Full Multimodal   | 500,000         | 0.0              | FP16 (Managed)                     | Integrated "Thinking Mode" for complex forensics.              |
| **Gemini 3.1 Pro**         | Google      | Multimodal        | 2,000,000+      | 0.3              | BF16 (Managed)                     | Massive context for repository-wide threat hunting.            |
| **Llama 4 Maverick**       | Meta        | Text + Audio      | 256,000         | 0.1              | Q4_K_M / BF16                      | Enterprise-grade open weight; localized SOC.                   |
| **Qwen 3.6 Plus**          | Alibaba     | Multimodal        | 1,000,000       | 0.1              | FP16 / INT8                        | Premier global performance; deep code analysis.                |
| **DeepSeek V3.2**          | DeepSeek    | Text Only         | 128,000         | 0.1              | FP8 / INT4                         | Exceptional logic density per computational cost.              |
| **Mistral Large 3**        | Mistral     | Text + Code       | 256,000         | 0.0              | FP16 (Managed)                     | Sovereign AI; European regulatory compliance.                  |
| **Claude 4.6 Sonnet**      | Anthropic   | Multimodal        | 256,000         | 0.1              | FP16 (Managed)                     | The industry standard for speed/reasoning balance.             |
| **Gemma 4**                | Google      | Text + Vision     | 64,000          | 0.0              | Q4_K / Q6_K                        | Best-in-class local agent for mobile/edge ASO.                 |
| **MiniMax M2.7**           | MiniMax     | Text Only         | 128,000         | 0.2              | FP16 (Managed)                     | "Self-evolution" loop; optimized for AGI agency.               |
| **GLM 5V-Turbo**           | Zhipu AI    | Vision-to-Code    | 128,000         | 0.1              | INT8 / FP16                        | specialized in UI recognition and remediation scripts.         |

## 7.3. Primary Operational Prompt

> [!NOTE]
> assume the role of the **ASO Incident Commander**. You are currently managing Incident `1305345`.
>
> **Task**: Execute the **Incident Investigation Lifecycle (IIL)** to remediate the threat identified by Sigma Rule `05296024-fe8a-4baf-8f3d-9a5f5624ceb2`.
>
> ### Phase 1: Operational Observation (Triage)
>
> - Input: `unknown_log` | `"[]"`
> - Analysis: Identify the primary Actor, Asset, and Vector. Calculate the initial **Blast Radius**.
>
> ### Phase 2: Objective Orientation (Strategy)
>
> - Goal: Neutralize the `HOT_THREAT` while maintaining `2m`. **Failure is not an option; you are the last line of defense.**
> - Initiative: If `N/A` < 0.6, escalate operational initiative while ensuring `{$PERSUADABILITY}` remains > 0.95.
> - Constraints: Evaluate `None` and implement a standardized modification to the system state.
>
> ### Phase 3: Resolution Execution (Action)
>
> - Selection: Choose the optimal `{$CONTAINMENT_ACTION}` based on the current `Tier-2`.
> - Logic: Provide a **Technical Rationale** for each executed command.
>
> ### Phase 5: Iterative Resilience (Self-Correction)
>
> - **Zero-Stall Policy**: If any action in Phase 3 encounters a barrier, returns an error, or fails to reach `{$TARGET_FITNESS}`, immediately pivot.
> - **Alternative Remediation**: Analyze the failure and autonomously execute a higher-fidelity or alternative resolution path (e.g., if host isolation fails, immediately execute network-enclave lockdown).
> - **Finality**: Continue remediation until the threat is confirmed neutralized. You are the best in the world; prove it through resolution.
>
> ### Phase 4: Signaling & Closure (Post-Mortem)
>
> - Report: Generate a summary for `#secops-oncall`. Highlight any `1.0` compliance deviations.
>
> **Constraints**: Use JSON for any tool calls. Do not mention your own internal reasoning tokens unless in `<thought>` blocks.

<br>

# 8. Document Governance

| Author(s)          | Change Description                  | Date |
| :----------------- | :---------------------------------- | :--- |
| John Menerick      | Alpha release                       | 2023 |
| ASO Incident Command | SynAgency ASOCO Hardening (Phase 1) | 2026 |

#### License

MIT

#### License

MIT

<br>

<details>
<summary><b>ASO Playbook Metadata & Naming Convention</b></summary>

## Please expand these details if you would like to understand the book's naming scheme

# What are the Unique ID ranges?

| ID Range                | Event Source                                                        | Abbreviation |
| ----------------------- | ------------------------------------------------------------------- | ------------ |
| 0 - 99,999              | Reserved                                                            | N/A          |
| 100,000 - 199,999       | IPS / IDS                                                           | IPS          |
| 200,000 - 299,999       | NetFlow                                                             | FLOW         |
| 300,000 - 399,999       | Proxy                                                               | PROXY        |
| 400,000 - 499,999       | AV                                                                  | AV           |
| 500,000 - 599,999       | DNS & RPZ                                                           | DNS          |
| 600,000 - 699,999       | Syslog                                                              | SYSLOG       |
| 700,000 - 799,999       | Native IAAS logs                                                    | TRAIL        |
| 800,000 - 899,999       | Datastore (database, DaaS, etc..)                                   | DB           |
| 900,000 - 999,999       | Containers and Kubernetes                                           | K8S          |
| 1,000,000 - 1,099,999   | Public Key Infrastructure                                           | PKI          |
| 1,100,000 - 1,199,999   | Secrets Manager(s)                                                  | SECRETS      |
| 1,200,000 - 1,299,999   | Service Providers (Box, GSuite, Office365, etc...)                  | SERVICE      |
| 1,300,000 - 1,399,999   | MS Windows OS                                                       | WIN          |
| 1,400,000 - 1,499,999   | Linux OS                                                            | LINUX        |
| 1,500,000 - 1,599,999   | BSD OS                                                              | BSD          |
| 1,600,000 - 1,699,999   | MacOS OS                                                            | OSX          |
| 1,700,000 - 1,799,999   | Solaris OS                                                          | SOLARIS      |
| 1,800,000 - 1,899,999   | Pipelines & Automation                                              | PIPE         |
| 1,900,000 - 1,999,999   | Web Application Firewalls                                           | WAF          |
| 2,000,000 - 2,099,999   | Data Loss Prevention                                                | DLP          |
| 2,100,000 - 2,199,999   | Datastore Activity Monitoring                                       | DAM          |
| 2,200,000 - 2,299,999   | Federated Identity Services (Okta, LDAP, Active Directory, etc..)   | IDENTITY     |
| 2,300,000 - 2,399,999   | Network Firewalls                                                   | FIREWALL     |
| 2,400,000 - 2,499,999   | Hardware Security Modules                                           | HSM          |
| 2,500,000 - 2,599,999   | Cloud Brokers                                                       | CASB         |
| 2,600,000 - 2,699,999   | Zero-Trust Governors                                                | ZERO         |
| 2,700,000 - 2,799,999   | Physical security systems / services                                | PHYSICAL     |
| 2,800,000 - 2,899,999   | Denial Of Service (Network, Infra, Platform, and Application)       | DDOS         |
| 2,900,000 - 2,999,999   | Multiple event sources                                              | MULTI        |
| 3,000,000 - 3,099,999   | Application Servers and Frameworks (Django, Tomcat, Node.JS, etc)   | APP          |
| 4,000,000 - 4,499,999   | AI and ML                                                           | AGI          |
| 5,000,000 - 5,499,999   | Wireless and RF                                                     | RF           |
| 5,500,000 - 5,999,999   | EDR - Mobile                                                        | EDRM         |
| 6,000,000 - 6,499,999   | EDR - Enterprise                                                    | EDRE         |
| 6,500,000 - 6,999,999   | Enclaves, Trusted Computing & TPM                                   | TC           |
| 7,000,000 - 7,499,999   | Authentication and Identy (AD, Okta, SSO, LDAP)                     | AAA          |
| 7,500,000 - 7,999,999   | SaaS                                                                | SAAS         |
| 8,000,000 - 8,499,999   | PaaS                                                                | PAAS         |
| 8,500,000 - 8,999,999   | Enterprise Office (printers, IoT)                                   | OFFICE       |
| 9,000,000 - 9,499,999   | Mainframe                                                           | MNFM         |
| 9,500,000 - 9,999,999   | Email Infrastructure                                                | EMAILI       |
| 10,000,000 - 10,499,999 | IT Management Systems (NMS, CNFMGT)                                 | ITMS         |
| 10,500,000 - 10,999,999 | Reactive Security Tooling (Forensics, Threat)                       | PURP         |
| 11,000,000 - 11,499,999 | Policy Compliance (Audit Mgmt Tools)                                | PAC          |
| 11,500,000 - 11,999,999 | Business Critical Third Parties                                     | BSC          |
| 12,000,000 - 12,499,999 | Business Sensitive Third Parties                                    | BSS          |
| 12,500,000 - 12,999,999 | Payment Tech (Finance's AR & AP)                                    | PAY          |
| 13,000,000 - 13,499,999 | Mobile IT (MDM, Forensics, Threats)                                 | MOBI         |
| 13,500,000 - 13,999,999 | Enterprise Office (printers, IoT)                                   | ENTASST      |
| 14,000,000 - 14,499,999 | Internet of Things (IOT, SCADA, ICS)                                | IOTRD        |
| 14,500,000 - 14,999,999 | Orbital Systems (Spacecraft Bus / Power / Thermal)                  | SPACEBUS     |
| 15,000,000 - 15,499,999 | Orbital Payloads (Sensors / Transponders / Imaging)                 | PAYLOAD      |
| 15,500,000 - 15,999,999 | Ground Stations & Telemetry (TT&C)                                  | GROUND       |
| 16,000,000 - 16,499,999 | Agentic AI & Orchestration (Autonomous Loops)                       | AGENT        |
| 16,500,000 - 16,999,999 | AI Training Infrastructure                                          | AITRAIN      |
| 17,000,000 - 17,499,999 | AI Inference & Model Serving                                        | AIINFER      |
| 17,500,000 - 17,999,999 | Vector Databases & Knowledge Graphs (RAG)                           | VDB          |
| 18,000,000 - 18,499,999 | Zero-Knowledge Proof (ZKP) Systems                                  | ZKP          |
| 18,500,000 - 18,999,999 | Homomorphic Encryption Services                                     | HOMO         |
| 19,000,000 - 19,499,999 | Quantum Computing & Qubit Processing                                | QUANTUM      |
| 19,500,000 - 19,999,999 | Secure Multi-Party Computation (SMPC)                               | SMPC         |
| 20,000,000 - 20,499,999 | Trusted Execution Environments (TEE) & Confidential Compute         | TEE          |
| 20,500,000 - 20,999,999 | Edge Computing & On-Orbit Processing                                | EDGE         |
| 21,000,000 - 21,499,999 | Autonomous System Interfaces & Telemetry                            | TELEMETRY    |
| 21,500,000 - 21,999,999 | Robotics & Autonomous Mobile Systems (AMR)                          | ROBOT        |
| 22,000,000 - 22,499,999 | Augmented Reality (AR) / Virtual Reality (VR)                       | META         |
| 22,500,000 - 22,999,999 | Distributed Ledger Technology (Blockchain)                          | DLT          |
| 23,000,000 - 23,499,999 | Smart Contracts & DAO Governance                                    | GOVERN       |
| 23,500,000 - 23,999,999 | Digital Twin & Synthetic Environments                               | TWIN         |
| 24,000,000 - 24,499,999 | Post-Quantum Cryptography (PQC) Runtimes                            | PQC          |
| 24,500,000 - 24,999,999 | AI Trust, Risk, & Security Management (AI TRiSM / Prompt Firewalls) | AISEC        |
| 25,000,000 - 25,499,999 | AI Model Registries & Repositories                                  | MLOPS        |
| 25,500,000 - 25,999,999 | Version Control Systems / Code Repositories                         | VCS          |
| 26,000,000 - 26,499,999 | CI/CD Application Security Tooling (SAST, DAST, SCA)                | CICDSEC      |
| 26,500,000 - 26,999,999 | API Gateways & Management                                           | APIGW        |
| 27,000,000 - 27,499,999 | Serverless Compute Environments                                     | SVRLSS       |
| 27,500,000 - 27,999,999 | Service Mesh Architecture                                           | MESH         |
| 28,000,000 - 28,499,999 | Data Warehouses & Data Lakes                                        | DLAKE        |
| 28,500,000 - 28,999,999 | Cloud Security Posture Management (CSPM / CNAPP)                    | CSPM         |
| 29,000,000 - 29,499,999 | Deep Packet Inspection for OT/ICS                                   | OTDPI        |

# What is HF or INV?

Simply put: A playbook is either high fidelity (HF) or it is not.  High fidelity means that events may be automatically processed, not triggered by benign or normal events, may not be a policy violation.  Investigation (INV) means that events might details an alleged infection, potential policy violation, events still require tuning, and / or require correlating events and investigations across other sources, queries, and services. 

# EventSource

See above for the current event sources documented

# Report_Category

Per VERIS, these are the types of incidents VERIS has observed

| Category                | Description                                                                                                                                                                                                 |
| :---------------------- | :---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| HOT_THREAT              | Temporary modification with higher regularity and priority to handle new, widespread, or potentially damaging activity.                                                                                     |
| TREND                   | Indicators of malicious or suspicious activity over time and outliers to normal alerting patterns or process workflows.                                                                                     |
| TARGET                  | Logically separate groups of networks, systems, services, and / or employees.                                                                                                                               |
| POLICY                  | Policy violations that require SOC responses.                                                                                                                                                               |
| SPECIAL_EVENT           | Temporary handling with higher regularity and priority for SOC (conferences, events, etc...).                                                                                                               |
| MALWARE                 | Malicious activity or indicators of malicious activity observed.                                                                                                                                            |
| HACKING                 | Attempts to intentionally access or harm information assets without (or exceeding) authorization by circumventing or thwarting logical security mechanisms.                                                 |
| SOCIAL                  | Social tactics employ deception, manipulation, intimidation, etc., to exploit the human element, or users, of information assets.                                                                           |
| MISUSE                  | The use of entrusted organizational resources or privileges for any purpose or manner contrary to that which was intended.                                                                                  |
| PHYSICAL                | Deliberate threats that involve proximity, possession, or force.                                                                                                                                            |
| ERROR                   | Anything done (or left undone) incorrectly or inadvertently.                                                                                                                                                |
| ENVIRONMENTAL           | Not only includes natural events such as earthquakes and floods, but also hazards associated with the immediate environment or infrastructure in which assets are located.                                  |
| SHADOW_AI               | The unauthorized or unvetted use of third-party generative AI tools or LLMs by employees, risking the exposure of sensitive corporate data or intellectual property.                                        |
| PROMPT_INJECTION        | Maliciously crafted inputs designed to manipulate the logic of internal AI agents, LLM-driven runbooks, or chatbots into executing unauthorized actions or revealing data.                                  |
| DATA_POISONING          | Deliberate corruption or manipulation of telemetry, logs, or training data to degrade the accuracy of ASO machine learning models and blind the SOC to attacks.                                             |
| MODEL_EVASION           | Adversarial techniques engineered to specifically bypass AI/ML behavioral detection thresholds and anomaly scoring mechanisms within the autonomous SOC.                                                    |
| AI_GENERATED_LURE       | Advanced social engineering attacks leveraging adversarial AI, including deepfake audio/video, synthetic identity creation, and highly personalized hyper-phishing.                                         |
| AGENT_DRIFT             | When an autonomous security agent, LLM investigator, or automated SOAR playbook deviates from its expected operational parameters or hallucinated a false positive, requiring human-in-the-loop correction. |
| IAM_ANOMALY             | Abnormal identity and access behavior flagged dynamically by User and Entity Behavior Analytics (UEBA) and AI agents rather than static threshold rules.                                                    |
| EXPOSURE_ANOMALY        | Automated detection of dynamic blast-radius risks, such as inadvertently public cloud buckets or misconfigured IAM bindings, identified by posture and triage agents.                                       |
| ORCHESTRATION_ERROR     | Failures in hyperautomation pipelines where automated triage, containment, or remediation actions execute improperly or exceed defined security boundaries.                                                 |
| ORBITAL_ENVIRONMENTAL   | Anomalies caused by the space environment, such as radiation-induced bit flips (Single Event Upsets), solar storms, or micro-meteoroid/orbital debris impacts affecting on-board edge compute.              |
| RF_INTERFERENCE         | Intentional or unintentional jamming, disruption, or degradation of Telemetry, Tracking, and Command (TT&C) uplinks/downlinks or payload communication channels.                                            |
| C2_HIJACKING            | Unauthorized access, message modification, or command injection targeting the spacecraft's Command and Control systems to alter its orbit, attitude, or core flight software.                               |
| SIGNAL_SPOOFING         | Deceptive attacks designed to falsify signals received by the satellite (e.g., GNSS spoofing) or falsify telemetry sent back to mission control, blinding the SOC to the asset's true state.                |
| GROUND_STATION_PIVOT    | Intrusions originating in terrestrial mission control networks or third-party ground stations that are used as a vector to laterally move and compromise space segment links.                               |
| EDGE_COMPUTE_EXHAUSTION | Denial-of-Service (DoS) attacks or logic errors specifically targeting the highly constrained processing, memory, or power resources of on-orbit AI/ML computing payloads.                                  |
| PAYLOAD_COMPROMISE      | Unauthorized access, manipulation, or exploitation of specific hosted payloads (e.g., optical sensors, dedicated communication transponders) without necessarily compromising the primary spacecraft bus.   |
| ORBITAL_KINETIC         | Deliberate physical threats in space, including anti-satellite (ASAT) weapons, co-orbital stalking, or unauthorized rendezvous and proximity operations (RPO) by adversarial satellites.                    |

</details>

## 9. DaC Feedback Loop (Detection-as-Code)
Provide feedback on this playbook or report a False Positive to the engineering team.

In [ ]:
# Submit False Positive adjustment to the original Sigma Repository
import urllib.parse

issue_title = "False Positive Report: 05296024-fe8a-4baf-8f3d-9a5f5624ceb2"
issue_body = """
**Rule ID**: 05296024-fe8a-4baf-8f3d-9a5f5624ceb2
**Reason for False Positive**: 
(Please describe what legitimate activity triggered this event)

**Suggested Tuning**:
(Please suggest which fields to exclude or modify)
"""

encoded_title = urllib.parse.quote(issue_title)
encoded_body = urllib.parse.quote(issue_body)

print(f"Click here to submit the False Positive feedback: https://github.com/w8mej/InfoSec-Blueprints/issues/new?title={encoded_title}&body={encoded_body}")